<a href="https://colab.research.google.com/github/yangyi02/droid/blob/main/pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### DROID: A Large-Scale In-The-Wild Robot Manipulation Dataset

![](https://droid-dataset.github.io/droid/assets/index/droid_teaser.jpg)

This Colab demonstrates how to load and visualize samples from the DROID dataset. Please also check out our [dataset visualizer](https://droid-dataset.github.io/dataset.html) to explore the dataset.

You can download the full dataset (1.7TB) using:
```
gsutil -m cp -r gs://gresearch/robotics/droid <your_local_path>
```

If you'd like to download an example version of the dataset with 100 episodes first (2GB), run:
```
gsutil -m cp -r gs://gresearch/robotics/droid_100 <your_local_path>
```

If you want to use DROID for policy training, please check out our [policy training repo](https://github.com/droid-dataset/droid_policy_learning).

### 环境配置与初始化

In [ ]:
# @title 安装 ZED SDK
!apt-get update -qq
!apt-get install -y zstd
sdk_installer = "ZED_SDK_Linux_Ubuntu22.run"
download_url = "https://download.stereolabs.com/zedsdk/5.2/cu12/ubuntu22"
!wget -q -O {sdk_installer} {download_url}
!chmod +x {sdk_installer}

# Silent installation, skip unnecessary tools
!./{sdk_installer} silent runtime_only skip_tools

# Install Python interface (pyzed) via precompiled wheel file
!find /usr/local/zed/ -name "pyzed*.whl" -exec pip install {} \;

import pyzed.sl as sl
print("pyzed 安装成功！")

In [ ]:
# @title 处理pyzed问题

# 💣 1. 强制物理删除坏掉的第三方 PPA 源列表文件
!rm -f /etc/apt/sources.list.d/*ubuntugis*.list
!rm -f /etc/apt/sources.list.d/*deadsnakes*.list
!rm -f /etc/apt/sources.list.d/*graphics-drivers*.list

# 🔄 2. 刷新系统软件列表（此时系统就不会再去连那个死掉的服务器了）
!apt-get update -qq

# 💉 3. 强制从 Ubuntu 官方稳定源安装这些被卡住的依赖
!apt-get install -y --fix-missing python3-setuptools python3-pkg-resources libturbojpeg libarchive-dev libusb-1.0-0-dev libv4l-0 mesa-utils

In [ ]:
# @title 安装 Python 依赖库

!pip install mediapy pyrender trimesh PyOpenGL-accelerate open3d pybullet polyscope

In [ ]:
# @title 导入通用 Python 库

import copy
import os
import sys
import gc
import glob
import cv2
import h5py
import json
from matplotlib import cm
import matplotlib.pyplot as plt
import mediapy as media
import numpy as np
from PIL import Image, ImageDraw
import plotly.graph_objects as go
import plotly.io as pio
import polyscope as ps
import pybullet as p
import pybullet_data
import pyrender
import pyzed.sl as sl
import random
from scipy.spatial.transform import Rotation as R, Slerp
import trimesh
import tensorflow_datasets as tfds
from tqdm import tqdm
import torch
import torch.nn.functional as F
import torch.optim as optim

os.environ['PYOPENGL_PLATFORM'] = 'egl'
pio.renderers.default = 'colab'
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [ ]:
# @title 克隆 Github 仓库 (S2M2, VGGT, CoTracker, SAM)

%cd /content

!git clone https://github.com/junhong-3dv/s2m2.git
sys.path.append("/content/s2m2/src")

!git clone https://github.com/facebookresearch/vggt.git
sys.path.append("/content/vggt")

!git clone https://github.com/facebookresearch/co-tracker.git
sys.path.append("/content/co-tracker")

!pip install -q git+https://github.com/facebookresearch/segment-anything.git

from s2m2.core.utils.model_utils import load_model
print("✅ [1/4] S2M2 高精度深度模型模块 导入成功！")

from vggt.models.vggt import VGGT
from vggt.utils.pose_enc import pose_encoding_to_extri_intri
print("✅ [2/4] VGGT 视觉外参大模型模块 导入成功！")

from cotracker.predictor import CoTrackerPredictor
print("✅ [3/4] CoTracker3 稠密点追踪模块 导入成功！")

from segment_anything import sam_model_registry, SamPredictor
print("✅ [4/4] SAM 导入成功！")

In [ ]:
# @title 下载 S2M2 模型并定义深度提取函数

from s2m2.core.utils.model_utils import load_model, run_stereo_matching

target_url = "https://huggingface.co/minimok/s2m2/resolve/main/CH384NTR3.pth"
os.makedirs("/content/s2m2/weights/pretrain_weights", exist_ok=True)
save_path = "/content/s2m2/weights/pretrain_weights/CH384NTR3.pth"

if not os.path.exists(save_path) or os.path.getsize(save_path) < 100 * 1024 * 1024:
    !wget -O {save_path} {target_url}

s2m2_model = load_model("/content/s2m2/weights/pretrain_weights", "XL", True, 3, device)
s2m2_model.eval()
s2m2_model = torch.compile(s2m2_model)

@torch.inference_mode()
def get_s2m2_disparity(img_left, img_right, device, conf_thresh=0.95):
    """纯粹的视差提取器：只推断，不过早转换物理深度"""
    left_torch = torch.from_numpy(img_left).permute(2, 0, 1).unsqueeze(0).to(device)
    right_torch = torch.from_numpy(img_right).permute(2, 0, 1).unsqueeze(0).to(device)

    pred_disp, _, pred_conf, _, _ = run_stereo_matching(
        s2m2_model, left_torch, right_torch, device, N_repeat=3
    )

    disp = pred_disp.cpu().numpy().squeeze()
    conf = pred_conf.cpu().numpy().squeeze()

    # 🌟 极简掩码过滤：置信度不足的地方，视差直接归零
    valid_mask = (disp > 0) & (conf >= conf_thresh)
    disp[~valid_mask] = 0.0

    return disp

In [ ]:
# @title 下载与初始化 CoTracker3 模型

from cotracker.predictor import CoTrackerPredictor

WEIGHTS_URL = "https://huggingface.co/facebook/cotracker3/resolve/main/scaled_offline.pth"
os.makedirs("/content/co-tracker/weights", exist_ok=True)
weights_path = os.path.join("/content/co-tracker/weights/", "cotracker3_offline.pth")

if not os.path.exists(weights_path):
    !wget -q {WEIGHTS_URL} -O {weights_path}

cotracker_model = CoTrackerPredictor(checkpoint=weights_path)
cotracker_model = cotracker_model.to(device)

print("✅ 模型加载成功！现在可以继续运行后续的 2D 跟踪代码了。")

In [ ]:
# @title 下载与初始化 VGGT 模型

from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images
from vggt.utils.pose_enc import pose_encoding_to_extri_intri

print("🚀 正在加载 VGGT-1B 模型至显存...")
vggt_model = VGGT.from_pretrained("facebook/VGGT-1B").to(device)
vggt_model.eval()
print("✅ VGGT 模型加载完成！")

# 1. 模型懒加载机制 (保护显存)
dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16

def estimate_multi_camera_vggt(img_list):
    """
    V14 核心升华版：支持任意数量 N 个相机的全局联合推理！
    第 0 张图被视为参考系原点，返回后续所有图相对于原点的 T 矩阵。
    """
    filenames = []
    # 动态保存临时文件以适配 VGGT 的预处理接口
    for i, img in enumerate(img_list):
        fname = f"tmp_vggt_{i}.png"
        cv2.imwrite(fname, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
        filenames.append(fname)

    images = load_and_preprocess_images(filenames).to(device)
    images_input = images.unsqueeze(0) # [1, N, 3, H, W]

    with torch.inference_mode():
        with torch.autocast(device_type="cuda", dtype=dtype):
            aggregated_tokens_list, ps_idx = vggt_model.aggregator(images_input)
            pose_enc = vggt_model.camera_head(aggregated_tokens_list)[-1]
            extrinsic, intrinsic = pose_encoding_to_extri_intri(pose_enc, images_input.shape[-2:])

    # 提取第 0 张图 (参考图) 到其他所有图的变换矩阵 T_{ref -> tgt}
    T_ref_to_tgts = []
    for i in range(1, len(img_list)):
        ext_mat = extrinsic[0, i].cpu().numpy()
        T = np.eye(4)
        T[:3, :] = ext_mat
        T_ref_to_tgts.append(T)

    return T_ref_to_tgts

In [ ]:
# @title 下载与初始化 Segment Anything 并加载 ViT-H 模型

print("⬇️ 正在下载 SAM ViT-H 权重...")
os.makedirs("/content/sam_weights", exist_ok=True)
sam_checkpoint = "/content/sam_weights/sam_vit_h_4b8939.pth"
if not os.path.exists(sam_checkpoint):
    !wget -q -O {sam_checkpoint} https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth

from segment_anything import sam_model_registry, SamPredictor

print("🚀 正在将 SAM 模型加载至显存 (可能需要十秒钟)...")
sam = sam_model_registry["vit_h"](checkpoint=sam_checkpoint)
sam.to(device=device)
sam_predictor = SamPredictor(sam)
print("✅ SAM 大模型满血复活！")

In [ ]:
# @title 下载 Robotiq URDF 与 3D 物理资产

# 1. 克隆包含定制版 Franka + Robotiq 模型的官方仓库 (强行指定 data 分支)
!git clone -b data https://github.com/NVlabs/PointWorld.git

In [ ]:
# @title 下载并解析 DROID 数据集元数据 (JSON)

root_path = "/content/droid_raw/1.0.1"
base_url = "https://huggingface.co/KarlP/droid/resolve/main"

# 🗑️ 移除 cam2base_extrinsic_superset.json
files = ["intrinsics.json", "camera_serials.json", "episode_id_to_path.json", "keep_ranges_1_0_1.json", "cam2base_extrinsic_superset.json"]

os.makedirs(root_path, exist_ok=True)
for f in files:
    os.system(f"wget -q -nc -P {root_path} {base_url}/{f}")

def load_json(name):
    with open(os.path.join(root_path, name), 'r') as f: return json.load(f)

serials_db, id_to_path = load_json(files[1]), load_json(files[2])
keep_ranges = load_json(files[3])
extrinsics_db = load_json(files[4])

# 🌟 解锁封印：现在的 valid_ids 只受限于基本元数据，直接打通全部数据！
valid_ids = sorted(set(serials_db.keys()) & set(id_to_path.keys()))
print(f"✅ 全局元数据准备完毕！匹配到 {len(valid_ids)} 个 Episode。")

# 🌟 新增：计算包含官方外参的 Episode 集合
episodes_with_ext = set(valid_ids) & set(extrinsics_db.keys())
print(f"   📸 其中包含官方预标定外参的 Episode 数量为: {len(episodes_with_ext)} 个。")

# 只采样有初始相机标定的数据
valid_ids = sorted(set(serials_db.keys()) & set(id_to_path.keys()) & set(extrinsics_db.keys()))
print(f"✅ 全局元数据准备完毕！匹配到包含官方预标定外参的 Episode 共 {len(valid_ids)} 个。")

### 数据准备

In [ ]:
# @title 下载 DROID Episode 并初始化全局状态池

def download_episode(episode_id, root_path, id_to_path, serials_db, keep_ranges_db):
    """一站式下载视频，并按层级分类初始化 scene_constants (元数据终极聚合版)"""
    # id_to_path[episode_id] 本身就是类似 "RPL/failure/2023-11-30-18h-16m-00s" 的相对路径
    relative_path = id_to_path[episode_id]
    episode_path = os.path.join(root_path, relative_path)

    if not os.path.exists(episode_path):
        os.makedirs(episode_path, exist_ok=True)
        os.system(f"gsutil -m -q cp -r \"gs://gresearch/robotics/droid_raw/1.0.1/{relative_path}/*\" \"{episode_path}/\"")
        print(f"  ⬇️ 成功下载数据: {episode_id}")
    else:
        print(f"  ⏭️ 数据已存在，跳过下载。")

    cam_info = serials_db[episode_id]
    wrist_serial = cam_info.get('wrist_cam_serial')

    # 🗑️ 彻底脱离 intrinsics_db 依赖，直接从串口数据中提取全部机位
    valid_cams = sorted(set(cam_info.values()))

    # 🌟 修复：完美还原官方 JSON 里的 absolute bucket path 格式
    base_prefix = "gs://xembodiment_data/r2d2/r2d2-data-full/"
    episode_key = f"{base_prefix}{relative_path}/recordings/MP4--{base_prefix}{relative_path}/trajectory.h5"

    valid_indices = None

    if episode_key in keep_ranges_db:
        ranges = keep_ranges_db[episode_key]
        indices = []
        for start, end in ranges:
            indices.extend(range(start, end))
        valid_indices = np.array(indices)
        print(f"  ✂️ 已预加载动作区间，共标记 {len(valid_indices)} 帧为有效关键帧。")
    else:
        # 为了防范极端情况，如果真找不到（尽管概率很小），保留全量帧
        print(f"  ⚠️ 未找到 {episode_id} 的 Idle 过滤信息，将默认保留全量帧。")

    # 🌟 极简封箱：结构化分为 meta, robot, camera 三大核心模块
    return {
        'meta': {
            'episode_id': episode_id,
            'episode_path': episode_path,
            'wrist_serial': wrist_serial,
            'valid_indices': valid_indices  # <--- 完美注入！
        },
        'robot': {},  # 预留位，供后续运动学解析填入
        'camera': {
            cam: {
                'baseline': 0.063 if cam == wrist_serial else 0.120,
            } for cam in valid_cams  # 字典推导式优雅注入所有相机
        }
    }

# ================= 主流程 =================
episode_id = random.choice(valid_ids)
print(f"🎯 选定处理的 Episode: {episode_id}")

# 🌟 移除了 intrinsics_db 传参
scene_constants = download_episode(episode_id, root_path, id_to_path, serials_db, keep_ranges)

print(f"✅ scene_constants 初始化就绪！")
print(f"   - 腕部相机: {scene_constants['meta']['wrist_serial']}")
print(f"   - 已加载机位: {list(scene_constants['camera'].keys())}")
if scene_constants['meta']['valid_indices'] is not None:
    print(f"   - 动作有效帧数: {len(scene_constants['meta']['valid_indices'])}")

In [ ]:
# @title 读取 DROID Episode (内参全量提取 + 双目原生畸变视频版)

def extract_svo_video(scene_constants):
    """解码 SVO 视频，全量提取左右眼标定数据，并同时抓取去畸变与原生带畸变的双目画面"""
    print("  🎥 极速解码 SVO 视频流与物理标定数据...")
    episode_path = scene_constants['meta']['episode_path']

    for cam in scene_constants['camera']:
        svo_path = glob.glob(os.path.join(episode_path, f"**/{cam}.svo"), recursive=True)[0]

        zed, init_params = sl.Camera(), sl.InitParameters()
        init_params.set_from_svo_file(svo_path)
        init_params.svo_real_time_mode = False
        zed.open(init_params)

        # 🌟 提取 ZED 原生标定数据
        cam_info = zed.get_camera_information()

        # --- 去畸变态 (Calibrated) ---
        calib = cam_info.camera_configuration.calibration_parameters
        K_calib_left = np.array([
            [calib.left_cam.fx, 0, calib.left_cam.cx],
            [0, calib.left_cam.fy, calib.left_cam.cy],
            [0, 0, 1]
        ], dtype=np.float32)
        disto_calib_left = np.array(calib.left_cam.disto, dtype=np.float32)

        K_calib_right = np.array([
            [calib.right_cam.fx, 0, calib.right_cam.cx],
            [0, calib.right_cam.fy, calib.right_cam.cy],
            [0, 0, 1]
        ], dtype=np.float32)
        disto_calib_right = np.array(calib.right_cam.disto, dtype=np.float32)

        # --- 原生物理态 (Raw) ---
        calib_raw = cam_info.camera_configuration.calibration_parameters_raw
        K_raw_left = np.array([
            [calib_raw.left_cam.fx, 0, calib_raw.left_cam.cx],
            [0, calib_raw.left_cam.fy, calib_raw.left_cam.cy],
            [0, 0, 1]
        ], dtype=np.float32)
        disto_raw_left = np.array(calib_raw.left_cam.disto, dtype=np.float32)

        K_raw_right = np.array([
            [calib_raw.right_cam.fx, 0, calib_raw.right_cam.cx],
            [0, calib_raw.right_cam.fy, calib_raw.right_cam.cy],
            [0, 0, 1]
        ], dtype=np.float32)
        disto_raw_right = np.array(calib_raw.right_cam.disto, dtype=np.float32)

        # ==========================================
        # 🌟 画面抓取循环 (新增左右眼 Raw 提取)
        # ==========================================
        all_left, all_right, all_left_raw, all_right_raw = [], [], [], []
        left_mat, right_mat = sl.Mat(), sl.Mat()
        left_raw_mat, right_raw_mat = sl.Mat(), sl.Mat()

        for _ in tqdm(range(zed.get_svo_number_of_frames()), desc=f"Decoding {cam}"):
            zed.grab()
            # 1. 提取去畸变后的双目画面 (Rectified)
            zed.retrieve_image(left_mat, sl.VIEW.LEFT)
            zed.retrieve_image(right_mat, sl.VIEW.RIGHT)
            # 2. 🌟 提取原生带畸变的双目画面 (Unrectified)
            zed.retrieve_image(left_raw_mat, sl.VIEW.LEFT_UNRECTIFIED)
            zed.retrieve_image(right_raw_mat, sl.VIEW.RIGHT_UNRECTIFIED)

            all_left.append(cv2.cvtColor(left_mat.get_data(), cv2.COLOR_BGRA2RGB))
            all_right.append(cv2.cvtColor(right_mat.get_data(), cv2.COLOR_BGRA2RGB))
            all_left_raw.append(cv2.cvtColor(left_raw_mat.get_data(), cv2.COLOR_BGRA2RGB))
            all_right_raw.append(cv2.cvtColor(right_raw_mat.get_data(), cv2.COLOR_BGRA2RGB))

        zed.close()

        # ==========================================
        # 🌟 扩充封箱
        # ==========================================
        scene_constants['camera'][cam].update({
            'K_mat': K_calib_left,
            'zed_calibration': {
                'calibrated': {
                    'K': K_calib_left, 'disto': disto_calib_left,
                    'K_right': K_calib_right, 'disto_right': disto_calib_right
                },
                'raw': {
                    'K': K_raw_left, 'disto': disto_raw_left,
                    'K_right': K_raw_right, 'disto_right': disto_raw_right
                }
            },
            'video_rgb': np.stack(all_left),
            'video_right': np.stack(all_right),
            'video_raw_rgb': np.stack(all_left_raw),
            'video_raw_right': np.stack(all_right_raw)  # <--- 🌟 新增：右眼原生畸变视频池
        })

    return scene_constants

# ================= 主流程 =================
scene_constants = extract_svo_video(scene_constants)

In [ ]:
# @title 🖨️ 打印全视角双目相机内参对比大盘
print("🔍 正在核对多视角双目相机的标定数据...")
print("=" * 60)

for cam_id, cam_data in scene_constants['camera'].items():
    print(f"📷 【相机机位: {cam_id}】")

    # 1. 推荐使用的新版去畸变内参 (默认左眼)
    print("  🟢 [K_mat] (ZED 实时去畸变内参 - 左眼):")
    print(cam_data.get('K_mat'))

    # 2. 备用的老版 JSON 缩放内参
    print("\n  🟡 [K_mat_json_scaled] (老版 JSON 缩放内参 - 左眼):")
    print(cam_data.get('K_mat_json_scaled'))

    # 3. 完整的双目物理标定数据块
    print("\n  🔴 [zed_calibration] (底层完整双目物理标定数据):")
    zed_calib = cam_data.get('zed_calibration', {})

    if zed_calib:
        calib_data = zed_calib.get('calibrated', {})
        print("    👉 理想针孔态 (Calibrated / Rectified):")
        print("      - 左眼 K 矩阵:\n", calib_data.get('K'))
        print("      - 左眼 畸变 (Disto):", calib_data.get('disto'))
        print("      - 右眼 K 矩阵:\n", calib_data.get('K_right'))
        print("      - 右眼 畸变 (Disto):", calib_data.get('disto_right'))

        raw_data = zed_calib.get('raw', {})
        print("\n    👉 物理真实态 (Raw / Distorted):")
        print("      - 左眼 K 矩阵:\n", raw_data.get('K'))
        print("      - 左眼 畸变 (Disto):", raw_data.get('disto'))
        print("      - 右眼 K 矩阵:\n", raw_data.get('K_right'))
        print("      - 右眼 畸变 (Disto):", raw_data.get('disto_right'))
    else:
        print("    ⚠️ 未找到 zed_calibration 数据块。")

    print("=" * 60)

In [ ]:
# @title 🔍 双目原生畸变 vs 极线校正对比渲染 (2x2)

def render_distortion_comparison_video(scene_constants, tgt_width=1280):
    print("🎨 正在渲染双目畸变对比视频...")
    wrist_cam = scene_constants['meta']['wrist_serial']
    cam_data = scene_constants['camera'][wrist_cam]

    # 提取四组画面
    video_rect_l = cam_data['video_rgb']
    video_raw_l = cam_data['video_raw_rgb']
    video_rect_r = cam_data['video_right']
    video_raw_r = cam_data['video_raw_right']

    n_frames, h, w, _ = video_rect_l.shape
    comparison_frames = []

    # 提取焦距，用于盖戳
    fx_raw_l = cam_data['zed_calibration']['raw']['K'][0,0]
    fx_rect_l = cam_data['zed_calibration']['calibrated']['K'][0,0]
    fx_raw_r = cam_data['zed_calibration']['raw']['K_right'][0,0]
    fx_rect_r = cam_data['zed_calibration']['calibrated']['K_right'][0,0]

    for t in tqdm(range(n_frames), desc="拼接画面"):
        img_raw_l = video_raw_l[t].copy()
        img_rect_l = video_rect_l[t].copy()
        img_raw_r = video_raw_r[t].copy()
        img_rect_r = video_rect_r[t].copy()

        # 统一绘制红色辅助网格线
        step_y, step_x = h // 6, w // 8
        for img in [img_raw_l, img_rect_l, img_raw_r, img_rect_r]:
            for y in range(0, h, step_y):
                cv2.line(img, (0, y), (w, y), (255, 0, 0), 1)
            for x in range(0, w, step_x):
                cv2.line(img, (x, 0), (x, h), (255, 0, 0), 1)

        # 定义盖戳小函数
        def add_label(img, text, color):
            cv2.putText(img, text, (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 0), 4)
            cv2.putText(img, text, (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 2)

        # 盖戳
        add_label(img_raw_l, f"LEFT RAW - Fx: {fx_raw_l:.1f}", (0, 255, 255))
        add_label(img_rect_l, f"LEFT RECTIFIED - Fx: {fx_rect_l:.1f}", (0, 255, 0))
        add_label(img_raw_r, f"RIGHT RAW - Fx: {fx_raw_r:.1f}", (0, 255, 255))
        add_label(img_rect_r, f"RIGHT RECTIFIED - Fx: {fx_rect_r:.1f}", (0, 255, 0))

        # 🌟 拼接：2x2 田字格
        top_row = np.concatenate([img_raw_l, img_rect_l], axis=1)
        bottom_row = np.concatenate([img_raw_r, img_rect_r], axis=1)
        combined = np.concatenate([top_row, bottom_row], axis=0)

        # 等比例缩放
        tgt_h = int(combined.shape[0] * tgt_width / combined.shape[1])
        comparison_frames.append(cv2.resize(combined, (tgt_width, tgt_h)))

    return comparison_frames

# 运行渲染并播放 (稍微增加一下高度显示比例)
comparison_video = render_distortion_comparison_video(scene_constants)
media.show_video(comparison_video, fps=15, codec='gif', height=360)

In [ ]:
# @title 机器人运动学解析

def make_4x4(vec_6d):
    """将 6DoF 向量 [x, y, z, rx, ry, rz] 优雅转换为 4x4 齐次变换矩阵"""
    transform = np.eye(4)
    transform[:3, :3] = R.from_euler('xyz', vec_6d[3:]).as_matrix()
    transform[:3, 3] = vec_6d[:3]
    return transform

def parse_robot_kinematics(scene_constants):
    """读取 H5 与 JSON，提取机器人运动学轨迹与动态手眼矩阵 (全量读取版)"""
    print("  🦾 解析机器人 H5 运动学与动态手眼矩阵...")
    ep_path = scene_constants['meta']['episode_path']

    # 🗑️ 彻底移除了从视频层级获取 n_frames 的限制逻辑

    # 🌟 2. 极简读取：一次性打开并提取需要的 H5 和 JSON 数据 (吸入全部帧)
    with h5py.File(f"{ep_path}/trajectory.h5", 'r') as f, \
         open(glob.glob(f"{ep_path}/metadata_*.json")[0]) as jf:

        # 🌟 核心修改：使用 [:] 直接提取 H5 文件中的全量数组
        ee_poses = f["observation/robot_state/cartesian_position"][:]
        joint_poses = f["observation/robot_state/joint_positions"][:]
        # 🌟 新增：提取夹爪开合度数据！(通常是 1 维的宽度或位置)
        gripper_poses = f["observation/robot_state/gripper_position"][:]
        wrist_ext = json.load(jf)["wrist_cam_extrinsics"]
        wrist_ext = wrist_ext.get('extrinsics', wrist_ext) if isinstance(wrist_ext, dict) else wrist_ext

    # 获取底层实际包含的全部帧数
    total_frames = len(ee_poses)

    # 🌟 3. 广播魔法：利用 scipy 的向量化能力，瞬间生成全序列末端位姿
    T_ee_all = np.tile(np.eye(4), (total_frames, 1, 1))
    T_ee_all[:, :3, :3] = R.from_euler('xyz', ee_poses[:, 3:]).as_matrix()
    T_ee_all[:, :3, 3] = ee_poses[:, :3]

    # 🌟 4. 结构化封箱：一次性压入 robot 层级
    scene_constants['robot'] = {
        'joint_positions': joint_poses,
        'gripper_positions': gripper_poses,
        'T_cam_ee_init': np.linalg.inv(make_4x4(ee_poses[0])) @ make_4x4(wrist_ext),
        'T_ee_base_all': T_ee_all
    }
    return scene_constants

# ================= 主流程 =================
scene_constants = parse_robot_kinematics(scene_constants)

In [ ]:
# @title 打印当前读取内容

def inspect_dict_structure(data, name="scene_constants", indent=0):
    """递归打印字典内各变量的形状、长度和类型"""
    spacing = "  " * indent
    if isinstance(data, dict):
        print(f"{spacing}📂 {name} (dict, {len(data)} keys)")
        for k, v in data.items():
            inspect_dict_structure(v, name=str(k), indent=indent + 1)
    elif isinstance(data, np.ndarray):
        print(f"{spacing}📊 {name}: ndarray, shape={data.shape}, dtype={data.dtype}")
    elif torch.is_tensor(data):
        print(f"{spacing}🔥 {name}: Tensor, shape={tuple(data.shape)}, dtype={data.dtype}")
    elif isinstance(data, (list, tuple)):
        print(f"{spacing}📜 {name}: {type(data).__name__}, len={len(data)}")
    else:
        # 对于字符串、浮点数等基础类型，直接打印前 50 个字符
        val_str = str(data)
        if len(val_str) > 50:
            val_str = val_str[:47] + "..."
        print(f"{spacing}🏷️ {name}: {type(data).__name__} = {val_str}")

# 执行全域透视
print("\n" + "="*50)
inspect_dict_structure(scene_constants)
print("="*50 + "\n")

In [ ]:
# @title 全域时序对齐 (解决木桶效应)

def align_temporal_streams(scene_constants):
    """物理世界同步算子：强行截断所有时序流，确保全局帧数绝对一致"""
    print("  ⏱️ 启动全域时序对齐检查...")

    # 🌟 1. 收集所有时序流的实际长度
    lengths = [
        len(scene_constants['robot']['joint_positions']),
        len(scene_constants['robot']['gripper_positions']), # <--- 新增：加入夹爪开合度数据的长度统计
        len(scene_constants['robot']['T_ee_base_all'])
    ]
    for cam_id, cam_data in scene_constants['camera'].items():
        lengths.append(len(cam_data['video_rgb']))
        lengths.append(len(cam_data['video_right']))

    # 🌟 2. 找到最短的“木桶短板”
    min_frames = min(lengths)
    max_frames = max(lengths)

    if min_frames == max_frames:
        print(f"    ✅ 时序完美对齐，共 {min_frames} 帧，无需处理。")
        return scene_constants

    print(f"    ⚠️ 发现时序偏差 (最大 {max_frames} 帧，最小 {min_frames} 帧)！")
    print(f"    ✂️ 正在挥下奥卡姆剃刀，全局强行截断至 {min_frames} 帧...")

    # 🌟 3. 无情截断：统一所有维度
    scene_constants['robot']['joint_positions'] = scene_constants['robot']['joint_positions'][:min_frames]
    scene_constants['robot']['gripper_positions'] = scene_constants['robot']['gripper_positions'][:min_frames] # <--- 新增：同步截断夹爪数据
    scene_constants['robot']['T_ee_base_all'] = scene_constants['robot']['T_ee_base_all'][:min_frames]

    for cam_id, cam_data in scene_constants['camera'].items():
        cam_data['video_rgb'] = cam_data['video_rgb'][:min_frames]
        cam_data['video_right'] = cam_data['video_right'][:min_frames]

    print("    ✅ 全域对齐完毕！所有维度现已牢不可破。")
    return scene_constants

# ================= 主流程 =================
scene_constants = align_temporal_streams(scene_constants)

In [ ]:
# @title 提取全局头部时序 (聚焦初始动作)

def extract_first_n_frames(scene_constants, n=48):
    """时序聚焦算子：提取所有时序张量的最前 N 帧，自动适配多模态维度"""
    print(f"  ✂️ 启动时序截断：聚焦全局最前 {n} 帧...")

    # 🌟 1. 获取当前全局帧数基准 (以 robot 关节数据为准)
    current_frames = len(scene_constants['robot']['joint_positions'])

    if current_frames <= n:
        print(f"    ✅ 当前总帧数 ({current_frames} 帧) 已满足要求，无需截断。")
        return scene_constants

    # 🌟 2. 截断 Robot 本体运动学时序
    scene_constants['robot']['joint_positions'] = scene_constants['robot']['joint_positions'][:n]
    scene_constants['robot']['gripper_positions'] = scene_constants['robot']['gripper_positions'][:n] # <--- 新增：同步截断夹爪数据的时序
    scene_constants['robot']['T_ee_base_all'] = scene_constants['robot']['T_ee_base_all'][:n]

    # 🌟 3. 截断全域 Camera 视角下的所有时序张量
    for cam_id, cam_data in scene_constants['camera'].items():
        # 基础双目视频流
        cam_data['video_rgb'] = cam_data['video_rgb'][:n]
        cam_data['video_right'] = cam_data['video_right'][:n]

    print(f"    ✅ 截断完毕！所有时序维度现已缩减为最前的 {n} 帧。")
    return scene_constants

# ================= 主流程 =================
scene_constants = extract_first_n_frames(scene_constants, n=250)

inspect_dict_structure(scene_constants)

In [ ]:
# @title 🎯 3D 视觉几何核心大一统算子库

def decode_disparity_np(disp, fx, baseline):
    """将原始视差转化为物理深度 (NumPy)"""
    z = np.zeros_like(disp)
    valid_mask = disp > 0  # 过滤无效视差
    z[valid_mask] = (fx * baseline) / disp[valid_mask]
    return z

def decode_disparity_pt(disp, fx, baseline):
    """将原始视差转化为物理深度 (PyTorch 完美梯度版)"""
    z = torch.zeros_like(disp)
    valid_mask = disp > 0  # 过滤无效视差
    z[valid_mask] = (fx * baseline) / disp[valid_mask]
    return z

def unproject_points_np(u, v, z, K, T_cam2world=None):
    """纯粹的 3D 射线反投影算子，只处理合法的物理深度 Z (NumPy)"""
    x_cam = (u - K[0, 2]) * z / K[0, 0]
    y_cam = (v - K[1, 2]) * z / K[1, 1]

    pts_cam = np.stack([x_cam, y_cam, z, np.ones_like(z)], axis=0)

    if T_cam2world is None:
        return pts_cam[:3, :].T
    return (T_cam2world @ pts_cam)[:3, :].T

def unproject_points_pt(u, v, z, K, T_cam2world=None):
    """纯粹的 3D 射线反投影算子，完美保持梯度穿透 (PyTorch)"""
    x_cam = (u - K[0, 2]) * z / K[0, 0]
    y_cam = (v - K[1, 2]) * z / K[1, 1]

    pts_cam = torch.stack([x_cam, y_cam, z, torch.ones_like(z)], dim=0)

    if T_cam2world is None:
        return pts_cam[:3, :].T
    return (T_cam2world @ pts_cam)[:3, :].T

def project_points_np(pts_world, K, T_cam2world):
    """将 3D 世界点阵拍回 2D 像素平面 (NumPy)"""
    T_world2cam = np.linalg.inv(T_cam2world)
    pts_homo = np.hstack([pts_world, np.ones((len(pts_world), 1))]).T
    pts_cam = T_world2cam @ pts_homo

    z_cam = pts_cam[2, :]
    u = np.zeros_like(pts_cam[0, :])
    v = np.zeros_like(pts_cam[1, :])

    valid_mask = z_cam > 0
    u[valid_mask] = (pts_cam[0, valid_mask] / z_cam[valid_mask]) * K[0, 0] + K[0, 2]
    v[valid_mask] = (pts_cam[1, valid_mask] / z_cam[valid_mask]) * K[1, 1] + K[1, 2]

    return u, v, z_cam

def project_points_pt(pts_world, K, T_cam2world):
    """将 3D 世界点阵拍回 2D 像素平面，完美保留计算图 (PyTorch)"""
    T_world2cam = torch.linalg.inv(T_cam2world)
    pts_homo = torch.cat([pts_world, torch.ones((len(pts_world), 1), device=pts_world.device)], dim=1).T
    pts_cam = T_world2cam @ pts_homo

    z_cam = pts_cam[2, :]
    u = torch.zeros_like(pts_cam[0, :])
    v = torch.zeros_like(pts_cam[1, :])

    valid_mask = z_cam > 0
    u[valid_mask] = (pts_cam[0, valid_mask] / z_cam[valid_mask]) * K[0, 0] + K[0, 2]
    v[valid_mask] = (pts_cam[1, valid_mask] / z_cam[valid_mask]) * K[1, 1] + K[1, 2]

    return u, v, z_cam

In [ ]:
# @title SVO 单帧提取深度

def unproject_to_3d(depth, color_img, K_mat, T_cam2world=None, min_depth=0., max_depth=1.5):
    """纯粹的几何升维算子：输入已校准的深度，专心做空间截断与反投影"""
    # 🌟 1. 空间屏蔽：用最符合直觉的物理阈值过滤
    mask = (depth > min_depth) & (depth < max_depth)
    v, u = np.where(mask)

    # 🌟 2. 几何升维：调用底层原生算子
    if T_cam2world is None:
        T_cam2world = np.eye(4)
    pts_world = unproject_points_np(u, v, depth[mask], K_mat, T_cam2world)

    return pts_world, color_img[mask]

def show_plotly_point_cloud(pts, cols, title="3D Point Cloud", max_points=150000, eye_pos=(-1.5, -1.5, 1.0)):
    """交互式点云渲染 (保持极简瘦身版不变)"""
    idx = np.random.permutation(len(pts))[:max_points]
    p, c = pts[idx], cols[idx]

    go.Figure(
        data=[go.Scatter3d(x=p[:, 0], y=p[:, 1], z=p[:, 2], mode='markers',
                           marker=dict(size=1.5, color=[f'rgb({r},{g},{b})' for r, g, b in c]))],
        layout=go.Layout(
            title=title, margin=dict(l=0, r=0, b=0, t=40), height=500, showlegend=False,
            scene=dict(aspectmode='data', camera=dict(eye=dict(x=eye_pos[0], y=eye_pos[1], z=eye_pos[2])))
        )
    ).show(renderer="colab")

# ================= 主流程 =================
# 1. 优雅取用：直接从 camera 层级提取首个机位数据
cam_serial = list(scene_constants['camera'].keys())[0]
cam_data = scene_constants['camera'][cam_serial]

img_left, img_right = cam_data['video_rgb'][0], cam_data['video_right'][0]
K_mat, baseline = cam_data['K_mat'], cam_data['baseline']

# 🌟 2. 推断层：纯视差推断
disp_map = get_s2m2_disparity(img_left, img_right, device=device)

# 🌟 3. 跨界层：将全图视差一次性解码为附带物理单位的深度图
depth_map = decode_disparity_np(disp_map, fx=K_mat[0, 0], baseline=baseline)

# 直接传入有效区域的点阵，由于是单帧展示，默认 T_cam2world=None (在相机坐标系下)
pts, cols = unproject_to_3d(depth_map, img_left, K_mat)

show_plotly_point_cloud(pts, cols, title=f"Point Cloud (Cam: {cam_serial})", max_points=100000, eye_pos=(0, -0.5, -1.5))

In [ ]:
# @title SVO 视频深度提取与视差可视化

def compute_stereo_depth(scene_constants, device):
    """基于双目视频池执行 S2M2 推断，将原始视差转换为物理深度并封入常量池"""
    print("  🧠 启动 S2M2 深度网络推断...")
    for cam_id in scene_constants['camera']:
        cam_data = scene_constants['camera'][cam_id]
        left_seq, right_seq = cam_data['video_rgb'], cam_data['video_right']
        disp_frames = [
            get_s2m2_disparity(left_img, right_img, device=device)
            for left_img, right_img in tqdm(zip(left_seq, right_seq), total=len(left_seq), desc=f"Depth [{cam_id}]")
        ]
        raw_disp = np.stack(disp_frames)

        fx = cam_data['K_mat'][0, 0]
        baseline = cam_data['baseline']
        cam_data['raw_depth'] = decode_disparity_np(raw_disp, fx, baseline)

    return scene_constants

def visualize_disparity_video(disp_array, vmax=100.0):
    """向量化预处理 + 优雅推导式完成视差图染色"""
    disp_norm = (np.clip(disp_array, 0, vmax) / vmax * 255).astype(np.uint8)

    return np.stack([
        cv2.cvtColor(cv2.applyColorMap(frame, cv2.COLORMAP_MAGMA), cv2.COLOR_BGR2RGB)
        for frame in disp_norm
    ])

def render_multicam_disparity_video(scene_constants, tgt_size=(128, 228), disp_vmax=100.0):
    """一站式提取常量池数据，生成 [左 | 右 | 视差] 的多视角拼接视频"""
    camera_rows = []

    # 纯粹的遍历：直接迭代 camera 层级
    for cam_data in scene_constants['camera'].values():

        # 1. 语义复苏：提取并缩放左右眼视频画面
        left_video = media.resize_video(cam_data['video_rgb'], tgt_size)
        right_video = media.resize_video(cam_data['video_right'], tgt_size)

        # 🌟 2. 视差还原：从存储的 depth 中反向推导 disparity 用于可视化渲染
        raw_depth = cam_data['raw_depth'].astype(np.float32)
        fx = cam_data['K_mat'][0, 0]
        baseline = cam_data['baseline']

        # 反向物理公式: d = (fx * baseline) / Z
        raw_disp = np.zeros_like(raw_depth)
        valid_mask = raw_depth > 0
        raw_disp[valid_mask] = (fx * baseline) / raw_depth[valid_mask]

        # 缩放 -> 渲染染色
        disp_video = visualize_disparity_video(media.resize_video(raw_disp, tgt_size), vmax=disp_vmax)

        # 3. 画面拼接：横向连接单个机位的三重视角 (Width 维度)
        camera_rows.append(np.concatenate([left_video, right_video, disp_video], axis=2))

    # 终极组装：纵向堆叠所有机位的画面 (Height 维度)
    return np.concatenate(camera_rows, axis=1)

# ================= 主流程 =================
scene_constants = compute_stereo_depth(scene_constants, device)
final_video = render_multicam_disparity_video(scene_constants)
media.show_video(final_video, codec='gif')

In [ ]:
# @title 🎯 提取极品通用静态遮罩 (八爪鱼正负样本排爆 + 大画幅膨胀校准版)

# ================= 模块 1：单帧推断算子 =================
def extract_single_frame_mask(img_rgb, predictor):
    """基于广角极核与负样本排爆，使用 SAM 提取单帧夹爪遮罩"""
    h, w = img_rgb.shape[:2]

    # 🌟 核心重构：膨胀 1.4 倍！以 w//2 + 100 为中心，全面向四周张开大网
    points = np.array([
        [w//2 - 120, h - 110],  # 🟢 左指根 (向左向外推)
        [w//2 + 500, h - 110],  # 🟢 右指根 (向右向外推)
        [w//2 - 250, h - 25],   # 🟢 左下角基座 (向左外延)
        [w//2 + 450, h - 25],   # 🟢 右下角基座 (向右外延)
        [w//2 + 100, h - 15],   # 🟢 真正的基座正中间 (中心锚点保持不变)
        [w//2 + 100, h - 300],  # 🔴 两指缝隙空隙 (负样本：随夹爪变大，大幅向上方推，防止误伤内侧)
    ])

    # 1 代表前景, 0 代表背景
    labels = np.array([1, 1, 1, 1, 1, 0])

    # BBox 上沿定在 h // 2
    bbox = np.array([0, h // 2, w, h])

    predictor.set_image(img_rgb)
    masks, scores, _ = predictor.predict(
        point_coords=points, point_labels=labels, box=bbox, multimask_output=True
    )

    valid_masks = []
    for m, s in zip(masks, scores):
        area_ratio = np.sum(m) / (w * h)
        if 0.02 < area_ratio < 0.45:
            valid_masks.append((m, s * area_ratio))

    if valid_masks:
        best_mask = max(valid_masks, key=lambda x: x[1])[0]
    else:
        best_mask = masks[np.argmax(scores)]

    return best_mask, bbox, points, labels

# ================= 模块 2：多帧共识与清洗算子 =================
def compute_consensus_mask(masks_list, consensus_thresh=0.5):
    vote_map = np.mean(masks_list, axis=0)
    consensus_mask = vote_map >= consensus_thresh

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(consensus_mask.astype(np.uint8))
    if num_labels > 1:
        largest_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
        consensus_mask = (labels == largest_label)

    return consensus_mask, vote_map

# ================= 模块 3：优雅的可视化渲染 =================
def visualize_consensus_result(ref_img, vote_map, final_mask, bbox, points, labels, num_frames):
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    fig.suptitle(f"Robust Consensus Mask (Aggregated from {num_frames} Frames)", fontsize=14, fontweight='bold')

    im = axes[0].imshow(vote_map, cmap='magma')
    axes[0].set_title("Pixel-wise Voting Heatmap")
    axes[0].axis('off')
    fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)

    blended = ref_img.copy()
    blended[final_mask] = [50, 255, 50]
    blended = cv2.addWeighted(ref_img, 0.6, blended, 0.4, 0)
    cv2.rectangle(blended, (bbox[0], bbox[1]), (bbox[2], bbox[3]), (0, 255, 255), 2)

    for p, l in zip(points, labels):
        cv2.circle(blended, (p[0], p[1]), 8, (255, 255, 255), -1)
        color = (0, 255, 0) if l == 1 else (255, 0, 0)
        cv2.circle(blended, (p[0], p[1]), 5, color, -1)

    axes[1].imshow(blended)
    axes[1].set_title("Final Universal Mask & Expanded Prompts")
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()

# ================= 模块 4：核心调度主干 =================
def build_universal_gripper_mask(scene_constants, predictor):
    cam_data = scene_constants['camera'][scene_constants['meta']['wrist_serial']]
    gripper_states = scene_constants['robot']['gripper_positions']

    closed_indices = np.where(gripper_states < 0.05)[0]
    if len(closed_indices) == 0:
        print("⚠️ 该片段无夹爪完全闭合帧，放弃提取。")
        return scene_constants

    print(f"🚀 启动提纯：基于 {len(closed_indices)} 帧闭合画面生成共识掩码...")

    masks_list, bbox, points, labels = [], None, None, None
    for idx in tqdm(closed_indices, desc="SAM 逐帧抠图"):
        img = cam_data['video_rgb'][idx].copy()
        mask, bbox, points, labels = extract_single_frame_mask(img, predictor)
        masks_list.append(mask)

    final_mask, vote_map = compute_consensus_mask(masks_list)

    if 'sam_real_masks' not in cam_data:
        cam_data['sam_real_masks'] = np.zeros((len(gripper_states), *final_mask.shape), dtype=bool)
    cam_data['sam_real_masks'][closed_indices] = final_mask
    print("✅ 广播完成！所有闭合帧现已共享完美的 [多帧共识] 静态遮罩。")

    ref_img = cam_data['video_rgb'][closed_indices[0]].copy()
    visualize_consensus_result(ref_img, vote_map, final_mask, bbox, points, labels, len(closed_indices))

    return scene_constants

# ================= 🚀 一键启动 =================
scene_constants = build_universal_gripper_mask(scene_constants, sam_predictor)

In [ ]:
# @title 🎯 夹爪表面点云可视化

def render_distilled_gripper_3d(median_depth, K_mat, rgb_img):
    v, u = np.where(median_depth > 0)
    z = median_depth[v, u]
    x = (u - K_mat[0, 2]) * z / K_mat[0, 0]
    y = (v - K_mat[1, 2]) * z / K_mat[1, 1]

    pts_3d = np.stack([x, y, z], axis=-1)
    fig = go.Figure(data=[go.Scatter3d(
        x=pts_3d[:, 0], y=pts_3d[:, 1], z=pts_3d[:, 2],
        mode='markers', marker=dict(size=2, color=rgb_img[v, u], opacity=0.8)
    )])
    fig.update_layout(
        title="Distilled Gripper Surface 🦾",
        scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Depth (Z)', aspectmode='data',
                   camera=dict(eye=dict(x=0, y=-0.5, z=-1.5), up=dict(x=0, y=-1, z=0))),
        margin=dict(l=0, r=0, b=0, t=40)
    )
    fig.show()

In [ ]:
# @title 🎯 时序深度积分 (无 Mask 暴力提纯 + 方差感知过滤版)

import warnings

def distill_empirical_gripper_depth_mad_aware(scene_constants, max_depth_thresh=1.5, max_mad_thresh=0.005, min_valid_frames=10):
    """
    使用极其强悍的 MAD (中位数绝对偏差) 替代脆弱的 Variance
    - max_mad_thresh: 0.005 米 (即 5 毫米)。意味着我们要求该像素在大部分时间里，深度波动不超过 5 毫米！
    """
    wrist_cam = scene_constants['meta']['wrist_serial']
    cam_data = scene_constants['camera'][wrist_cam]
    gripper_states = scene_constants['robot']['gripper_positions']

    closed_indices = np.where(gripper_states < 0.05)[0]
    if len(closed_indices) == 0:
        return None, None, None

    h, w = cam_data['video_rgb'][0].shape[:2]
    num_frames = len(closed_indices)

    print(f"🧠 启动 MAD 感知时序积分：正在构建 {num_frames} 帧容量的全画幅储蓄罐...")
    depth_bank = np.full((num_frames, h, w), np.nan, dtype=np.float32)

    for i, idx in enumerate(tqdm(closed_indices, desc="📥 收集有效深度像素")):
        raw_depth = cam_data['raw_depth'][idx].astype(np.float32)
        valid_pixels = (raw_depth > 0.01) & (raw_depth < max_depth_thresh)
        depth_bank[i, valid_pixels] = raw_depth[valid_pixels]

    print("🔨 正在计算鲁棒统计特征 (中位数与 MAD)...")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)

        # 1. 计算全局中位数深度
        median_depth = np.nanmedian(depth_bank, axis=0)

        # 🌟 核心升级：计算 MAD (中位数绝对偏差)
        # 偏差矩阵 = | 每一帧深度 - 中位数深度 |
        abs_deviation = np.abs(depth_bank - median_depth)
        # MAD = 偏差矩阵的中位数
        mad_map = np.nanmedian(abs_deviation, axis=0)

    valid_counts = np.sum(~np.isnan(depth_bank), axis=0)

    # 🌟 绝杀：利用 MAD 过滤动态背景与散粒噪点
    # 5毫米的 MAD 容忍度，足以包容 S2M2 网络正常的亚像素抖动，同时无情击杀背景
    reliable_pixels = (mad_map < max_mad_thresh) & (valid_counts >= min_valid_frames)

    print(f"  🔪 MAD 大刀砍掉了 {np.sum(~reliable_pixels & (valid_counts > 0))} 个不稳定的动态背景像素！")

    # 将不靠谱的像素直接抹零
    median_depth[~reliable_pixels] = np.nan

    median_depth = np.nan_to_num(median_depth, nan=0.0).astype(np.float32)
    mad_map = np.nan_to_num(mad_map, nan=0.0).astype(np.float32)

    print("✅ MAD 感知蒸馏完成！")
    return median_depth, valid_counts, mad_map

# ================= 启动运行 =================
wrist_cam = scene_constants['meta']['wrist_serial']
cam_data = scene_constants['camera'][wrist_cam]

# 调用时，注意将阈值修改为线性的物理距离（比如 0.005 米），而不是之前方差的平方单位！
median_depth_map, valid_counts_map, mad_map = distill_empirical_gripper_depth_mad_aware(
    scene_constants, max_depth_thresh=1.5, max_mad_thresh=0.001, min_valid_frames=10
)

if median_depth_map is not None:
    # --- 2D 可视化验证大盘 (4图联动) ---
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    ref_rgb = cam_data['video_rgb'][np.where(scene_constants['robot']['gripper_positions'] < 0.05)[0][0]]

    axes[0].imshow(ref_rgb)
    axes[0].set_title("Reference RGB")
    axes[0].axis('off')

    im1 = axes[1].imshow(valid_counts_map, cmap='viridis')
    axes[1].set_title("Valid Depth Count")
    axes[1].axis('off')
    fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

    # 🌟 修复：改为可视化 MAD 热力图！(单位为米 m，vmax 设为 0.01 即 1 厘米，用来凸显噪点分布)
    im2 = axes[2].imshow(mad_map, cmap='magma', vmax=0.01)
    axes[2].set_title("Temporal MAD Map (m)")
    axes[2].axis('off')
    fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

    # 🌟 修复标题
    im3 = axes[3].imshow(median_depth_map, cmap='plasma')
    axes[3].set_title("MAD-Filtered Depth")
    axes[3].axis('off')
    fig.colorbar(im3, ax=axes[3], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()

    # --- 3D 升维验证 ---
    print("🚀 正在渲染【MAD 过滤后】的 3D 点云表面...")
    render_distilled_gripper_3d(median_depth_map, cam_data['K_mat'], ref_rgb)

    # 将完美的深度图存回常量库，供后续注入原始视频流使用
    cam_data['empirical_gripper_depth'] = median_depth_map

In [ ]:
# @title 🎯 时序深度积分 (提取纯真实物理数据的夹爪表面)

import warnings

def distill_empirical_gripper_depth(scene_constants, max_depth_thresh=0.15):
    """
    基于时序中位数，从数千帧含噪深度中“蒸馏”出完美的夹爪表面
    """
    wrist_cam = scene_constants['meta']['wrist_serial']
    cam_data = scene_constants['camera'][wrist_cam]
    gripper_states = scene_constants['robot']['gripper_positions']

    # 1. 物理状态锁定
    closed_indices = np.where(gripper_states < 0.05)[0]
    if len(closed_indices) == 0:
        print("⚠️ 未找到夹爪闭合帧，放弃提取。")
        return None

    if 'sam_real_masks' not in cam_data:
        print("⚠️ 找不到 sam_real_masks，请确保上一节的通用遮罩提取已运行！")
        return None

    h, w = cam_data['video_rgb'][0].shape[:2]
    num_frames = len(closed_indices)

    print(f"🧠 启动时序深度积分：正在构建 {num_frames} 帧容量的像素级储蓄罐...")

    # 🌟 2. 构建 3D 储蓄罐 (使用 NaN 初始化，这是核心技巧，方便后续直接忽略无效值)
    depth_bank = np.full((num_frames, h, w), np.nan, dtype=np.float32)

    for i, idx in enumerate(tqdm(closed_indices, desc="📥 收集有效深度像素")):
        raw_depth = cam_data['raw_depth'][idx].astype(np.float32)
        mask = cam_data['sam_real_masks'][idx]

        # 🌟 核心物理截断法则：
        # 1. 必须在完美的 Mask 内部
        # 2. 必须有深度值 (> 0.01)
        # 3. 必须符合夹爪的物理极限距离 (< max_depth_thresh，例如 15cm)
        valid_pixels = (mask > 0) & (raw_depth > 0) & (raw_depth < max_depth_thresh)

        # 把有效的深度塞进储蓄罐
        depth_bank[i, valid_pixels] = raw_depth[valid_pixels]

    # 🌟 3. 提取时序中位数 (极强抗噪)
    print("🔨 正在对数十万像素进行中位数提纯 (过滤离群的黑色拉丝)...")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        # np.nanmedian 会聪明地忽略掉储蓄罐里的 NaN，只对有效的收集值求中位数
        median_depth = np.nanmedian(depth_bank, axis=0)

    # 计算一下每个像素收集到了多少帧的有效数据，这叫“置信度图”
    valid_counts = np.sum(~np.isnan(depth_bank), axis=0)

    # 把全是 NaN 的无效像素填回 0.0
    median_depth = np.nan_to_num(median_depth, nan=0.0).astype(np.float32)

    print("✅ 纯真实数据蒸馏完成！")
    return median_depth, valid_counts

# ================= 启动运行 =================
wrist_cam = scene_constants['meta']['wrist_serial']
cam_data = scene_constants['camera'][wrist_cam]

# 执行蒸馏 (物理截断距离设为 0.15 米)
median_depth_map, valid_counts_map = distill_empirical_gripper_depth(scene_constants, max_depth_thresh=0.15)

if median_depth_map is not None:
    # --- 2D 可视化验证大盘 ---
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    ref_rgb = cam_data['video_rgb'][np.where(scene_constants['robot']['gripper_positions'] < 0.05)[0][0]]

    axes[0].imshow(ref_rgb)
    axes[0].set_title("Reference RGB")
    axes[0].axis('off')

    # 热力图展示每个像素被成功估算深度的次数
    im1 = axes[1].imshow(valid_counts_map, cmap='viridis')
    axes[1].set_title("Valid Depth Count per Pixel")
    axes[1].axis('off')
    fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

    # 最终的提纯深度图
    im2 = axes[2].imshow(median_depth_map, cmap='plasma')
    axes[2].set_title("Distilled Median Depth")
    axes[2].axis('off')
    fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()

    # --- 3D 升维验证 ---
    print("🚀 正在渲染提纯后的 3D 点云表面...")
    render_distilled_gripper_3d(median_depth_map, cam_data['K_mat'], ref_rgb)

    # 你可以选择将这个完美的深度图存回常量库，供后续使用
    cam_data['empirical_gripper_depth'] = median_depth_map

In [ ]:
# @title 🧬 将提纯夹爪深度无缝注入原始视频流

wrist_cam = scene_constants['meta']['wrist_serial']
cam_data = scene_constants['camera'][wrist_cam]
gripper_states = scene_constants['robot']['gripper_positions']

# 提取我们刚才保存的终极完美版深度 (请确保字典 key 与你上一步保存的一致)
empirical_depth = cam_data.get('empirical_gripper_depth')

if empirical_depth is None:
    print("⚠️ 找不到提纯后的夹爪深度，请确认你已经运行了上一步！")
else:
    # 1. 物理状态锁定：只针对夹爪闭合的帧
    closed_indices = np.where(gripper_states < 0.05)[0]

    # 2. 空间屏蔽锁定：只替换我们确信是夹爪本体的像素
    valid_mask = empirical_depth > 0

    print(f"💉 启动深度注入手术：准备覆写 {len(closed_indices)} 帧数据...")

    # 3. 核心广播魔法：用 np.where 直接在原数组内存里“瞒天过海”
    # 如果 valid_mask 为 True，填入我们提纯的 empirical_depth，否则保留原先的 raw_depth
    cam_data['raw_depth'][closed_indices] = np.where(
        valid_mask,
        empirical_depth,
        cam_data['raw_depth'][closed_indices]
    )

    replaced_pixels_per_frame = np.sum(valid_mask)
    total_replaced = len(closed_indices) * replaced_pixels_per_frame
    print(f"✅ 注入完成！共计 {total_replaced} 个含噪深度像素被完美的物理真理替换。")

final_video = render_multicam_disparity_video(scene_constants)
media.show_video(final_video, codec='gif')

### 相机外參标定

In [ ]:
# @title 3D 单帧融合点云与交互式可视化

def render_fused_point_cloud(scene_constants, scene_state, frame_idx=0,
                             max_render_points=150000, eye_pos=(-1.2, -1.2, 0.8), use_tint=False):

    # 🌟 1. 拥抱新架构：直接提取 camera 层级的 key，并利用 sorted 保证多机位合并顺序的绝对一致性
    camera_ids = sorted(scene_constants['camera'].keys())

    # 🌟 2. 调色板保留：严格保留原版染色矩阵 (绿、红、蓝)，用于 Debug 模式下区分机位
    tint_colors = np.array([[0, 50, 0], [50, 0, 0], [0, 0, 50]])

    fused_points, fused_colors = [], []

    # 🌟 3. 语义复苏遍历：使用清晰的命名
    for idx, cam_id in enumerate(camera_ids):
        cam_data = scene_constants['camera'][cam_id]
        cam_state = scene_state[cam_id]

        # 🌟 直接提取原始深度图 (彻底剥离 scale 和 shift)
        raw_depth = cam_data['raw_depth'][frame_idx].astype(np.float32)

        # 🌟 直接传入 raw_depth 提取 3D 点 (依赖 unproject_to_3d 内部的深度范围截断)
        points_3d, colors_rgb = unproject_to_3d(
            raw_depth,
            cam_data['video_rgb'][frame_idx],
            cam_data['K_mat'],
            T_cam2world=cam_state['extrinsics'][frame_idx]
        )

        # 染色处理
        if use_tint:
            colors_rgb = np.clip(colors_rgb.astype(int) + tint_colors[idx % len(tint_colors)], 0, 255).astype(np.uint8)

        fused_points.append(points_3d)
        fused_colors.append(colors_rgb)

    # 🌟 4. 极致清爽的堆叠与渲染
    show_plotly_point_cloud(
        pts=np.vstack(fused_points),
        cols=np.vstack(fused_colors),
        title=f"Fused Point Cloud (Frame {frame_idx})" + (" 🎨 [Tinted Debug Mode]" if use_tint else ""),
        max_points=max_render_points,
        eye_pos=eye_pos
    )

In [ ]:
# @title 2D Mask 全视场联合体检函数

import inspect

def render_multiview_mask_inspection(scene_constants, scene_state, pb_renderer, frame_idx=0):
    """
    全视场三联屏监视器 (支持新老两代 Renderer 无缝切换 + 数据结构极致解耦版)
    """
    camera_ids = list(scene_constants['camera'].keys())
    wrist_cam = scene_constants['meta']['wrist_serial']

    # 🌟 1. 提取姿态数据
    joint_angles = scene_constants['robot']['joint_positions'][frame_idx]
    gripper_state = scene_constants['robot']['gripper_positions'][frame_idx]

    # 🌟 2. 智能路由 A：检测引擎是否支持控制夹爪
    sig = inspect.signature(pb_renderer.update_robot_pose)
    pb_renderer.update_robot_pose(joint_angles, gripper_state=gripper_state)

    # 开启多联屏画布
    fig, axes = plt.subplots(1, len(camera_ids), figsize=(12, 3))
    if len(camera_ids) == 1: axes = [axes]

    fig.suptitle(f"Multi-View Segmentation Mask Inspection (Frame {frame_idx})",
                 fontsize=20, fontweight='bold', y=1.05)

    for i, cam_id in enumerate(camera_ids):

        # 🌟 核心进化：彻底干掉 if/else 特判！
        # 无论你是环境相机还是腕部相机，直接从统一大盘中抽出当前帧的 4x4 绝对位姿
        extrinsics = scene_state[cam_id]['extrinsics'][frame_idx]

        intrinsics = scene_constants['camera'][cam_id]['K_mat']
        img_rgb = scene_constants['camera'][cam_id]['video_rgb'][frame_idx].copy()
        h_img, w_img = img_rgb.shape[:2]

        # 🌟 3. 智能路由 B：检测引擎用的是哪个渲染方法
        robot_mask = pb_renderer.render_mask(extrinsics, intrinsics, w_img, h_img) > 0

        # 亮绿色半透明叠加
        overlay = img_rgb.copy()
        overlay[robot_mask] = [50, 255, 50]
        blended_img = cv2.addWeighted(img_rgb, 0.6, overlay, 0.4, 0)

        # 绘制
        cam_type = "Wrist Camera" if cam_id == wrist_cam else "External Camera"
        axes[i].imshow(blended_img)
        axes[i].set_title(f"[{cam_type}]\nCam ID: {cam_id}", fontsize=15)
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
# @title 通用数字孪生影棚

import importlib.util
import numpy as np
import pybullet as p
import pybullet_data

# ==========================================
# 🌟 2. 极致美学的双生替身物理影棚 (完美镜像接骨版)
# ==========================================
class PyBulletRenderer_Robotiq:
    def __init__(self, ghost_urdf="PointWorld/assets/franka_description/franka_panda_robotiq_2f85_og.urdf"):
        """🌟 极致美学的双生替身物理影棚 (满血修复物理连杆与左右镜像)"""
        if p.isConnected(): p.disconnect()
        p.connect(p.DIRECT)
        p.setAdditionalSearchPath(pybullet_data.getDataPath())

        if importlib.util.find_spec('eglRendererPlugin'):
            p.loadPlugin(importlib.util.find_spec('eglRendererPlugin').origin, "_eglRendererPlugin")

        # ------------------------------------------
        # 实体 (The Body)：原厂极瘦机械臂
        # ------------------------------------------
        self.robot_id = p.loadURDF("franka_panda/panda.urdf", useFixedBase=True)
        self.arm_joints = [i for i in range(p.getNumJoints(self.robot_id)) if "panda_joint" in p.getJointInfo(self.robot_id, i)[1].decode('utf-8') and p.getJointInfo(self.robot_id, i)[2] != p.JOINT_FIXED]

        self.hidden_robot_links = []
        for i in range(-1, p.getNumJoints(self.robot_id)):
            name = p.getBodyInfo(self.robot_id)[0].decode('utf-8') if i == -1 else p.getJointInfo(self.robot_id, i)[12].decode('utf-8')
            if "hand" in name or "finger" in name:
                p.changeVisualShape(self.robot_id, i, rgbaColor=[0, 0, 0, 0])
                self.hidden_robot_links.append(i)

        # ------------------------------------------
        # 替身 (The Ghost)：携带夹爪的 PointWorld 机械臂
        # ------------------------------------------
        self.ghost_id = p.loadURDF(ghost_urdf, useFixedBase=True)
        self.ghost_arm_joints = [i for i in range(p.getNumJoints(self.ghost_id)) if "panda_joint" in p.getJointInfo(self.ghost_id, i)[1].decode('utf-8') and p.getJointInfo(self.ghost_id, i)[2] != p.JOINT_FIXED]

        # 🌟 修复核心：精确识别连杆层级与左右侧运动学镜像
        self.gripper_joints = []
        self.gripper_signs = []

        for i in range(p.getNumJoints(self.ghost_id)):
            info = p.getJointInfo(self.ghost_id, i)
            joint_name = info[1].decode('utf-8')
            joint_type = info[2]

            if joint_type != p.JOINT_FIXED and "panda_joint" not in joint_name:
                self.gripper_joints.append(i)

                # 🌟 骨科手术 2.0：应对右侧手指的反向坐标轴
                base_sign = -1 if "right" in joint_name else 1

                # 💥 致命穿模 Bug 修复：加入了 "finger_tip"，保持平行四边形机构！
                if "inner_finger" in joint_name or "follower" in joint_name or "finger_tip" in joint_name:
                    self.gripper_signs.append(base_sign * -1)
                else:
                    self.gripper_signs.append(base_sign)

        self.hidden_ghost_links = []
        for i in range(-1, p.getNumJoints(self.ghost_id)):
            name = p.getBodyInfo(self.ghost_id)[0].decode('utf-8') if i == -1 else p.getJointInfo(self.ghost_id, i)[12].decode('utf-8')
            if "panda_link" in name:
                p.changeVisualShape(self.ghost_id, i, rgbaColor=[0, 0, 0, 0])
                self.hidden_ghost_links.append(i)

    def _get_projection_matrix(self, intrinsics, width, height):
        fx, fy, cx, cy = intrinsics[0, 0], intrinsics[1, 1], intrinsics[0, 2], intrinsics[1, 2]
        near, far = 0.01, 10.0
        return [2.0 * fx / width, 0.0, 0.0, 0.0,
                0.0, 2.0 * fy / height, 0.0, 0.0,
                1.0 - 2.0 * cx / width, 2.0 * cy / height - 1.0, (far + near) / (near - far), -1.0,
                0.0, 0.0, 2.0 * far * near / (near - far), 0.0]

    def update_robot_pose(self, joint_angles, gripper_state=None, gripper_width_offset=0.08):
        for i, angle in zip(self.arm_joints, joint_angles):
            p.resetJointState(self.robot_id, i, angle)
        for i, angle in zip(self.ghost_arm_joints, joint_angles):
            p.resetJointState(self.ghost_id, i, angle)

        if gripper_state is not None and len(self.gripper_joints) > 0:
            raw_val = gripper_state[0] if isinstance(gripper_state, (list, np.ndarray)) else gripper_state

            # 🌟 回归正道的正向映射！
            # DROID 数据: 0 (完全张开) -> 1 (完全闭合)
            # Robotiq 85: 0 (完全张开) -> 0.8028 (完全闭合)
            raw_val = np.clip(raw_val, 0.0, 1.0)
            max_urdf_radian = 0.8028

            # 🌟 核心：强行减去 offset，突破原厂物理极限让夹爪外扩
            angle = (raw_val * max_urdf_radian) - gripper_width_offset

            for i, sign in zip(self.gripper_joints, self.gripper_signs):
                p.resetJointState(self.ghost_id, i, angle * sign)

        # 🌟 强制刷新渲染网格
        p.performCollisionDetection()

    def render_depth(self, extrinsics, intrinsics, width, height):
        cam_pos, target_pos = extrinsics[:3, 3], extrinsics[:3, 3] + extrinsics[:3, 2]
        view_matrix = p.computeViewMatrix(cam_pos, target_pos, -extrinsics[:3, 1])
        proj_matrix = self._get_projection_matrix(intrinsics, width, height)
        _, _, _, depth_buffer, _ = p.getCameraImage(
            width, height, viewMatrix=view_matrix, projectionMatrix=proj_matrix,
            renderer=p.ER_BULLET_HARDWARE_OPENGL
        )
        metric_depth = 0.1 / (10.0 - 9.99 * np.reshape(depth_buffer, (height, width)))
        return np.where(metric_depth < 9.9, metric_depth, 0.0)

    def render_mask(self, extrinsics, intrinsics, width, height):
        cam_pos, target_pos = extrinsics[:3, 3], extrinsics[:3, 3] + extrinsics[:3, 2]
        view_matrix = p.computeViewMatrix(cam_pos, target_pos, -extrinsics[:3, 1])
        proj_matrix = self._get_projection_matrix(intrinsics, width, height)
        _, _, _, _, seg_buffer = p.getCameraImage(
            width, height, viewMatrix=view_matrix, projectionMatrix=proj_matrix,
            renderer=p.ER_BULLET_HARDWARE_OPENGL, flags=p.ER_SEGMENTATION_MASK_OBJECT_AND_LINKINDEX
        )
        seg_array = np.reshape(seg_buffer, (height, width)).astype(np.int32)
        obj_ids, link_ids = seg_array & 0xFFFFFF, (seg_array >> 24) - 1
        valid_robot = (obj_ids == self.robot_id) & ~np.isin(link_ids, self.hidden_robot_links)
        valid_ghost = (obj_ids == self.ghost_id) & ~np.isin(link_ids, self.hidden_ghost_links)
        return valid_robot | valid_ghost

In [ ]:
# @title 第零阶段：读取数据集外參（如果存在的话）

def init_camera_states(scene_constants, extrinsics_db):
    """组装多机位初始 3D 物理状态 (全局字典传参 + 内部安全路由版)"""
    print("  🌐 初始化全场相机 3D 物理状态...")
    wrist_serial = scene_constants['meta']['wrist_serial']
    robot_data = scene_constants['robot']
    n_frames = len(robot_data['T_ee_base_all'])

    # 🌟 内部安全提取：直接从全局字典中摸取当前 Episode 的外参
    # 如果该 Episode 是那 5 万个没有外参的数据之一，则优雅地返回空字典 {}
    episode_id = scene_constants['meta']['episode_id']
    episode_extrinsics = extrinsics_db.get(episode_id, {})

    scene_state = {}

    # 🌟 纯粹的遍历：直接迭代 camera 层级的 keys，彻底干掉丑陋的 if 过滤
    for cam_id in scene_constants['camera'].keys():

        if cam_id == wrist_serial:
            # 🦾 腕部相机：不随时间变化的量是【初始手眼标定矩阵】(4x4)
            base_ext = robot_data['T_cam_ee_init']
            cam_trajectory = robot_data['T_ee_base_all'] @ base_ext
        elif cam_id in episode_extrinsics:
            ext_data = episode_extrinsics[cam_id]
            ext_vec = ext_data.get('extrinsics', ext_data) if isinstance(ext_data, dict) else ext_data

            # 📷 环境相机：不随时间变化的量是【静态外参矩阵】(4x4)
            base_ext = make_4x4(ext_vec)
            cam_trajectory = np.tile(base_ext, (n_frames, 1, 1))
        else:
            print(f"    ⚠️ 未找到环境相机 [{cam_id}] 的预标定外参，初始化为 None。")
            base_ext = None
            cam_trajectory = None

        # 🌟 极致清爽的字典组装：既保存物理本质(base_extrinsic)，又保留供渲染器调用的时序轨迹(extrinsics)
        scene_state[cam_id] = {
            'base_extrinsic': base_ext,     # <--- 核心新增：提纯出来的 4x4 静态变量，专门喂给后续的优化器
            'extrinsics': cam_trajectory,   # 保留 (N, 4, 4) 轨迹，保证下游渲染和重投影代码无需修改
        }

    return scene_state

# 🌟 调用时直接传入整个全局 extrinsics_db，彻底告别 KeyError！
init_scene_state = init_camera_states(scene_constants, extrinsics_db)

# 🌟 智能路由：检查是否所有相机的初始外参都已存在
all_extrinsics_exist = all(state['extrinsics'] is not None for state in init_scene_state.values())

if all_extrinsics_exist:
    # 统一使用 init_scene_state 进行后续渲染（哪怕外参是 None，后续有保护机制就不会崩）
    render_fused_point_cloud(
        scene_constants=scene_constants,
        scene_state=init_scene_state,
        use_tint=True
    )

    pb_renderer_ultimate = PyBulletRenderer_Robotiq()

    render_multiview_mask_inspection(
        scene_constants=scene_constants,
        scene_state=init_scene_state,
        pb_renderer=pb_renderer_ultimate,
    )
else:
  print("  ✅ 没有检测到官方完整的外参真值，跳过 init_scene_state 渲染！")

In [ ]:
# @title 第一阶段：VGGT 视觉物理链式锚定 (仅使用第1帧极速静态版)

def vggt_warmup_extrinsics(scene_constants): # 🗑️ 彻底砍掉 scene_state 参数输入
    print(f"\n🌍 启动 VGGT 物理锚定：仅抽取第1帧推导绝对位姿...")
    wrist_serial = scene_constants['meta']['wrist_serial']
    ext_cams = [cam for cam in scene_constants['camera'].keys() if cam != wrist_serial]

    ref_cam = ext_cams[0]
    other_cams = ext_cams[1:]

    # 🌟 1. 破茧重生：直接在这里初始化全新的全局状态大盘！
    new_scene_state = {}
    robot_data = scene_constants['robot']
    n_frames_total = len(scene_constants['camera'][ref_cam]['video_rgb'])

    # 🌟 2. 顺手把腕部相机 (Wrist) 的全时序运动学轨迹算出来，奠定物理基石
    new_scene_state[wrist_serial] = {
        'base_extrinsic': robot_data['T_cam_ee_init'], # <--- 新增：保存静态手眼矩阵作为基底
        'extrinsics': robot_data['T_ee_base_all'] @ robot_data['T_cam_ee_init'],
    }

    # === 取消循环，仅提取第1帧 [0] 进行全量推理 ===
    print("  📸 正在提取多视角图像的第1帧并送入大模型...")

    # 1. 组装输入序列: [Ref] + [Other_Exts...] + [Wrist] (全部取第1帧 [0])
    img_ref = scene_constants['camera'][ref_cam]['video_rgb'][0]
    img_others = [scene_constants['camera'][cam]['video_rgb'][0] for cam in other_cams]
    img_wrist = scene_constants['camera'][wrist_serial]['video_rgb'][0]

    img_list = [img_ref] + img_others + [img_wrist]

    # 2. 一次推理，拿到所有相对位姿！
    T_rel_list = estimate_multi_camera_vggt(img_list)

    T_ref_to_others = T_rel_list[:-1]
    T_ref_to_wrist = T_rel_list[-1]

    # 3. 获取腕部相机在第1帧的绝对物理位姿 (GT)
    T_ee_base_first = scene_constants['robot']['T_ee_base_all'][0]
    T_cam_ee = scene_constants['robot']['T_cam_ee_init']
    T_wrist_to_base_first = T_ee_base_first @ T_cam_ee

    # 4. 终极链式法则解算绝对位姿
    T_ref_to_base = T_wrist_to_base_first @ T_ref_to_wrist

    # 🌟 3. 将计算出的静态外参直接平铺压入全新的状态字典 (同时剥离并保存 4x4 的 base_extrinsic)
    new_scene_state[ref_cam] = {
        'base_extrinsic': T_ref_to_base, # <--- 新增：保存纯粹的静态基础位姿
        'extrinsics': np.tile(T_ref_to_base, (n_frames_total, 1, 1)),
    }

    for tgt_cam, T_ref_to_tgt in zip(other_cams, T_ref_to_others):
        T_tgt_to_base = T_ref_to_base @ np.linalg.inv(T_ref_to_tgt)
        new_scene_state[tgt_cam] = {
            'base_extrinsic': T_tgt_to_base, # <--- 新增：保存纯粹的静态基础位姿
            'extrinsics': np.tile(T_tgt_to_base, (n_frames_total, 1, 1)),
        }

    print("  ✅ 极速物理锚定完成！")
    return new_scene_state

# 🌟 2. 核心拦截：在组装 3D 状态之前，用 VGGT 链式法则彻底洗掉垃圾外参！
vggt_scene_state = vggt_warmup_extrinsics(scene_constants)

In [ ]:
# @title 第二阶段：外部相机-机械臂本体精准对齐 (GPU 全局批处理终极美学版)

def axis_angle_to_matrix(rot_vec):
    """优雅可导的 Rodrigues 旋转公式 (引入 torch.where 极简分支)"""
    theta2 = torch.sum(rot_vec ** 2)
    theta = torch.sqrt(theta2 + 1e-16)
    k = rot_vec / theta

    # 利用反对称矩阵特性，直接进行优雅赋值
    K = torch.zeros((3, 3), device=rot_vec.device)
    K[0, 1], K[0, 2] = -k[2], k[1]
    K[1, 0], K[1, 2] =  k[2], -k[0]
    K[2, 0], K[2, 1] = -k[1],  k[0]

    R_exact = torch.eye(3, device=rot_vec.device) + torch.sin(theta)*K + (1-torch.cos(theta))*torch.mm(K, K)

    # 泰勒展开近似 (应对极小角度的奇异点)
    K_approx = torch.zeros_like(K)
    K_approx[0, 1], K_approx[0, 2] = -rot_vec[2], rot_vec[1]
    K_approx[1, 0], K_approx[1, 2] =  rot_vec[2], -rot_vec[0]
    K_approx[2, 0], K_approx[2, 1] = -rot_vec[1],  rot_vec[0]
    R_approx = torch.eye(3, device=rot_vec.device) + K_approx

    # 🌟 极简张量条件分支，抛弃算术遮罩
    return torch.where(theta2 < 1e-8, R_approx, R_exact)

def get_foreground_robot_points(T_init, K, obs_depth, pb_renderer, max_pts):
    """极速机器人点云提取 (引入 Pythonic 的采样布尔魔法)"""
    h_img, w_img = obs_depth.shape
    render_d = pb_renderer.render_depth(T_init, K, w_img, h_img)

    v_r, u_r = np.where(render_d > 0)
    z_r = render_d[v_r, u_r]

    if len(z_r) < max_pts: return None

    P_cam_r = np.stack([(u_r - K[0, 2]) * z_r / K[0, 0], (v_r - K[1, 2]) * z_r / K[1, 1], z_r, np.ones_like(z_r)])
    pts_robot_world = (T_init @ P_cam_r)[:3, :].T

    # 🌟 绝杀！利用动态布尔值 replace=(...)，一行代码搞定所有逻辑判定
    idx = np.random.choice(len(pts_robot_world), max_pts, replace=(len(pts_robot_world) < max_pts))
    return torch.tensor(pts_robot_world[idx], dtype=torch.float32, device=device)

def compute_robot_loss_batched(batch_X, T_opt, K, batch_obs):
    """榨干 GPU 算力的全局 Batched 纯水管算子 (无中间冗余变量)"""
    B, _, h_img, w_img = batch_obs.shape

    P_c = (batch_X - T_opt[:3, 3]) @ T_opt[:3, :3]
    Z_pred = P_c[..., 2]

    u = K[0, 0] * P_c[..., 0] / Z_pred + K[0, 2]
    v = K[1, 1] * P_c[..., 1] / Z_pred + K[1, 2]

    # 🌟 一气呵成：计算归一化坐标并直接堆叠送入网格
    grid = torch.stack([(u / (w_img - 1)) * 2 - 1, (v / (h_img - 1)) * 2 - 1], dim=-1).unsqueeze(1)
    Z_obs_raw = F.grid_sample(batch_obs, grid, mode='bilinear', padding_mode='border', align_corners=True).squeeze(1).squeeze(1)

    # 🌟 大一统联合物理遮罩：将内外边界判定全部融为一体
    valid_mask = (Z_pred > 0.) & (Z_pred < 1.5) & (Z_obs_raw > 0.) & (Z_obs_raw < 1.5) & \
                 (u >= 0) & (u < w_img - 1) & (v >= 0) & (v < h_img - 1)

    diff = torch.abs(Z_obs_raw[valid_mask] - Z_pred[valid_mask])
    return torch.nan_to_num(diff.mean(), nan=0.0)

# ================= 第一阶段主循环 =================

def run_stage2_robot_alignment(scene_constants, init_scene_state, pb_renderer):
    OUTER_LOOPS = 5
    INNER_LOOPS = 100
    MAX_ROBOT_PTS = 2000

    print(f"\n🦾 启动第一阶段优化 (GPU 全局批处理终极美学版)...")

    ext_cams = [c for c in scene_constants['camera'].keys() if c != scene_constants['meta']['wrist_serial']]
    n_frames = len(scene_constants['camera'][ext_cams[0]]['video_rgb'])

    def make_T(delta):
        T = torch.eye(4, device=device)
        T[:3, :3], T[:3, 3] = axis_angle_to_matrix(delta[:3]), delta[3:]
        return T

    # 🌟 利用 deepcopy 彻底干掉丑陋的嵌套字典推导式
    pybullet_scene_state = copy.deepcopy(init_scene_state)
    frame_point_counts = {t: 0 for t in range(n_frames)}

    for cam in ext_cams:
        print(f"\n  📷 正在独立优化外部相机: [{cam}] ...")

        # 🌟 极致解耦：直接读取纯净的 4x4 静态物理基底，告别 [0] 索引补丁！
        T_init_t = torch.tensor(init_scene_state[cam]['base_extrinsic'], dtype=torch.float32, device=device)
        K_t = torch.tensor(scene_constants['camera'][cam]['K_mat'], dtype=torch.float32, device=device)
        K_np = scene_constants['camera'][cam]['K_mat']

        d_ext = torch.zeros(6, requires_grad=True, device=device)
        optimizer = optim.Adam([d_ext], lr=0.001)

        for outer_step in range(OUTER_LOOPS):
            with torch.no_grad():
                T_cur_np = (T_init_t @ make_T(d_ext)).cpu().numpy()

            cache_X, cache_obs = [], []

            for t in range(n_frames):
                pb_renderer.update_robot_pose(scene_constants['robot']['joint_positions'][t])

                d_obs = scene_constants['camera'][cam]['raw_depth'][t].astype(np.float32)

                r_pts_t = get_foreground_robot_points(T_cur_np, K_np, d_obs, pb_renderer, MAX_ROBOT_PTS)

                if r_pts_t is not None:
                    frame_point_counts[t] += len(r_pts_t)
                    cache_X.append(r_pts_t)
                    cache_obs.append(torch.tensor(d_obs, dtype=torch.float32, device=device)[None, ...])

            if not cache_X:
                print(f"    ⚠️ 未找到有效点云，跳过优化！")
                continue

            batch_X = torch.stack(cache_X)
            batch_obs = torch.stack(cache_obs)

            for inner_step in range(INNER_LOOPS):
                optimizer.zero_grad()

                loss_rob = compute_robot_loss_batched(batch_X, T_init_t @ make_T(d_ext), K_t, batch_obs)
                loss_rob.backward()
                optimizer.step()

                if inner_step % 50 == 0 or inner_step == INNER_LOOPS - 1:
                    print(f"    Outer {outer_step+1}/{OUTER_LOOPS} | Inner {inner_step:03d} | Robot Loss: {loss_rob.item():.4f}")

        with torch.no_grad():
            T_final_np = (T_init_t @ make_T(d_ext)).cpu().numpy()
            print(f"  ✅ [{cam}] 刚体对齐完毕！外参纠正: {np.linalg.norm(d_ext[3:].detach().cpu().numpy()) * 1000:.2f}mm")

            # 🌟 同步更新物理基底和供下游渲染的时序轨迹，完美承上启下！
            pybullet_scene_state[cam]['base_extrinsic'] = T_final_np
            pybullet_scene_state[cam]['extrinsics'] = np.tile(T_final_np, (n_frames, 1, 1))

    return pybullet_scene_state

# ================= 启动调用与渲染展示 =================
all_extrinsics_exist = all(state['extrinsics'] is not None for state in init_scene_state.values())

pb_renderer_ultimate = PyBulletRenderer_Robotiq()

pybullet_scene_state = run_stage2_robot_alignment(
    scene_constants=scene_constants,
    init_scene_state=init_scene_state if all_extrinsics_exist else vggt_scene_state,
    pb_renderer=pb_renderer_ultimate
)

render_fused_point_cloud(
    scene_constants=scene_constants,
    scene_state=pybullet_scene_state,
    use_tint=True
)

render_multiview_mask_inspection(
    scene_constants=scene_constants,
    scene_state=pybullet_scene_state,
    pb_renderer=pb_renderer_ultimate,
)

In [ ]:
# @title 第二阶段：腕部相机-夹爪本体精准对齐 (GPU 专属定制版)

def get_foreground_gripper_points(T_cam_world, K, obs_depth, pb_renderer, max_pts):
    """🔪 精准手术刀：只提取夹爪本体的点云，彻底剔除机械臂连杆"""
    h_img, w_img = obs_depth.shape
    cam_pos, target_pos = T_cam_world[:3, 3], T_cam_world[:3, 3] + T_cam_world[:3, 2]

    view_matrix = p.computeViewMatrix(cam_pos, target_pos, -T_cam_world[:3, 1])
    proj_matrix = pb_renderer._get_projection_matrix(K, w_img, h_img)

    # 🌟 调用底层接口，同时提取深度与语义分割掩码
    _, _, _, depth_buffer, seg_buffer = p.getCameraImage(
        w_img, h_img, viewMatrix=view_matrix, projectionMatrix=proj_matrix,
        renderer=p.ER_BULLET_HARDWARE_OPENGL, flags=p.ER_SEGMENTATION_MASK_OBJECT_AND_LINKINDEX
    )

    metric_depth = 0.1 / (10.0 - 9.99 * np.reshape(depth_buffer, (h_img, w_img)))
    seg_array = np.reshape(seg_buffer, (h_img, w_img)).astype(np.int32)
    obj_ids = seg_array & 0xFFFFFF

    # 🌟 核心过滤：仅仅保留代表夹爪的 ghost_id 对应的像素
    valid_ghost = (obj_ids == pb_renderer.ghost_id)
    v_r, u_r = np.where((metric_depth < 9.9) & valid_ghost)
    z_r = metric_depth[v_r, u_r]

    if len(z_r) < 100: return None

    # 获取在相机坐标系下的齐次坐标
    P_cam_r = np.stack([(u_r - K[0, 2]) * z_r / K[0, 0], (v_r - K[1, 2]) * z_r / K[1, 1], z_r, np.ones_like(z_r)])

    # 随机降采样以喂入 GPU
    idx = np.random.choice(len(z_r), max_pts, replace=(len(z_r) < max_pts))
    return P_cam_r[:, idx]

def compute_wrist_loss_batched(batch_P_ee, T_cam_ee_opt, K, batch_obs):
    """🦇 Wrist 专属算子：通过优化中的手眼逆矩阵将末端点拨回相机系"""
    B, _, h_img, w_img = batch_obs.shape

    # 🌟 获取 T_ee_cam (相机到末端的逆变换)
    T_ee_cam = torch.linalg.inv(T_cam_ee_opt)

    # 将被锚定在末端坐标系(EE)的点，重新投影到当前的相机坐标系
    P_c = batch_P_ee @ T_ee_cam[:3, :3].T + T_ee_cam[:3, 3]
    Z_pred = P_c[..., 2]

    u = K[0, 0] * P_c[..., 0] / Z_pred + K[0, 2]
    v = K[1, 1] * P_c[..., 1] / Z_pred + K[1, 2]

    grid = torch.stack([(u / (w_img - 1)) * 2 - 1, (v / (h_img - 1)) * 2 - 1], dim=-1).unsqueeze(1)
    Z_obs_raw = F.grid_sample(batch_obs, grid, mode='bilinear', padding_mode='border', align_corners=True).squeeze(1).squeeze(1)

    valid_mask = (Z_pred > 0.) & (Z_pred < 1.5) & (Z_obs_raw > 0.) & (Z_obs_raw < 1.5) & \
                 (u >= 0) & (u < w_img - 1) & (v >= 0) & (v < h_img - 1)

    diff = torch.abs(Z_obs_raw[valid_mask] - Z_pred[valid_mask])
    return torch.nan_to_num(diff.mean(), nan=0.0)

# ================= 腕部专属对齐引擎 =================

def run_stage2_wrist_alignment(scene_constants, init_scene_state, pb_renderer):
    OUTER_LOOPS = 5
    INNER_LOOPS = 100
    MAX_ROBOT_PTS = 2000

    print(f"\n🦾 启动腕部实验：纯夹爪物理锚定 (仅优化 Wrist Camera)...")

    wrist_cam = scene_constants['meta']['wrist_serial']
    n_frames = len(scene_constants['camera'][wrist_cam]['video_rgb'])

    pybullet_scene_state = copy.deepcopy(init_scene_state)
    frame_point_counts = {t: 0 for t in range(n_frames)}

    # 提取初始的手眼矩阵 (T_cam_ee_init)
    T_cam_ee_init_np = init_scene_state[wrist_cam]['base_extrinsic']
    T_cam_ee_init_t = torch.tensor(T_cam_ee_init_np, dtype=torch.float32, device=device)

    K_np = scene_constants['camera'][wrist_cam]['K_mat']
    K_t = torch.tensor(K_np, dtype=torch.float32, device=device)

    T_ee_base_all = scene_constants['robot']['T_ee_base_all']

    d_ext = torch.zeros(6, requires_grad=True, device=device)
    optimizer = optim.Adam([d_ext], lr=0.001)

    def make_T(delta):
        T = torch.eye(4, device=device)
        T[:3, :3], T[:3, 3] = axis_angle_to_matrix(delta[:3]), delta[3:]
        return T

    for outer_step in range(OUTER_LOOPS):
        with torch.no_grad():
            T_cam_ee_np = (T_cam_ee_init_t @ make_T(d_ext)).cpu().numpy()

        cache_P_ee, cache_obs = [], []

        for t in range(n_frames):
            # 🌟 必须同步更新关节和夹爪状态
            pb_renderer.update_robot_pose(scene_constants['robot']['joint_positions'][t],
                                          gripper_state=scene_constants['robot']['gripper_positions'][t])

            T_cam_world_np = T_ee_base_all[t] @ T_cam_ee_np
            d_obs = scene_constants['camera'][wrist_cam]['raw_depth'][t].astype(np.float32)

            # 获取相机系下的纯夹爪点云
            P_cam_r = get_foreground_gripper_points(T_cam_world_np, K_np, d_obs, pb_renderer, MAX_ROBOT_PTS)

            if P_cam_r is not None:
                # 🌟 数据解耦锚定：将相机系的点云，通过当前手眼矩阵转移到末端坐标系(EE)锁定
                P_ee_r = (T_cam_ee_np @ P_cam_r)[:3, :].T

                frame_point_counts[t] += len(P_ee_r)
                cache_P_ee.append(torch.tensor(P_ee_r, dtype=torch.float32, device=device))
                cache_obs.append(torch.tensor(d_obs, dtype=torch.float32, device=device)[None, ...])

        if not cache_P_ee:
            print(f"    ⚠️ 未找到有效的夹爪点云！请检查渲染逻辑。")
            break

        batch_P_ee = torch.stack(cache_P_ee)
        batch_obs = torch.stack(cache_obs)

        for inner_step in range(INNER_LOOPS):
            optimizer.zero_grad()

            T_cam_ee_opt = T_cam_ee_init_t @ make_T(d_ext)
            loss_rob = compute_wrist_loss_batched(batch_P_ee, T_cam_ee_opt, K_t, batch_obs)
            loss_rob.backward()
            optimizer.step()

            # 🌟 核心修改：增加平移和旋转的改变量监控
            if inner_step % 50 == 0 or inner_step == INNER_LOOPS - 1:
                with torch.no_grad():
                    # 旋转向量的模长即为旋转的弧度，转换为角度
                    rot_deg = torch.norm(d_ext[:3]).item() * (180.0 / np.pi)
                    # 平移向量的模长即为平移量，转换为毫米
                    shift_mm = torch.norm(d_ext[3:]).item() * 1000.0
                print(f"    Outer {outer_step+1}/{OUTER_LOOPS} | Inner {inner_step:03d} | Wrist Loss: {loss_rob.item():.4f} | Shift: {shift_mm:.2f}mm | Rot: {rot_deg:.2f}°")

    with torch.no_grad():
        T_cam_ee_final = (T_cam_ee_init_t @ make_T(d_ext)).cpu().numpy()
        shift_mm = torch.norm(d_ext[3:]).item() * 1000.0
        rot_deg = torch.norm(d_ext[:3]).item() * (180.0 / np.pi)
        print(f"  ✅ [Wrist Cam: {wrist_cam}] 手眼标定收敛！纯夹爪对齐修正量 -> 平移: {shift_mm:.2f}mm, 旋转: {rot_deg:.2f}°")

        # 🌟 覆盖更新状态大盘
        pybullet_scene_state[wrist_cam]['base_extrinsic'] = T_cam_ee_final
        pybullet_scene_state[wrist_cam]['extrinsics'] = T_ee_base_all @ T_cam_ee_final

    return pybullet_scene_state

# ================= 启动验证 =================

pb_renderer_ultimate = PyBulletRenderer_Robotiq()

# 覆盖并获取专为 Wrist 优化的 scene_state
wrist_optimized_state = run_stage2_wrist_alignment(
    scene_constants=scene_constants,
    init_scene_state=pybullet_scene_state,
    pb_renderer=pb_renderer_ultimate
)

# 可视化渲染验证修改效果 (你可以指定一个你想看的手眼画面)
render_multiview_mask_inspection(
    scene_constants=scene_constants,
    scene_state=wrist_optimized_state,
    pb_renderer=pb_renderer_ultimate,
)

In [ ]:
# @title 第三阶段实验：全域环境缝合与机器人本体联合优化 (全机位大一统版)

def batched_chamfer_distance(p1, p2):
    """纯矩阵化并行倒角距离 (硬编码 5cm 物理截断：你的最后一道防线)"""
    dist_matrix = torch.cdist(p1, p2)
    min_dist_12 = torch.min(dist_matrix, dim=2)[0]
    min_dist_21 = torch.min(dist_matrix, dim=1)[0]

    valid_12 = min_dist_12 < 0.05
    valid_21 = min_dist_21 < 0.05

    loss = torch.tensor(0.0, device=device)
    if valid_12.any(): loss += min_dist_12[valid_12].mean()
    if valid_21.any(): loss += min_dist_21[valid_21].mean()

    overlap_ratio = (valid_12.sum() + valid_21.sum()) / (p1.shape[0] * (p1.shape[1] + p2.shape[1]) + 1e-6)
    return loss, overlap_ratio.item()

def get_cam_points_t(t, cam_data):
    """预计算算子：无差别提取视场内的所有点云 (包含桌面、墙壁、人体等全域环境)"""
    depth = cam_data['raw_depth'][t].astype(np.float32)
    K_mat_np = cam_data['K_mat']

    valid_mask = (depth > 0.) & (depth < 1.5)
    vs, us = np.where(valid_mask)
    if len(us) < 100: return None

    zs_obs = depth[vs, us]
    x_c = (us - K_mat_np[0, 2]) * zs_obs / K_mat_np[0, 0]
    y_c = (vs - K_mat_np[1, 2]) * zs_obs / K_mat_np[1, 1]

    # 构建相机系齐次坐标
    P_cam = np.stack([x_c, y_c, zs_obs, np.ones_like(zs_obs)], axis=0)

    # 💥 BBox 物理法则被彻底粉碎，所有 P_cam 直接进入采样池！
    if P_cam.shape[1] < 100: return None
    if P_cam.shape[1] > 2000:
        idx = np.random.choice(P_cam.shape[1], 2000, replace=False)
    else:
        idx = np.random.choice(P_cam.shape[1], 2000, replace=True)

    return torch.tensor(P_cam[:, idx], dtype=torch.float32, device=device)

def run_stage3_joint_alignment(scene_constants, stage2_scene_state, pb_renderer):
    print(f"\n🌍 启动第三阶段：大一统环境缝合与机器人本体联合优化 (融合 Wrist 纯净手眼锚定)...")

    camera_ids = list(scene_constants['camera'].keys())
    wrist_cam = scene_constants['meta']['wrist_serial']
    ext_cams = [c for c in camera_ids if c != wrist_cam]
    cam1, cam2 = ext_cams[0], ext_cams[1]

    n_frames = len(scene_constants['camera'][cam1]['video_rgb'])
    robot_joints_seq = scene_constants['robot']['joint_positions']
    gripper_states_seq = scene_constants['robot']['gripper_positions']

    def to_t(arr): return torch.tensor(arr, dtype=torch.float32, device=device)
    def make_T(delta):
        T = torch.eye(4, device=device)
        T[:3, :3], T[:3, 3] = axis_angle_to_matrix(delta[:3]), delta[3:]
        return T

    T_ee_all = scene_constants['robot']['T_ee_base_all']

    # 🌟 极致解耦继承：提取第二阶段优化的完美 4x4 物理基底！
    T_cam_ee_init = stage2_scene_state[wrist_cam]['base_extrinsic']
    init_p1 = stage2_scene_state[cam1]['base_extrinsic']
    init_p2 = stage2_scene_state[cam2]['base_extrinsic']

    K_np1, K_np2, K_np_w = scene_constants['camera'][cam1]['K_mat'], scene_constants['camera'][cam2]['K_mat'], scene_constants['camera'][wrist_cam]['K_mat']
    K_t1, K_t2, K_t_w = to_t(K_np1), to_t(K_np2), to_t(K_np_w)

    cache_P1, cache_P2, cache_Pw, cache_Tee = [], [], [], []
    cache_X1, cache_obs1, cache_X2, cache_obs2 = [], [], [], []
    cache_P_ee, cache_obs_w = [], []  # 🌟 新增：Wrist 夹爪专属缓存

    print(f"  🔍 正在预计算 Chamfer 环境点云、外部 Robot 物理点云 与 腕部夹爪锚点...")
    for t in range(n_frames):
        # --- 1. 提取 Chamfer 环境点云 ---
        p1 = get_cam_points_t(t, scene_constants['camera'][cam1])
        p2 = get_cam_points_t(t, scene_constants['camera'][cam2])
        pw = get_cam_points_t(t, scene_constants['camera'][wrist_cam])
        if p1 is not None and p2 is not None and pw is not None:
            cache_P1.append(p1); cache_P2.append(p2); cache_Pw.append(pw); cache_Tee.append(to_t(T_ee_all[t]))

        # 🌟 必须同步更新关节和夹爪状态，否则无法精准提取掩码
        pb_renderer.update_robot_pose(robot_joints_seq[t], gripper_state=gripper_states_seq[t])

        # --- 2. 提取 外部相机 Robot 物理点云 ---
        d_obs1 = scene_constants['camera'][cam1]['raw_depth'][t].astype(np.float32)
        r_pts1 = get_foreground_robot_points(init_p1, K_np1, d_obs1, pb_renderer, max_pts=2000)
        if r_pts1 is not None:
            cache_X1.append(r_pts1)
            cache_obs1.append(torch.tensor(d_obs1, dtype=torch.float32, device=device)[None, ...])

        d_obs2 = scene_constants['camera'][cam2]['raw_depth'][t].astype(np.float32)
        r_pts2 = get_foreground_robot_points(init_p2, K_np2, d_obs2, pb_renderer, max_pts=2000)
        if r_pts2 is not None:
            cache_X2.append(r_pts2)
            cache_obs2.append(torch.tensor(d_obs2, dtype=torch.float32, device=device)[None, ...])

        # --- 3. 🌟 提取 腕部夹爪纯物理点云 (解耦锁定到末端 EE 系) ---
        T_cam_world_np = T_ee_all[t] @ T_cam_ee_init
        d_obs_w = scene_constants['camera'][wrist_cam]['raw_depth'][t].astype(np.float32)
        P_cam_r = get_foreground_gripper_points(T_cam_world_np, K_np_w, d_obs_w, pb_renderer, max_pts=2000)

        if P_cam_r is not None:
            # 数据解耦锚定：将相机系的点云，通过当前手眼矩阵转移到末端坐标系(EE)锁定
            P_ee_r = (T_cam_ee_init @ P_cam_r)[:3, :].T
            cache_P_ee.append(torch.tensor(P_ee_r, dtype=torch.float32, device=device))
            cache_obs_w.append(torch.tensor(d_obs_w, dtype=torch.float32, device=device)[None, ...])

    # 堆叠 Batch 张量
    batch_P1, batch_P2, batch_Pw, batch_Tee = torch.stack(cache_P1), torch.stack(cache_P2), torch.stack(cache_Pw), torch.stack(cache_Tee)
    batch_X1, batch_obs1 = torch.stack(cache_X1), torch.stack(cache_obs1)
    batch_X2, batch_obs2 = torch.stack(cache_X2), torch.stack(cache_obs2)
    batch_P_ee = torch.stack(cache_P_ee) if cache_P_ee else None
    batch_obs_w = torch.stack(cache_obs_w) if cache_obs_w else None

    print(f"  ✅ 数据准备完毕！启动 GPU 大一统联合优化引擎 (Chamfer + Robot + Wrist)...")

    d1 = torch.zeros(6, requires_grad=True, device=device)
    d2 = torch.zeros(6, requires_grad=True, device=device)
    dhe = torch.zeros(6, requires_grad=True, device=device)

    optimizer = optim.Adam([d1, d2, dhe], lr=0.001)

    T1_init_t, T2_init_t, Tee_init_t = to_t(init_p1), to_t(init_p2), to_t(T_cam_ee_init)

    for step in range(500):
        optimizer.zero_grad()

        # 🌟 核心魔法 1：计算 Chamfer 游离态缝合
        bc1 = (T1_init_t @ make_T(d1) @ batch_P1)[:, :3, :].transpose(1, 2)
        bc2 = (T2_init_t @ make_T(d2) @ batch_P2)[:, :3, :].transpose(1, 2)
        T_wrist_world = batch_Tee @ (Tee_init_t @ make_T(dhe))
        bcw = torch.bmm(T_wrist_world, batch_Pw)[:, :3, :].transpose(1, 2)

        l12, o12 = batched_chamfer_distance(bc1, bc2)
        l1w, o1w = batched_chamfer_distance(bc1, bcw)
        l2w, o2w = batched_chamfer_distance(bc2, bcw)
        loss_chamfer = l12 + l1w + l2w

        # 🌟 核心魔法 2：计算 外部相机 Robot 绝对物理锚定
        l_rob1 = compute_robot_loss_batched(batch_X1, T1_init_t @ make_T(d1), K_t1, batch_obs1)
        l_rob2 = compute_robot_loss_batched(batch_X2, T2_init_t @ make_T(d2), K_t2, batch_obs2)

        # 🌟 核心魔法 3：计算 腕部夹爪专属物理对齐！
        l_wrist = torch.tensor(0.0, device=device)
        if batch_P_ee is not None:
            l_wrist = compute_wrist_loss_batched(batch_P_ee, Tee_init_t @ make_T(dhe), K_t_w, batch_obs_w)

        # 🌟 绝杀：真正的大一统联合 Loss！
        loss_total = loss_chamfer + 1.0 * (l_rob1 + l_rob2 + l_wrist)
        loss_total.backward()
        optimizer.step()

        if step % 50 == 0 or step == 500 - 1:
            bg_overlap = (o12 + o1w + o2w) / 3.0 * 100
            shift_c1 = torch.norm(d1[:3]).item() * 1000
            shift_c2 = torch.norm(d2[:3]).item() * 1000
            shift_w = torch.norm(dhe[:3]).item() * 1000

            print(f"    Step {step:03d} | "
                  f"Chmf: {loss_chamfer.item():.4f} | "
                  f"Rob1: {l_rob1.item():.4f} | Rob2: {l_rob2.item():.4f} | Wrst: {l_wrist.item():.4f} | "
                  f"BG Overlap: {bg_overlap:.1f}% | "
                  f"Shift -> C1: {shift_c1:.2f}mm, C2: {shift_c2:.2f}mm, W: {shift_w:.2f}mm")

    with torch.no_grad():
        final_p1 = (T1_init_t @ make_T(d1)).cpu().numpy()
        final_p2 = (T2_init_t @ make_T(d2)).cpu().numpy()
        final_cam_ee = (Tee_init_t @ make_T(dhe)).cpu().numpy()

    print(f"\n✅ 3D 大一统联合优化收官！全机位相互牵制下产生的协同位移：")
    print(f"  📷 [静态 {cam1}]: {np.linalg.norm(d1[:3].detach().cpu().numpy()) * 1000:.2f} mm")
    print(f"  📷 [静态 {cam2}]: {np.linalg.norm(d2[:3].detach().cpu().numpy()) * 1000:.2f} mm")
    print(f"  🦾 [运动 Wrist]: {np.linalg.norm(dhe[:3].detach().cpu().numpy()) * 1000:.2f} mm")

    # 🌟 终极组装：保存 4x4 的纯净物理基底与 N 帧的动态重投影轨迹
    ultimate_scene_state = {cam: {} for cam in camera_ids}

    ultimate_scene_state[cam1].update({
        'base_extrinsic': final_p1,
        'extrinsics': np.tile(final_p1, (n_frames, 1, 1))
    })

    ultimate_scene_state[cam2].update({
        'base_extrinsic': final_p2,
        'extrinsics': np.tile(final_p2, (n_frames, 1, 1))
    })

    ultimate_scene_state[wrist_cam].update({
        'base_extrinsic': final_cam_ee,
        'extrinsics': T_ee_all @ final_cam_ee
    })

    return ultimate_scene_state

# ================= 启动调用与渲染展示 =================
pb_renderer_ultimate = PyBulletRenderer_Robotiq()

ultimate_scene_state = run_stage3_joint_alignment(
    scene_constants=scene_constants,
    stage2_scene_state=wrist_optimized_state,
    pb_renderer=pb_renderer_ultimate
)

render_fused_point_cloud(
    scene_constants=scene_constants,
    scene_state=ultimate_scene_state,
    use_tint=True
)

render_multiview_mask_inspection(
    scene_constants=scene_constants,
    scene_state=ultimate_scene_state,
    pb_renderer=pb_renderer_ultimate,
)

In [ ]:
# @title 第四阶段实验：全域环境缝合与机器人本体联合优化 (精细雕琢版)

def run_stage4_joint_alignment(scene_constants, stage3_scene_state, pb_renderer):
    print(f"\n🌍 启动第三阶段：大一统环境缝合与机器人本体联合优化 (融合 Wrist 纯净手眼锚定)...")

    camera_ids = list(scene_constants['camera'].keys())
    wrist_cam = scene_constants['meta']['wrist_serial']
    ext_cams = [c for c in camera_ids if c != wrist_cam]
    cam1, cam2 = ext_cams[0], ext_cams[1]

    n_frames = len(scene_constants['camera'][cam1]['video_rgb'])
    robot_joints_seq = scene_constants['robot']['joint_positions']
    gripper_states_seq = scene_constants['robot']['gripper_positions']

    def to_t(arr): return torch.tensor(arr, dtype=torch.float32, device=device)
    def make_T(delta):
        T = torch.eye(4, device=device)
        T[:3, :3], T[:3, 3] = axis_angle_to_matrix(delta[:3]), delta[3:]
        return T

    T_ee_all = scene_constants['robot']['T_ee_base_all']

    # 🌟 极致解耦继承：提取第二阶段优化的完美 4x4 物理基底！
    T_cam_ee_init = stage3_scene_state[wrist_cam]['base_extrinsic']
    init_p1 = stage3_scene_state[cam1]['base_extrinsic']
    init_p2 = stage3_scene_state[cam2]['base_extrinsic']

    K_np1, K_np2, K_np_w = scene_constants['camera'][cam1]['K_mat'], scene_constants['camera'][cam2]['K_mat'], scene_constants['camera'][wrist_cam]['K_mat']
    K_t1, K_t2, K_t_w = to_t(K_np1), to_t(K_np2), to_t(K_np_w)

    cache_P1, cache_P2, cache_Pw, cache_Tee = [], [], [], []
    cache_X1, cache_obs1, cache_X2, cache_obs2 = [], [], [], []
    cache_P_ee, cache_obs_w = [], []  # 🌟 新增：Wrist 夹爪专属缓存

    print(f"  🔍 正在预计算 Chamfer 环境点云、外部 Robot 物理点云 与 腕部夹爪锚点...")
    for t in range(n_frames):
        # --- 1. 提取 Chamfer 环境点云 ---
        p1 = get_cam_points_t(t, scene_constants['camera'][cam1])
        p2 = get_cam_points_t(t, scene_constants['camera'][cam2])
        pw = get_cam_points_t(t, scene_constants['camera'][wrist_cam])
        if p1 is not None and p2 is not None and pw is not None:
            cache_P1.append(p1); cache_P2.append(p2); cache_Pw.append(pw); cache_Tee.append(to_t(T_ee_all[t]))

        # 🌟 必须同步更新关节和夹爪状态，否则无法精准提取掩码
        pb_renderer.update_robot_pose(robot_joints_seq[t], gripper_state=gripper_states_seq[t])

        # --- 2. 提取 外部相机 Robot 物理点云 ---
        d_obs1 = scene_constants['camera'][cam1]['raw_depth'][t].astype(np.float32)
        r_pts1 = get_foreground_robot_points(init_p1, K_np1, d_obs1, pb_renderer, max_pts=2000)
        if r_pts1 is not None:
            cache_X1.append(r_pts1)
            cache_obs1.append(torch.tensor(d_obs1, dtype=torch.float32, device=device)[None, ...])

        d_obs2 = scene_constants['camera'][cam2]['raw_depth'][t].astype(np.float32)
        r_pts2 = get_foreground_robot_points(init_p2, K_np2, d_obs2, pb_renderer, max_pts=2000)
        if r_pts2 is not None:
            cache_X2.append(r_pts2)
            cache_obs2.append(torch.tensor(d_obs2, dtype=torch.float32, device=device)[None, ...])

        # --- 3. 🌟 提取 腕部夹爪纯物理点云 (解耦锁定到末端 EE 系) ---
        T_cam_world_np = T_ee_all[t] @ T_cam_ee_init
        d_obs_w = scene_constants['camera'][wrist_cam]['raw_depth'][t].astype(np.float32)
        P_cam_r = get_foreground_gripper_points(T_cam_world_np, K_np_w, d_obs_w, pb_renderer, max_pts=2000)

        if P_cam_r is not None:
            # 数据解耦锚定：将相机系的点云，通过当前手眼矩阵转移到末端坐标系(EE)锁定
            P_ee_r = (T_cam_ee_init @ P_cam_r)[:3, :].T
            cache_P_ee.append(torch.tensor(P_ee_r, dtype=torch.float32, device=device))
            cache_obs_w.append(torch.tensor(d_obs_w, dtype=torch.float32, device=device)[None, ...])

    # 堆叠 Batch 张量
    batch_P1, batch_P2, batch_Pw, batch_Tee = torch.stack(cache_P1), torch.stack(cache_P2), torch.stack(cache_Pw), torch.stack(cache_Tee)
    batch_X1, batch_obs1 = torch.stack(cache_X1), torch.stack(cache_obs1)
    batch_X2, batch_obs2 = torch.stack(cache_X2), torch.stack(cache_obs2)
    batch_P_ee = torch.stack(cache_P_ee) if cache_P_ee else None
    batch_obs_w = torch.stack(cache_obs_w) if cache_obs_w else None

    print(f"  ✅ 数据准备完毕！启动 GPU 大一统联合优化引擎 (Chamfer + Robot + Wrist)...")

    d1 = torch.zeros(6, requires_grad=True, device=device)
    d2 = torch.zeros(6, requires_grad=True, device=device)
    dhe = torch.zeros(6, requires_grad=True, device=device)

    optimizer = optim.Adam([d1, d2, dhe], lr=0.0001)

    T1_init_t, T2_init_t, Tee_init_t = to_t(init_p1), to_t(init_p2), to_t(T_cam_ee_init)

    for step in range(500):
        optimizer.zero_grad()

        # 🌟 核心魔法 1：计算 Chamfer 游离态缝合
        bc1 = (T1_init_t @ make_T(d1) @ batch_P1)[:, :3, :].transpose(1, 2)
        bc2 = (T2_init_t @ make_T(d2) @ batch_P2)[:, :3, :].transpose(1, 2)
        T_wrist_world = batch_Tee @ (Tee_init_t @ make_T(dhe))
        bcw = torch.bmm(T_wrist_world, batch_Pw)[:, :3, :].transpose(1, 2)

        l12, o12 = batched_chamfer_distance(bc1, bc2)
        l1w, o1w = batched_chamfer_distance(bc1, bcw)
        l2w, o2w = batched_chamfer_distance(bc2, bcw)
        loss_chamfer = l12 + l1w + l2w

        # 🌟 核心魔法 2：计算 外部相机 Robot 绝对物理锚定
        l_rob1 = compute_robot_loss_batched(batch_X1, T1_init_t @ make_T(d1), K_t1, batch_obs1)
        l_rob2 = compute_robot_loss_batched(batch_X2, T2_init_t @ make_T(d2), K_t2, batch_obs2)

        # 🌟 核心魔法 3：计算 腕部夹爪专属物理对齐！
        l_wrist = torch.tensor(0.0, device=device)
        if batch_P_ee is not None:
            l_wrist = compute_wrist_loss_batched(batch_P_ee, Tee_init_t @ make_T(dhe), K_t_w, batch_obs_w)

        # 🌟 绝杀：真正的大一统联合 Loss！
        loss_total = loss_chamfer + 0.1 * (l_rob1 + l_rob2 + l_wrist)
        loss_total.backward()
        optimizer.step()

        if step % 50 == 0 or step == 500 - 1:
            bg_overlap = (o12 + o1w + o2w) / 3.0 * 100
            shift_c1 = torch.norm(d1[:3]).item() * 1000
            shift_c2 = torch.norm(d2[:3]).item() * 1000
            shift_w = torch.norm(dhe[:3]).item() * 1000

            print(f"    Step {step:03d} | "
                  f"Chmf: {loss_chamfer.item():.4f} | "
                  f"Rob1: {l_rob1.item():.4f} | Rob2: {l_rob2.item():.4f} | Wrst: {l_wrist.item():.4f} | "
                  f"BG Overlap: {bg_overlap:.1f}% | "
                  f"Shift -> C1: {shift_c1:.2f}mm, C2: {shift_c2:.2f}mm, W: {shift_w:.2f}mm")

    with torch.no_grad():
        final_p1 = (T1_init_t @ make_T(d1)).cpu().numpy()
        final_p2 = (T2_init_t @ make_T(d2)).cpu().numpy()
        final_cam_ee = (Tee_init_t @ make_T(dhe)).cpu().numpy()

    print(f"\n✅ 3D 大一统联合优化收官！全机位相互牵制下产生的协同位移：")
    print(f"  📷 [静态 {cam1}]: {np.linalg.norm(d1[:3].detach().cpu().numpy()) * 1000:.2f} mm")
    print(f"  📷 [静态 {cam2}]: {np.linalg.norm(d2[:3].detach().cpu().numpy()) * 1000:.2f} mm")
    print(f"  🦾 [运动 Wrist]: {np.linalg.norm(dhe[:3].detach().cpu().numpy()) * 1000:.2f} mm")

    # 🌟 终极组装：保存 4x4 的纯净物理基底与 N 帧的动态重投影轨迹
    ultimate_scene_state = {cam: {} for cam in camera_ids}

    ultimate_scene_state[cam1].update({
        'base_extrinsic': final_p1,
        'extrinsics': np.tile(final_p1, (n_frames, 1, 1))
    })

    ultimate_scene_state[cam2].update({
        'base_extrinsic': final_p2,
        'extrinsics': np.tile(final_p2, (n_frames, 1, 1))
    })

    ultimate_scene_state[wrist_cam].update({
        'base_extrinsic': final_cam_ee,
        'extrinsics': T_ee_all @ final_cam_ee
    })

    return ultimate_scene_state

# ================= 启动调用与渲染展示 =================
pb_renderer_ultimate = PyBulletRenderer_Robotiq()

ultimate_scene_state = run_stage4_joint_alignment(
    scene_constants=scene_constants,
    stage2_scene_state=ultimate_scene_state,
    pb_renderer=pb_renderer_ultimate
)

render_fused_point_cloud(
    scene_constants=scene_constants,
    scene_state=ultimate_scene_state,
    use_tint=True
)

render_multiview_mask_inspection(
    scene_constants=scene_constants,
    scene_state=ultimate_scene_state,
    pb_renderer=pb_renderer_ultimate,
)

In [ ]:
# @title 剔除静止发呆帧 (双轨联动 + 自动推断切片版)

def filter_idle_frames(scene_constants, scene_state):
    """
    双轨联动：同时对 constants 和 state 进行安全深拷贝与全域静止帧切除。
    采用“自动长度侦测”魔法：凡是时间维度等于 n_original 的数据统统一刀切！
    """
    print("✂️ 启动全域发呆帧双轨切除程序...")

    # 🌟 1. 绝对防御：深拷贝，绝不污染你原本辛辛苦苦跑出来的数据大盘！
    filtered_constants = copy.deepcopy(scene_constants)
    filtered_state = copy.deepcopy(scene_state)

    valid_indices = filtered_constants['meta'].get('valid_indices')
    if valid_indices is None:
        print("  ⚠️ 未找到有效帧过滤数据，跳过切除。")
        return filtered_constants, filtered_state

    n_original = len(filtered_constants['robot']['joint_positions'])
    valid_indices = valid_indices[valid_indices < n_original]

    if len(valid_indices) == 0:
        print("  ❌ 警告：该视频过滤后没有任何有效帧保留！")
        return filtered_constants, filtered_state

    print(f"  ✅ 准备切除静止发呆帧！保留关键动作帧数: {len(valid_indices)} / {n_original}")

    # ==========================================
    # 🌟 2. 过滤 Constants (机器人本体 + 自动扫描相机数据)
    # ==========================================
    # 明确切除机器人本体流
    for key in ['joint_positions', 'gripper_positions', 'T_ee_base_all']:
        if key in filtered_constants['robot']:
            filtered_constants['robot'][key] = filtered_constants['robot'][key][valid_indices]

    # 相机多模态数据：动态侦测（无论你加了多少如 raw_depth, sam_masks，只要符合帧数长度，全部自动切除）
    for cam_id, cam_data in filtered_constants['camera'].items():
        for key, value in cam_data.items():
            if isinstance(value, (list, np.ndarray)) and len(value) == n_original:
                if isinstance(value, list):
                    cam_data[key] = [value[i] for i in valid_indices]
                else:
                    cam_data[key] = value[valid_indices]
                # print(f"    - 侦测到动态时序数据并切除: [{cam_id}] {key}")

    # ==========================================
    # 🌟 3. 过滤 Scene State (外部相机外参轨迹 + 腕部相机动态外参)
    # ==========================================
    if filtered_state is not None:
        for cam_id, state_data in filtered_state.items():
            # 找到动态重投影轨迹阵 (N, 4, 4)
            if 'extrinsics' in state_data and len(state_data['extrinsics']) == n_original:
                state_data['extrinsics'] = state_data['extrinsics'][valid_indices]
                # print(f"    - 侦测到动态相机姿态并切除: [{cam_id}] extrinsics")

    print("  🎉 双轨时间轴完全对齐！切除手术完美收官。")
    return filtered_constants, filtered_state

# ================= 启动调用与变量重命名 =================
# 命名规则：加上 active_ 前缀，代表这是“去除了静止发呆后的纯动作精炼版”
# @title 🔍 查验核对：全域切片后的数据结构大盘

# 1. 启动双轨切除与变量重命名
active_scene_constants, active_ultimate_scene_state = filter_idle_frames(
    scene_constants=scene_constants,
    scene_state=ultimate_scene_state
)

def lightweight_inspect(d, name="Dictionary", indent=0):
    """轻量级高颜值结构扫描器：专为查验时间轴 Shape 对齐而生"""
    if indent == 0:
        print(f"\n🧊 正在扫描数据大盘: 【{name}】")
        print("=" * 60)

    for key, value in d.items():
        # 跳过一些极其冗长且无时间维度的静态配置，让视野更聚焦
        if key in ['K_mat', 'D_mat', 'camera_models', 'meta']:
            continue

        spacing = "│   " * indent + "├── "

        if isinstance(value, dict):
            print(f"{'│   ' * indent}├── 📂 {key}/")
            lightweight_inspect(value, name, indent + 1)
        elif hasattr(value, 'shape'):  # 兼容 numpy 数组和 torch 张量
            print(f"{spacing}📊 {key:<20} -> shape: {value.shape}")
        elif isinstance(value, list):
            item_shape = value[0].shape if (len(value) > 0 and hasattr(value[0], 'shape')) else "mixed/scalar"
            print(f"{spacing}📜 {key:<20} -> list  (len={len(value)}), items: {item_shape}")
        else:
            if indent == 0:
                print(f"{spacing}📝 {key:<20} -> {type(value).__name__}")

# ================= 启动结构树打印 =================

if active_scene_constants:
    # 查验 Robot 本体流
    lightweight_inspect(active_scene_constants['robot'], name="active_scene_constants ['robot']")

    # 抽查一个相机的视觉流 (以 Wrist Cam 为例)
    wrist_cam = active_scene_constants['meta']['wrist_serial']
    lightweight_inspect(active_scene_constants['camera'][wrist_cam], name=f"active_scene_constants ['camera']['{wrist_cam}']")

if active_ultimate_scene_state:
    # 查验外部 4x4 外参轨迹阵
    lightweight_inspect(active_ultimate_scene_state, name="active_ultimate_scene_state")

print("\n🎯 查验重点：请核对上面打印的所有 shape 的第 0 维（比如从 194 变成了 157），它们必须完全一致！")

In [ ]:
# @title 2D 机械臂分割可视化 (双极态极限对比验证版)

def inspect_gripper_extremes(
    scene_constants,
    scene_state,
    pb_renderer,
    tgt_width=1800
):
    """
    自动抽取夹爪 State 接近 0 和绝对值最大 的两帧，上下对比渲染
    """
    gripper_states = scene_constants['robot']['gripper_positions']

    # 🌟 1. 寻找两个极限帧的索引
    # 找绝对值最大的一帧 (通常是最大张开或最大抓取)
    idx_max = np.argmax(np.abs(gripper_states))
    val_max = gripper_states[idx_max]

    # 找最接近 0 的一帧 (通常是完全闭合或完全张开的另一个极限)
    idx_zero = np.argmin(np.abs(gripper_states))
    val_zero = gripper_states[idx_zero]

    print(f"🎯 锁定验证帧 [State ~ 0]: 第 {idx_zero} 帧 | 夹爪数据: {val_zero:.4f}")
    print(f"🎯 锁定验证帧 [State Max]: 第 {idx_max} 帧 | 夹爪数据: {val_max:.4f}")

    camera_ids = list(scene_constants['camera'].keys())
    wrist_serial = scene_constants['meta']['wrist_serial']

    # 🌟 2. 内部封装一个单帧渲染的闭包函数，避免代码重复
    def render_single_frame(frame_idx, gripper_val):
        current_joints = scene_constants['robot']['joint_positions'][frame_idx]
        pb_renderer.update_robot_pose(current_joints, gripper_state=gripper_val)

        frame_views = []
        for cam_id in camera_ids:
            cam_data = scene_constants['camera'][cam_id]
            cam_state = scene_state[cam_id]

            img_rgb = cam_data['video_rgb'][frame_idx].copy()
            h_img, w_img = img_rgb.shape[:2]

            # 渲染 Mask
            robot_mask = pb_renderer.render_mask(
                extrinsics=cam_state['extrinsics'][frame_idx],
                intrinsics=cam_data['K_mat'],
                width=w_img,
                height=h_img
            ) > 0

            # 赛博朋克风混色
            overlay = img_rgb.copy()
            overlay[robot_mask] = [50, 150, 255]
            blended_img = cv2.addWeighted(img_rgb, 0.6, overlay, 0.4, 0)

            # 文本与阴影绘制
            is_wrist = (cam_id == wrist_serial)
            cam_type = "Wrist Cam" if is_wrist else "Ext Cam"
            cv2.putText(blended_img, f"{cam_type} [{cam_id}]", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 0), 4)
            cv2.putText(blended_img, f"{cam_type} [{cam_id}]", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 2)

            frame_views.append(blended_img)

        # 横向拼接并缩放
        row_concat = np.concatenate(frame_views, axis=1)
        tgt_height = int(row_concat.shape[0] * (tgt_width / row_concat.shape[1]))
        return cv2.resize(row_concat, (tgt_width, tgt_height))

    # 🌟 3. 分别渲染两张极限帧的底片
    img_zero = render_single_frame(idx_zero, val_zero)
    img_max = render_single_frame(idx_max, val_max)

    # 🌟 4. 使用 matplotlib 上下分屏对比展示
    fig, axes = plt.subplots(2, 1, figsize=(18, 12))

    axes[0].imshow(img_zero)
    axes[0].set_title(f"Gripper State ~ 0 (Frame {idx_zero} | State {val_zero:.4f})", fontsize=16, fontweight='bold')
    axes[0].axis('off')

    axes[1].imshow(img_max)
    axes[1].set_title(f"Gripper Maximum Value (Frame {idx_max} | State {val_max:.4f})", fontsize=16, fontweight='bold')
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()

# ================= 启动验证 =================
pb_renderer_ultimate = PyBulletRenderer_Robotiq()

inspect_gripper_extremes(
    scene_constants=active_scene_constants,
    scene_state=active_ultimate_scene_state,
    pb_renderer=pb_renderer_ultimate,
    tgt_width=1800
)

In [ ]:
# @title 2D 机械臂分割可视化 (修复夹爪动态开合版)

def render_segmentation_video(
    scene_constants,
    scene_state,
    pb_renderer,
    tgt_width=1200
):
    """
    将 PyBullet 物理引擎中机械臂的三维姿态，重投影并叠加到 2D 真实视频上
    """
    # 🌟 1. 自动提取全局状态
    camera_ids = list(scene_constants['camera'].keys())
    wrist_serial = scene_constants['meta']['wrist_serial']

    # 动态获取总帧数
    n_frames = len(scene_constants['camera'][camera_ids[0]]['video_rgb'])
    video_frames = []

    # 🌟 2. 影视级逐帧渲染循环
    for frame_idx in tqdm(range(n_frames), desc=f"🎥 渲染分割视频"):

        # 🌟 核心修复：同时提取当前帧的机械臂关节角度和夹爪开合度
        current_joints = scene_constants['robot']['joint_positions'][frame_idx]
        current_gripper = scene_constants['robot']['gripper_positions'][frame_idx]

        # 强制物理引擎中的机械臂摆出当前真实关节姿态 (包含夹爪！)
        pb_renderer.update_robot_pose(current_joints, gripper_state=current_gripper)

        frame_views = []

        for cam_id in camera_ids:
            cam_data = scene_constants['camera'][cam_id]
            cam_state = scene_state[cam_id]

            img_rgb = cam_data['video_rgb'][frame_idx].copy()
            h_img, w_img = img_rgb.shape[:2]

            # 🌟 3. 核心魔法：利用最新位姿在物理引擎中拍下绝对精确的遮罩
            robot_mask = pb_renderer.render_mask(
                extrinsics=cam_state['extrinsics'][frame_idx],
                intrinsics=cam_data['K_mat'],
                width=w_img,
                height=h_img
            ) > 0

            # 🌟 4. 赛博朋克风混色：为机械臂穿上蓝色半透明紧身衣
            overlay = img_rgb.copy()
            overlay[robot_mask] = [50, 150, 255]
            blended_img = cv2.addWeighted(img_rgb, 0.6, overlay, 0.4, 0)

            # 🌟 5. 极简文本与阴影绘制 (防止高光背景看不清文字)
            is_wrist = (cam_id == wrist_serial)
            label_color = (0, 255, 255) if is_wrist else (0, 255, 0)

            cv2.putText(blended_img, f"Cam [{cam_id}]", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 0), 4)
            cv2.putText(blended_img, f"Cam [{cam_id}]", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 2)

            frame_views.append(blended_img)

        # 🌟 6. 横向拼接当前帧的所有机位，并进行等比例缩放
        row_concat = np.concatenate(frame_views, axis=1)
        tgt_height = int(row_concat.shape[0] * (tgt_width / row_concat.shape[1]))
        video_frames.append(cv2.resize(row_concat, (tgt_width, tgt_height)))

    return video_frames

# ================= 启动渲染 =================
pb_renderer_ultimate = PyBulletRenderer_Robotiq()

seg_video_frames = render_segmentation_video(
    scene_constants=active_scene_constants,
    scene_state=active_ultimate_scene_state,
    pb_renderer=pb_renderer_ultimate,
    tgt_width=1200
)

media.show_video(seg_video_frames, fps=15, codec='gif')

In [ ]:
# @title 2D 联合交叉绘制相机坐标轴验证

def render_cross_camera_axes(scene_constants, scene_state, axis_len=0.15, tgt_w=1200):
    # 🌟 1. 语义复苏：直接从全新的 camera 层级提取相机列表，干掉丑陋的 if 过滤
    cams = list(scene_constants['camera'].keys())
    n_frames = len(scene_state[cams[0]]['extrinsics'])

    # 巧妙构造局部齐次坐标轴矩阵 (4x4) -> [原点, X, Y, Z]
    axes_3d = np.array([[0, 0, 0, 1], [axis_len, 0, 0, 1],
                        [0, axis_len, 0, 1], [0, 0, axis_len, 1]]).T

    video_frames = []

    # 🌟 2. 纯粹的渲染循环：保留清晰的帧索引与相机迭代，加入进度条避免无聊等待
    for frame_idx in tqdm(range(n_frames), desc="🎥 渲染交叉坐标轴"):
        camera_views = []

        for obs_cam in cams:
            cam_data = scene_constants['camera'][obs_cam]
            img_rgb = cam_data['video_rgb'][frame_idx].copy()
            h_img, w_img = img_rgb.shape[:2]
            K_mat = cam_data['K_mat']

            # 观察视角的逆矩阵 (用于将目标相机坐标变换到观察相机坐标系)
            obs_pose_inv = np.linalg.inv(scene_state[obs_cam]['extrinsics'][frame_idx])

            for tgt_cam in cams:
                if obs_cam == tgt_cam:
                    continue

                # 🌟 核心魔法：一行完成 3D 相对变换与相机投影
                tgt_pose = scene_state[tgt_cam]['extrinsics'][frame_idx]
                pts_cam = (obs_pose_inv @ tgt_pose @ axes_3d)[:3, :]

                if pts_cam[2, 0] < 0:
                    continue  # 剔除在相机背后的点

                # 🌟 极简解包：利用 map 和 转置(T) 直接获取 OpenCV 需要的 tuple
                uv = K_mat @ pts_cam
                org, px, py, pz = map(tuple, (uv[:2] / uv[2]).astype(int).T)

                # 若原点在画面内，则绘制红绿蓝坐标轴
                if 0 <= org[0] < w_img and 0 <= org[1] < h_img:
                    cv2.line(img_rgb, org, px, (255, 0, 0), 3)
                    cv2.line(img_rgb, org, py, (0, 255, 0), 3)
                    cv2.line(img_rgb, org, pz, (0, 0, 255), 3)
                    cv2.circle(img_rgb, org, 5, (0, 0, 0), -1)
                    cv2.circle(img_rgb, org, 2, (255, 255, 255), -1)

                    cv2.putText(img_rgb, f"Cam {tgt_cam}", (org[0]+8, org[1]-8), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 3)
                    cv2.putText(img_rgb, f"Cam {tgt_cam}", (org[0]+8, org[1]-8), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

            # 水印绘制
            cv2.putText(img_rgb, f"View: {obs_cam}", (15, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 3)
            cv2.putText(img_rgb, f"View: {obs_cam}", (15, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
            camera_views.append(img_rgb)

        # 🌟 3. 极简拼接与等比例缩放：将变量名换为具备画面感的表述
        row_concat = np.concatenate(camera_views, axis=1)
        tgt_h = int(row_concat.shape[0] * tgt_w / row_concat.shape[1])
        video_frames.append(cv2.resize(row_concat, (tgt_w, tgt_h)))

    return video_frames

all_viz_frames = render_cross_camera_axes(
    scene_constants=active_scene_constants,
    scene_state=active_ultimate_scene_state,
)

media.show_video(all_viz_frames, fps=15, codec='gif')

In [ ]:
# @title 3D 单帧融合点云与交互式可视化

render_fused_point_cloud(
    scene_constants=active_scene_constants,
    scene_state=active_ultimate_scene_state,
)

In [ ]:
# @title 4D 全域点云环绕视频

def get_look_at_matrix(eye, target, up=(0, 0, 1)):
    """3D 运镜辅助：一行流生成 OpenGL 风格相机朝向矩阵"""
    z_axis = np.array(eye, dtype=float) - target
    z_axis /= np.linalg.norm(z_axis) + 1e-6

    x_axis = np.cross(up, z_axis)
    x_axis /= np.linalg.norm(x_axis) + 1e-6

    y_axis = np.cross(z_axis, x_axis)

    view_matrix = np.eye(4)
    view_matrix[:3, :4] = np.column_stack((x_axis, y_axis, z_axis, eye))
    return view_matrix

def render_cinematic_4d_orbit(scene_constants, scene_state, max_render_points=400000, width=640,
                              height=360, orbit_center=(0.4, 0.0, 0.0), orbit_radius=1.2,
                              camera_height=0.5, angle_start=np.pi/2):

    # 🌟 1. 拥抱新架构：直接提取 camera 层级的 keys，彻底干掉丑陋的 if 过滤
    camera_ids = sorted(scene_constants['camera'].keys())
    n_frames = len(scene_state[camera_ids[0]]['extrinsics'])

    # 🌟 2. 场景与渲染器初始化
    scene = pyrender.Scene(bg_color=[0.0, 0.0, 0.0, 1.0])
    cam_node = scene.add(pyrender.PerspectiveCamera(yfov=np.pi/3.0, aspectRatio=width/height), pose=np.eye(4))
    light_node = scene.add(pyrender.DirectionalLight(color=[1.0, 1.0, 1.0], intensity=4.0), pose=np.eye(4))
    renderer = pyrender.OffscreenRenderer(width, height)

    video_frames = []

    # 🌟 3. 极速 4D 渲染循环：加入语义化变量
    for frame_idx in tqdm(range(n_frames), desc=f"🎥 渲染 4D 运镜"):

        # --- A. 提取并融合多视角点云 ---
        points, colors = [], []
        for cam_id in camera_ids:
            cam_data = scene_constants['camera'][cam_id]
            cam_state = scene_state[cam_id]

            # 提取指定帧的 3D 空间点与色彩
            points_3d, colors_rgb = unproject_to_3d(
                cam_data['raw_depth'][frame_idx],
                cam_data['video_rgb'][frame_idx],
                cam_data['K_mat'],
                T_cam2world=cam_state['extrinsics'][frame_idx]
            )
            points.append(points_3d)
            colors.append(colors_rgb)

        points = np.vstack(points)
        colors = np.vstack(colors)

        # --- B. 显存保护机制 (降维打击，只需两行) ---
        sample_idx = np.random.permutation(len(points))[:max_render_points]
        points, colors = points[sample_idx], colors[sample_idx]

        # --- C. 计算平滑环绕运镜位姿 ---
        angle = angle_start + (frame_idx * np.pi / n_frames)
        eye_pos = [orbit_center[0] + orbit_radius * np.cos(angle),
                   orbit_center[1] + orbit_radius * np.sin(angle),
                   camera_height]

        viz_pose = get_look_at_matrix(eye_pos, orbit_center)
        scene.set_pose(cam_node, pose=viz_pose)
        scene.set_pose(light_node, pose=viz_pose)

        # --- D. 压入渲染、拔出销毁、盖水印一气呵成 ---
        mesh_node = scene.add(pyrender.Mesh.from_points(points, colors=colors))
        color_img, _ = renderer.render(scene, flags=pyrender.RenderFlags.RGBA)
        scene.remove_node(mesh_node)

        # 🌟 仅拷贝 RGB 通道打断只读锁定，跳过 Alpha 通道的冗余拷贝
        img_rgb = color_img[:, :, :3].copy()
        cv2.putText(img_rgb, f"Frame: {frame_idx:03d}", (30, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

        video_frames.append(img_rgb)

    renderer.delete()
    return video_frames

# ================= 主流程 =================
output_frames = render_cinematic_4d_orbit(
    scene_constants=active_scene_constants,
    scene_state=active_ultimate_scene_state,
)

media.show_video(output_frames, fps=15, codec='gif')

### 3D 点追踪提取

In [ ]:
# @title 截取最后48帧并执行 2D 点追踪 (全自动切片版)

def extract_last_n_frames(scene_constants, scene_state, n=48):
    """时序后置截断算子：将全域状态锁定至最后 N 帧，生成独立副本"""
    print(f"✂️ 启动时序后置截断：精准剥离全局最后 {n} 帧...")

    # 🌟 1. 绝对防御：深拷贝，绝不污染原版数据
    new_constants = copy.deepcopy(scene_constants)
    new_state = copy.deepcopy(scene_state)

    current_frames = len(new_constants['robot']['joint_positions'])
    if current_frames <= n:
        print(f"  ✅ 当前总帧数 ({current_frames} 帧) 已满足要求，无需截断。")
        return new_constants, new_state

    print(f"  🗑️ 原帧数: {current_frames} -> 目标帧数: {n}")

    # ==========================================
    # 🌟 2. 截断 Robot 本体运动学时序 (自动扫描)
    # ==========================================
    for key, value in new_constants['robot'].items():
        if isinstance(value, (list, np.ndarray)) and len(value) == current_frames:
            new_constants['robot'][key] = value[-n:]

    # ==========================================
    # 🌟 3. 截断全域 Camera 视角下的时序张量 (自动扫描)
    # ==========================================
    for cam_id, cam_data in new_constants['camera'].items():
        # 遍历时用 list() 包裹，防止后续 pop 操作引发字典迭代错误
        for key, value in list(cam_data.items()):
            if isinstance(value, (list, np.ndarray)) and len(value) == current_frames:
                # 无论是 video_rgb, raw_depth 还是 sam_masks，只要符合全长，统统一刀切！
                if isinstance(value, list):
                    cam_data[key] = value[-n:]
                else:
                    cam_data[key] = value[-n:]

        # 洗掉之前跑过的旧 tracks，防患于未然
        cam_data.pop('tracks_2d', None)
        cam_data.pop('vis_2d', None)

    # ==========================================
    # 🌟 4. 截断 3D 物理外参状态 (自动扫描)
    # ==========================================
    if new_state is not None:
        for cam_id, state_data in new_state.items():
            if 'extrinsics' in state_data and len(state_data['extrinsics']) == current_frames:
                state_data['extrinsics'] = state_data['extrinsics'][-n:]

    print(f"  ✅ 截断完毕！所有时序维度现已锁定为黄金的最后 {n} 帧。")
    return new_constants, new_state

# ================= 主流程 =================
sliced_scene_constants, sliced_scene_state = extract_last_n_frames(
    active_scene_constants,
    active_ultimate_scene_state,
    n=48
)

In [ ]:
# @title 截取随机连续48帧并执行 2D 点追踪 (全自动切片版)

import random
import copy
import numpy as np

def extract_random_n_frames(scene_constants, scene_state, n=48):
    """时序随机截断算子：将全域状态锁定至随机连续的 N 帧，生成独立副本"""
    print(f"✂️ 启动时序随机截断：精准剥离全局随机连续 {n} 帧...")

    # 🌟 1. 绝对防御：深拷贝，绝不污染原版数据
    new_constants = copy.deepcopy(scene_constants)
    new_state = copy.deepcopy(scene_state)

    current_frames = len(new_constants['robot']['joint_positions'])
    if current_frames <= n:
        print(f"  ✅ 当前总帧数 ({current_frames} 帧) 已满足要求，无需截断。")
        return new_constants, new_state

    # 🌟 计算随机起始索引
    start_idx = random.randint(0, current_frames - n)
    end_idx = start_idx + n

    print(f"  🎲 原帧数: {current_frames} -> 目标帧数: {n} (随机截取区间: [{start_idx}:{end_idx}])")

    # ==========================================
    # 🌟 2. 截断 Robot 本体运动学时序 (自动扫描)
    # ==========================================
    for key, value in new_constants['robot'].items():
        if isinstance(value, (list, np.ndarray)) and len(value) == current_frames:
            new_constants['robot'][key] = value[start_idx:end_idx]

    # ==========================================
    # 🌟 3. 截断全域 Camera 视角下的时序张量 (自动扫描)
    # ==========================================
    for cam_id, cam_data in new_constants['camera'].items():
        # 遍历时用 list() 包裹，防止后续 pop 操作引发字典迭代错误
        for key, value in list(cam_data.items()):
            if isinstance(value, (list, np.ndarray)) and len(value) == current_frames:
                # 无论是 video_rgb, raw_depth 还是 sam_masks，只要符合全长，统统一刀切！
                if isinstance(value, list):
                    cam_data[key] = value[start_idx:end_idx]
                else:
                    cam_data[key] = value[start_idx:end_idx]

        # 洗掉之前跑过的旧 tracks，防患于未然
        cam_data.pop('tracks_2d', None)
        cam_data.pop('vis_2d', None)

    # ==========================================
    # 🌟 4. 截断 3D 物理外参状态 (自动扫描)
    # ==========================================
    if new_state is not None:
        for cam_id, state_data in new_state.items():
            if 'extrinsics' in state_data and len(state_data['extrinsics']) == current_frames:
                state_data['extrinsics'] = state_data['extrinsics'][start_idx:end_idx]

    print(f"  ✅ 截断完毕！所有时序维度现已锁定为黄金的随机 {n} 帧。")
    return new_constants, new_state

# ================= 主流程 =================
sliced_scene_constants, sliced_scene_state = extract_random_n_frames(
    active_scene_constants,
    active_ultimate_scene_state,
    n=48
)

In [ ]:
# @title 2D 点追踪

def extract_2d_tracks(model, scene_constants):
    """利用 CoTracker3 对全视角视频进行 2D 稠密追踪"""
    print("  🎯 执行 CoTracker3 稠密追踪...")

    # 🌟 1. 结构升级：直接遍历全新的 camera 层级，彻底干掉冗余的 if 过滤
    for cam_id in scene_constants['camera']:
        cam_data = scene_constants['camera'][cam_id]

        # 🌟 2. 语义复苏：提取当前机位的 RGB 视频并进行张量维度转换
        video_tensor = torch.from_numpy(cam_data['video_rgb']).permute(0, 3, 1, 2)[None].float().to(device)

        # 🌟 3. 极速推理
        with torch.no_grad():
            pred_tracks, pred_vis = model(video_tensor, grid_size=30, grid_query_frame=0, backward_tracking=False)

        # 🌟 4. 优雅落盘：将追踪坐标与可见度掩码精准压入对应的相机口袋
        cam_data.update({
            'tracks_2d': pred_tracks[0].cpu().numpy(),
            'vis_2d': pred_vis[0].cpu().numpy()
        })

    return scene_constants

def render_2d_tracking_video(video_frames, tracks, visibility, global_colors=None, linewidth=3, tracks_leave_trace=20):
    """极速版 2D 渲染引擎 (终极防线：连线也严格遵守遮挡掩码)"""
    n_frames, n_points, _ = tracks.shape
    point_radius = int(linewidth * 2)

    # 获取原始视频的真实分辨率
    h_img, w_img = video_frames[0].shape[:2]

    track_pts = tracks.copy()

    # 1. 基础物理边界屏蔽 (屏蔽飞出屏幕范围的幻觉坐标)
    is_valid = (track_pts[..., 0] >= 0) & (track_pts[..., 0] < w_img) & \
               (track_pts[..., 1] >= 0) & (track_pts[..., 1] < h_img)

    # 🌟 核心防鬼影连线修复：画拖尾不仅要求在屏幕内，还必须没被物理遮挡！
    is_drawable = is_valid & visibility

    # 2. 拷贝视频帧防污染，并取整坐标
    video_frames = [f.copy() for f in video_frames]
    track_pts = np.round(track_pts).astype(np.int32)

    # 3. 极简调色板
    if global_colors is None:
        y_coords = tracks[0, :, 1]
        norm = plt.Normalize(y_coords.min(), y_coords.max())
        global_colors = plt.cm.gist_rainbow(norm(y_coords))[:, :3] * 255
    point_colors = [tuple(map(int, c)) for c in global_colors]

    # 4. 影视级逐帧渲染循环
    for t in range(n_frames):
        current_img = video_frames[t]
        trace_len = min(t, tracks_leave_trace)

        # --- A. 绘制带有透明度渐变的流星拖尾 ---
        for step in range(trace_len):
            past_t = t - trace_len + step
            alpha = (step / (trace_len + 1)) ** 2
            overlay = current_img.copy()

            # 🌟 修复：使用 is_drawable 替代 is_valid！
            # 只有在过去和现在都实打实可见的点，才允许连线！瞬间斩断穿模的直线！
            valid_edges = np.where(is_drawable[past_t] & is_drawable[past_t + 1])[0]
            for i in valid_edges:
                cv2.line(overlay, tuple(track_pts[past_t, i]), tuple(track_pts[past_t + 1, i]), point_colors[i], linewidth, cv2.LINE_AA)
            cv2.addWeighted(overlay, alpha, current_img, 1 - alpha, 0, current_img)

        # --- B. 绘制当前跟踪点 (区分可见与被遮挡) ---
        occ_overlay = current_img.copy()
        has_occlusion = False

        # 依然只处理当前帧在画面内的点
        active_points = np.where(is_valid[t])[0]

        for i in active_points:
            pt_coord = tuple(track_pts[t, i])

            if visibility[t, i]:
                # 可见点：实心圆
                cv2.circle(current_img, pt_coord, point_radius, point_colors[i], -1, cv2.LINE_AA)
            else:
                # 遮挡点：空心圆圈，并标记渲染透明遮罩
                cv2.circle(occ_overlay, pt_coord, point_radius, point_colors[i], 1, cv2.LINE_AA)
                has_occlusion = True

        # 一次性混合当前帧的所有被遮挡点，避免反复调用 addWeighted
        if has_occlusion:
            cv2.addWeighted(occ_overlay, 0.35, current_img, 0.65, 0, current_img)

    return video_frames

# ================= 主流程 =================
from cotracker.predictor import CoTrackerPredictor

# 🌟 1. 模型加载
model = CoTrackerPredictor(checkpoint="/content/co-tracker/weights/cotracker3_offline.pth").to(device)

sliced_scene_constants = extract_2d_tracks(model, sliced_scene_constants)

# 🌟 3. 极速渲染验证
all_viz_videos = []

for cam_id in sliced_scene_constants['camera']:
    print(f"🎨 正在渲染相机 {cam_id} 的 2D 追踪特效 (最后 48 帧)...")
    cam_data = sliced_scene_constants['camera'][cam_id]

    frames = render_2d_tracking_video(
        cam_data['video_rgb'],
        cam_data['tracks_2d'],
        cam_data['vis_2d']
    )
    all_viz_videos.append(np.array(frames))

# 🌟 4. 播放最后 48 帧的追踪效果
media.show_video(np.concatenate(all_viz_videos, axis=2), fps=15, codec='gif', height=256)

In [ ]:
# @title 2D 点追踪重投影可视化

def lift_tracks_to_3d(tracks_2d, vis_2d, depth, K_mat, extrinsics):
    """模块 A: 纯粹的 3D 升维算子 (严谨越界阻断)"""
    n_frames, n_points = tracks_2d.shape[:2]
    h_img, w_img = depth.shape[1:3]
    traj_3d = np.zeros((n_frames, n_points, 3))
    zs_src = np.zeros((n_frames, n_points))

    for t in range(n_frames):
        pts = tracks_2d[t]

        # 🌟 严谨越界检查：飞出屏幕的点绝对不能生硬 clip 去查边缘深度，必须直接判死刑！
        in_bounds = (pts[:, 0] >= 0) & (pts[:, 0] < w_img) & (pts[:, 1] >= 0) & (pts[:, 1] < h_img)

        us = np.clip(np.round(pts[:, 0]).astype(int), 0, w_img - 1)
        vs = np.clip(np.round(pts[:, 1]).astype(int), 0, h_img - 1)
        z_raw = depth[t, vs, us].copy()

        # 🌟 核心防鬼影：被遮挡的、或者飞出屏幕的，强制抹除深度，斩断幻觉源头
        invalid_mask = (~vis_2d[t]) | (~in_bounds)
        z_raw[invalid_mask] = 0.0

        zs_src[t] = z_raw
        traj_3d[t] = unproject_points_np(pts[:, 0], pts[:, 1], zs_src[t], K_mat, extrinsics[t])

    return traj_3d, zs_src

def project_to_camera(traj_3d, zs_src, depth, K_mat, extrinsics, depth_margin=0.05):
    """模块 B: 目标视角解算算子 (清爽版掩码逻辑 + 严谨越界修复)"""
    n_frames, n_points = traj_3d.shape[:2]
    h_img, w_img = depth.shape[1:3]

    # 初始化：所有不合法坐标默认丢到宇宙边缘
    proj_trk = np.full((n_frames, n_points, 2), -1000.0)
    proj_vis = np.zeros((n_frames, n_points), dtype=bool)

    for t in range(n_frames):
        u, v, z_pred = project_points_np(traj_3d[t], K_mat, extrinsics[t])

        # 🌟 逻辑重构：先筛选出真正的“物理合法点”
        valid_z = (zs_src[t] > 0.05) & (z_pred > 0.05)
        in_bounds = (u >= 0) & (u < w_img) & (v >= 0) & (v < h_img)
        valid_mask = valid_z & in_bounds

        # 仅对合法点赋予正确的 2D 坐标 (其余保持 -1000.0)
        proj_trk[t, valid_mask, 0] = u[valid_mask]
        proj_trk[t, valid_mask, 1] = v[valid_mask]

        # 🌟 遮挡计算：彻底杜绝对 -1000.0 坐标查深度的丑陋逻辑
        if valid_mask.any():
            # 🌟 终极修复：增加 np.clip 限制！
            # 防止 1279.6 被 round 成 1280 导致越界崩溃
            ui = np.clip(np.round(u[valid_mask]).astype(int), 0, w_img - 1)
            vi = np.clip(np.round(v[valid_mask]).astype(int), 0, h_img - 1)

            depth_sensor = depth[t, vi, ui]
            is_occ = (depth_sensor > 0) & (depth_sensor < z_pred[valid_mask] - depth_margin)

            # 将未被遮挡的合法点标记为 True
            proj_vis[t, valid_mask] = ~is_occ

    return proj_trk, proj_vis

# ================= 数据解耦模块 =================
def compute_reprojection_data(src_cam, scene_constants, scene_state):
    """数据推导模块：处理 3D 轨迹与各视角的重投影结果"""
    camera_ids = list(scene_constants['camera'].keys())
    src_data = scene_constants['camera'][src_cam]
    src_state = scene_state[src_cam]
    tracks_2d = src_data['tracks_2d']
    vis_src = src_data['vis_2d']

    traj_3d, zs_src = lift_tracks_to_3d(
        tracks_2d=tracks_2d, vis_2d=vis_src, depth=src_data['raw_depth'], K_mat=src_data['K_mat'], extrinsics=src_state['extrinsics']
    )

    trk_dict, vis_dict = {}, {}
    for tgt_cam in camera_ids:
        # 🌟 源视角特权：底层数据直接阻断重投影！原汁原味返回！
        if tgt_cam == src_cam:
            trk_dict[tgt_cam] = tracks_2d
            vis_dict[tgt_cam] = vis_src
        else:
            tgt_data = scene_constants['camera'][tgt_cam]
            tgt_state = scene_state[tgt_cam]
            trk_dict[tgt_cam], vis_dict[tgt_cam] = project_to_camera(
                traj_3d=traj_3d, zs_src=zs_src, depth=tgt_data['raw_depth'], K_mat=tgt_data['K_mat'], extrinsics=tgt_state['extrinsics']
            )

    return traj_3d, zs_src, vis_src, trk_dict, vis_dict, tracks_2d

# ================= 渲染解耦模块 =================
def render_all_tracks(src_cam, tracks_2d, vis_src, trk_dict, vis_dict, scene_constants):
    """画笔模块：无脑接盘渲染"""
    camera_ids = list(scene_constants['camera'].keys())
    h_img, w_img = scene_constants['camera'][src_cam]['video_rgb'].shape[1:3]

    y_vals = tracks_2d[0, :, 1]
    colors = plt.cm.gist_rainbow(plt.Normalize(y_vals.min(), y_vals.max())(y_vals))[:, :3] * 255

    res_vids = []
    tgt_size = (320, int(320 * h_img / w_img))
    text_org = (20, 50)

    for tgt_cam in camera_ids:
        # 🌟 现在画笔不再需要知道谁是源视角了。
        # 源视角拿到的 vis_dict 已经是 vis_src；跨视角拿到的是 proj_vis。
        # 统统执行 & vis_src 即可实现完美连带拦截。
        combined_vis = vis_dict[tgt_cam] & vis_src

        frames = render_2d_tracking_video(
            video_frames=scene_constants['camera'][tgt_cam]['video_rgb'],
            tracks=trk_dict[tgt_cam],
            visibility=combined_vis,
            global_colors=colors,
            linewidth=4
        )

        label = f"Src:{src_cam} -> Tgt:{tgt_cam}"
        cam_vid = []
        for img in frames:
            cv2.putText(img, label, text_org, cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 0), 4)
            cv2.putText(img, label, text_org, cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 2)
            cam_vid.append(cv2.resize(img, tgt_size))
        res_vids.append(np.array(cam_vid))

    return np.concatenate(res_vids, axis=2)

# ================= 主流程 =================
camera_grids = []

for cam_id in sliced_scene_constants['camera']:
    traj_3d, zs_src, vis_src, trk_dict, vis_dict, tracks_2d = compute_reprojection_data(
        cam_id, sliced_scene_constants, sliced_scene_state
    )

    grid_video = render_all_tracks(
        cam_id, tracks_2d, vis_src, trk_dict, vis_dict, sliced_scene_constants
    )
    camera_grids.append(grid_video)

if len(camera_grids) > 0:
    media.show_video(np.concatenate(camera_grids, axis=1), fps=15, codec='gif')
else:
    print("❌ 渲染失败，未能生成视频画面。")

In [ ]:
# @title 选定视点与 Source 视角基础升维 (Sliced 全机位遍历版)

camera_ids = list(sliced_scene_constants['camera'].keys())
print(f"🌍 启动全机位 Debug 升维，共检测到 {len(camera_ids)} 个相机视角。")

# 🌟 终极升级：使用大字典容纳所有相机的升维结果和它对应的 Target 物证
debug_pool = {}

for src_cam in camera_ids:
    # 自动包揽剩下的所有相机作为当前 Source 的 Target
    tgt_cams = [cam for cam in camera_ids if cam != src_cam]

    print(f"\n🔍 [Debug 模式] 当前 Source 视角: {src_cam} | Target 视角: {tgt_cams}")

    # 1. 获取当前 Source 数据与状态
    src_data = sliced_scene_constants['camera'][src_cam]
    src_state = sliced_scene_state[src_cam]
    src_tracks_2d = src_data['tracks_2d']
    src_vis_2d = src_data['vis_2d']
    T, N = src_vis_2d.shape

    # 2. 准备该 Source 对应的 Target 的数据和状态字典
    tgt_data_dict = {}
    tgt_state_dict = {}
    for tgt_cam in tgt_cams:
        tgt_data_dict[tgt_cam] = sliced_scene_constants['camera'][tgt_cam]
        tgt_state_dict[tgt_cam] = sliced_scene_state[tgt_cam]

    # 3. Source 视角基础 3D 升维 (使用大一统算子)
    # 🚨 核心修复：防鬼影参数 `src_vis_2d` 依然坚挺！
    src_traj_3d, src_zs = lift_tracks_to_3d(
        tracks_2d=src_tracks_2d,
        vis_2d=src_vis_2d,          # <--- 斩断幽灵飞线的绝对核心！
        depth=src_data['raw_depth'],
        K_mat=src_data['K_mat'],
        extrinsics=src_state['extrinsics']
    )
    src_pts_3d_t0 = src_traj_3d[0]

    # 4. 极致封箱：将所有推导结果压入大盘
    debug_pool[src_cam] = {
        'tgt_cams': tgt_cams,
        'tgt_data_dict': tgt_data_dict,
        'tgt_state_dict': tgt_state_dict,
        'src_traj_3d': src_traj_3d,
        'src_zs': src_zs,
        'src_pts_3d_t0': src_pts_3d_t0,
        'src_tracks_2d': src_tracks_2d,
        'src_vis_2d': src_vis_2d,
        'T': T,
        'N': N
    }

    print(f"  ✅ Source [{src_cam}] 视角升维完成。共处理 {N} 个追踪点，T={T} 帧。")
    print(f"     已为其准备好 {len(tgt_cams)} 个 Target 视角供下游交叉验证。")

print("\n🎉 所有机位的 Source 升维与 Target 分配全部完成！数据已安全存入 `debug_pool`。")

In [ ]:
# @title 跨视角物理投影与遮挡可视化 (生成 Queries - Sliced 全机位遍历版)

def generate_cross_view_queries(src_pts_3d_t0, src_zs_t0, tgt_depth_t0, K_tgt, extrinsics_tgt, depth_margin=0.01, device='cuda:0'):
    """
    阶段二：跨视角 Explicit Query 构造算子
    🌟 修复：加入 src_zs_t0 拦截无深度的幽灵点，并加上 np.clip 防止索引越界
    """
    h_img, w_img = tgt_depth_t0.shape

    # 1. 物理重投影：3D 世界 -> Target 像素平面
    u, v, z_pred = project_points_np(src_pts_3d_t0, K_tgt, extrinsics_tgt)

    # 🌟 防越界修复：加入 np.clip 防止极个别点 round 后恰好等于边界值
    ui = np.clip(np.round(u).astype(int), 0, w_img - 1)
    vi = np.clip(np.round(v).astype(int), 0, h_img - 1)

    # 🌟 核心杀招：必须满足源头有深度！
    valid_src_z = src_zs_t0 > 0

    # 2. 空间屏蔽
    in_bounds = (u >= 0) & (u < w_img) & (v >= 0) & (v < h_img) & (z_pred > 0) & valid_src_z
    valid_mask = np.zeros_like(in_bounds, dtype=bool)

    # 3. 深度遮挡裁决
    valid_u = ui[in_bounds]
    valid_v = vi[in_bounds]
    valid_z_pred = z_pred[in_bounds]

    z_sensor = tgt_depth_t0[valid_v, valid_u]

    # 🌟 极简去鬼影大法：
    # 1. z_sensor > 0: 目标视角下必须有真实的深度读数（杜绝把没有深度的白墙/机械臂当作透明）
    # 2. z_sensor >= valid_z_pred - depth_margin: 实际深度不能明显比理论深度浅（没被前面的物体遮挡）
    is_reliable_and_visible = (z_sensor > 0) & (z_sensor >= valid_z_pred - depth_margin)
    valid_mask[in_bounds] = is_reliable_and_visible

    # 4. 组装 CoTracker 弹药
    surviving_u = u[valid_mask]
    surviving_v = v[valid_mask]
    t_zeros = np.zeros_like(surviving_u)

    queries_np = np.stack([t_zeros, surviving_u, surviving_v], axis=-1)
    queries = torch.tensor(queries_np, dtype=torch.float32, device=device)

    return queries, valid_mask

print("🌍 启动全机位 Cross View Queries 生成与可视化...")

# 🌟 大循环：遍历池子里的每一个 Source 视角
for src_cam, pool_data in debug_pool.items():

    # 1. 优雅解包当前 Source 视角的所有前置数据
    tgt_cams = pool_data['tgt_cams']
    tgt_data_dict = pool_data['tgt_data_dict']
    tgt_state_dict = pool_data['tgt_state_dict']
    src_pts_3d_t0 = pool_data['src_pts_3d_t0']
    src_zs = pool_data['src_zs']
    N = pool_data['N']

    queries_dict = {}
    valid_query_mask_dict = {}

    # 2. 开启当前 Source 视角专属的多联屏画布
    # 画布可以更加紧凑了，因为不需要给 title 留空间
    fig, axes = plt.subplots(1, len(tgt_cams), figsize=(5 * len(tgt_cams), 3.5))
    if len(tgt_cams) == 1: axes = [axes]

    for idx, tgt_cam in enumerate(tgt_cams):
        # 摸取当前 Target 视角的数据
        tgt_data = tgt_data_dict[tgt_cam]
        tgt_state = tgt_state_dict[tgt_cam]

        # 运行 Query 生成算子
        queries, valid_query_mask = generate_cross_view_queries(
            src_pts_3d_t0=src_pts_3d_t0,
            src_zs_t0=src_zs[0],
            tgt_depth_t0=tgt_data['raw_depth'][0],
            K_tgt=tgt_data['K_mat'],
            extrinsics_tgt=tgt_state['extrinsics'][0],
            device=device
        )

        # 存入字典
        queries_dict[tgt_cam] = queries
        valid_query_mask_dict[tgt_cam] = valid_query_mask

        # 准备在 Target 的第 0 帧上画画
        tgt_img_t0 = tgt_data['video_rgb'][0].copy()

        u, v, z_pred = project_points_np(src_pts_3d_t0, tgt_data['K_mat'], tgt_state['extrinsics'][0])
        valid_count = 0

        # 保持原样：画绿点和红叉
        for i in range(N):
            ui, vi = int(np.round(u[i])), int(np.round(v[i]))
            if 0 <= ui < tgt_img_t0.shape[1] and 0 <= vi < tgt_img_t0.shape[0]:
                if valid_query_mask[i]:
                    # 🟢 存活的有效 Query：画绿色实心圆
                    cv2.circle(tgt_img_t0, (ui, vi), 4, (0, 255, 0), -1)
                    valid_count += 1
                else:
                    # 🔴 被剔除的 Query：画红色叉叉
                    cv2.drawMarker(tgt_img_t0, (ui, vi), (255, 0, 0), cv2.MARKER_CROSS, 10, 2)

        # 🌟 核心修改：把投射关系和 Survivors 直接打在画面左上角
        text_title = f"{src_cam} -> {tgt_cam}"
        text_survivors = f"Survivors: {valid_count} / {N}"

        # 画黑边描边防看不清，然后再画主色
        cv2.putText(tgt_img_t0, text_title, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 0), 4)
        cv2.putText(tgt_img_t0, text_title, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)

        cv2.putText(tgt_img_t0, text_survivors, (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 0), 4)
        cv2.putText(tgt_img_t0, text_survivors, (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)

        # 绘制到对应的子图上 (完全铺满)
        axes[idx].imshow(tgt_img_t0)
        axes[idx].axis('off')

    # 🌟 3. 极致封箱：把这一组生成的弹药压回 debug_pool
    debug_pool[src_cam]['queries_dict'] = queries_dict
    debug_pool[src_cam]['valid_query_mask_dict'] = valid_query_mask_dict

    # 🌟 删除了 plt.suptitle，因为标题已经融入画面
    plt.tight_layout()
    plt.subplots_adjust(wspace=0.02) # 可以让两张图贴得更紧凑一点
    plt.show()

In [ ]:
# @title Target 视角独立追踪可视化 (Sliced 全机位遍历版)

# 🌟 新增：用于收集所有 Source 视角的整行拼图
all_source_grids = []

# 🌟 大循环：遍历 debug_pool 里的每一个 Source
for src_cam, pool_data in debug_pool.items():

    # 1. 优雅解包当前 Source 的相关数据
    tgt_cams = pool_data['tgt_cams']
    tgt_data_dict = pool_data['tgt_data_dict']
    queries_dict = pool_data['queries_dict']
    valid_query_mask_dict = pool_data['valid_query_mask_dict']
    src_tracks_2d = pool_data['src_tracks_2d']
    src_vis_2d = pool_data['src_vis_2d'] # <--- 🌟 新增解包 Source 视角的遮挡掩码
    T = pool_data['T']
    N = pool_data['N']

    # 初始化存放当前 Source 对应所有 Target 追踪结果的字典
    tgt_tracks_2d_N_dict = {}
    tgt_vis_2d_N_dict = {}
    all_tgt_track_vids = []

    # 🌟 提取全局色彩基准：保证 Target 里每个点的颜色与 Source 里完全一致，方便 Debug 对比！
    y_vals = src_tracks_2d[0, :, 1]
    global_colors = plt.cm.gist_rainbow(plt.Normalize(y_vals.min(), y_vals.max())(y_vals))[:, :3] * 255

    # ==========================================
    # 🌟🌟🌟 新增核心：首先渲染 Source 视角的原生追踪画面，作为护法基准！
    # ==========================================
    src_data = sliced_scene_constants['camera'][src_cam]
    src_track_vid = render_2d_tracking_video(
        video_frames=src_data['video_rgb'],
        tracks=src_tracks_2d,
        visibility=src_vis_2d,
        global_colors=global_colors,
        linewidth=4,
    )

    cam_vid_src = []
    for img in src_track_vid:
        cv2.putText(img, f"Source: [{src_cam}]", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 0), 4)
        cv2.putText(img, f"Source: [{src_cam}]", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2) # 用绿色标记 Source，彰显它的真理地位
        cam_vid_src.append(img)

    all_tgt_track_vids.append(np.array(cam_vid_src))
    # ==========================================

    for tgt_cam in tgt_cams:
        tgt_data = tgt_data_dict[tgt_cam]
        queries = queries_dict[tgt_cam]
        valid_query_mask = valid_query_mask_dict[tgt_cam]

        # 🌟 防呆设计：如果该视角所有点都被遮挡了，直接生成空壳数组并跳过推断
        if len(queries) == 0:
            print(f"    ⚠️ 该视角下没有存活的 Queries，跳过追踪。")
            tgt_tracks_2d_N_dict[tgt_cam] = np.zeros((T, N, 2), dtype=np.float32)
            tgt_vis_2d_N_dict[tgt_cam] = np.zeros((T, N), dtype=bool)
            continue

        # 2. 张量准备
        tgt_video_tensor = torch.from_numpy(tgt_data['video_rgb']).permute(0, 3, 1, 2)[None].float().to(device)

        # 3. 独立追踪
        with torch.no_grad():
            pred_tracks_B, pred_vis_B = model(
                tgt_video_tensor,
                queries=queries[None, ...],
                backward_tracking=False
            )

        tgt_tracks_2d_M = pred_tracks_B[0].cpu().numpy()
        tgt_vis_2d_M = pred_vis_B[0].cpu().numpy()

        # 4. 对齐回原始 N 长度矩阵
        tgt_tracks_2d_N = np.zeros((T, N, 2), dtype=np.float32)
        tgt_vis_2d_N = np.zeros((T, N), dtype=bool)
        valid_indices = np.where(valid_query_mask)[0]
        tgt_tracks_2d_N[:, valid_indices, :] = tgt_tracks_2d_M
        tgt_vis_2d_N[:, valid_indices] = tgt_vis_2d_M

        # 🌟 压入字典，交给下阶段验证使用
        tgt_tracks_2d_N_dict[tgt_cam] = tgt_tracks_2d_N
        tgt_vis_2d_N_dict[tgt_cam] = tgt_vis_2d_N

        # 5. 渲染 Target 的 2D 追踪动画
        tgt_track_vid = render_2d_tracking_video(
            video_frames=tgt_data['video_rgb'],
            tracks=tgt_tracks_2d_N,
            visibility=tgt_vis_2d_N,
            global_colors=global_colors, # <--- 🌟 注入跨视角一致性色彩
            linewidth=4,
        )

        # 打上机位水印
        cam_vid = []
        for img in tgt_track_vid:
            cv2.putText(img, f"Tgt: [{tgt_cam}]", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 0), 4)
            cv2.putText(img, f"Tgt: [{tgt_cam}]", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)
            cam_vid.append(img)

        all_tgt_track_vids.append(np.array(cam_vid))

    # 🌟 6. 极致封箱：把这一组的追踪结果压回 debug_pool
    debug_pool[src_cam]['tgt_tracks_2d_N_dict'] = tgt_tracks_2d_N_dict
    debug_pool[src_cam]['tgt_vis_2d_N_dict'] = tgt_vis_2d_N_dict

    # 🌟 7. 横向拼接当前 Source 对应的所有视角，并压入全局列表
    if len(all_tgt_track_vids) > 1: # 大于 1 说明除了 Source 还有至少一个 Target 存活
        row_grid = np.concatenate(all_tgt_track_vids, axis=2) # Width 维度水平拼接
        all_source_grids.append(row_grid)
    else:
        print(f"❌ Source [{src_cam}] 对应的所有 Target 视角均无有效追踪点。")

# 🌟 8. 终极大一统展示：将所有 Source 的行拼接在一起，形成超级大盘
if len(all_source_grids) > 0:
    print("🎬 正在展示全景大一统追踪阵列...")
    # Height 维度垂直拼接，高度调大以适应多行显示
    media.show_video(np.concatenate(all_source_grids, axis=1), fps=15, height=540, codec='gif')

In [ ]:
# @title 闭环重投影与误差连线可视化 (全视角动态视频版 + 左上角出勤率大盘)

def lift_and_project_for_validation(tgt_tracks_2d, tgt_vis_2d, tgt_depths, K_tgt, ext_tgt, K_src, ext_src):
    """
    阶段三：逆向升维与闭环重投影算子
    """
    T, N, _ = tgt_tracks_2d.shape
    H, W = tgt_depths.shape[1:]

    tgt_traj_3d = np.zeros((T, N, 3), dtype=np.float32)
    proj_2d_in_src = np.zeros((T, N, 2), dtype=np.float32)
    valid_3d_mask = np.zeros((T, N), dtype=bool)

    for t in range(T):
        u = tgt_tracks_2d[t, :, 0]
        v = tgt_tracks_2d[t, :, 1]
        vis = tgt_vis_2d[t]

        ui = np.clip(np.round(u).astype(int), 0, W - 1)
        vi = np.clip(np.round(v).astype(int), 0, H - 1)
        z_tgt = tgt_depths[t, vi, ui]

        # 🌟 核心有效性筛选 (保留 1.5m 防爆截断)
        valid_mask_t = vis & (z_tgt > 0) & (z_tgt < 1.5)
        valid_3d_mask[t] = valid_mask_t

        if not valid_mask_t.any():
            continue

        pts_3d = unproject_points_np(
            u[valid_mask_t],
            v[valid_mask_t],
            z_tgt[valid_mask_t],
            K_tgt,
            ext_tgt[t]
        )
        tgt_traj_3d[t, valid_mask_t] = pts_3d

        u_proj, v_proj, z_proj = project_points_np(
            pts_3d,
            K_src,
            ext_src[t]
        )

        front_mask = z_proj > 0
        valid_indices = np.where(valid_mask_t)[0]

        valid_front_indices = valid_indices[front_mask]
        proj_2d_in_src[t, valid_front_indices, 0] = u_proj[front_mask]
        proj_2d_in_src[t, valid_front_indices, 1] = v_proj[front_mask]

        invalid_front_indices = valid_indices[~front_mask]
        valid_3d_mask[t, invalid_front_indices] = False

    return tgt_traj_3d, proj_2d_in_src, valid_3d_mask

print("🔄 启动全机位闭环重投影与误差分析大盘构建 (动态视频版 + 出勤率大盘)...")

all_source_vids = []

# 🌟 大循环：遍历 debug_pool 里的每一个 Source
for src_cam, pool_data in debug_pool.items():

    tgt_cams = pool_data['tgt_cams']
    tgt_data_dict = pool_data['tgt_data_dict']
    tgt_state_dict = pool_data['tgt_state_dict']
    src_tracks_2d = pool_data['src_tracks_2d']
    tgt_tracks_2d_N_dict = pool_data['tgt_tracks_2d_N_dict']
    tgt_vis_2d_N_dict = pool_data['tgt_vis_2d_N_dict']
    N = pool_data['N']
    T = pool_data['T']

    src_data = sliced_scene_constants['camera'][src_cam]
    src_state = sliced_scene_state[src_cam]
    src_video = src_data['video_rgb']

    tgt_traj_3d_N_dict = {}
    proj_2d_in_src_dict = {}
    valid_3d_mask_dict = {}

    # 1. 纯净的数据推演层：先算完所有目标相机的物证
    for tgt_cam in tgt_cams:
        tgt_tracks_2d_N = tgt_tracks_2d_N_dict[tgt_cam]
        tgt_vis_2d_N = tgt_vis_2d_N_dict[tgt_cam]
        tgt_data = tgt_data_dict[tgt_cam]
        tgt_state = tgt_state_dict[tgt_cam]

        tgt_traj_3d_N, proj_2d_in_src, valid_3d_mask = lift_and_project_for_validation(
            tgt_tracks_2d_N, tgt_vis_2d_N, tgt_data['raw_depth'],
            tgt_data['K_mat'], tgt_state['extrinsics'],
            src_data['K_mat'], src_state['extrinsics']
        )

        tgt_traj_3d_N_dict[tgt_cam] = tgt_traj_3d_N
        proj_2d_in_src_dict[tgt_cam] = proj_2d_in_src
        valid_3d_mask_dict[tgt_cam] = valid_3d_mask

    # 🌟 极致封箱：把这一组的闭环物证压回 debug_pool
    debug_pool[src_cam]['tgt_traj_3d_N_dict'] = tgt_traj_3d_N_dict
    debug_pool[src_cam]['proj_2d_in_src_dict'] = proj_2d_in_src_dict
    debug_pool[src_cam]['valid_3d_mask_dict'] = valid_3d_mask_dict

    # 🌟 上帝视角：提前计算每个 Target 相机视角下，各个点在 48 帧里的总出勤数
    total_attendance_dict = {}
    for tgt_cam in tgt_cams:
        total_attendance_dict[tgt_cam] = np.sum(valid_3d_mask_dict[tgt_cam], axis=0)

    # 2. 影视级渲染层：遍历时间轴，渲染该 Source 视角下的误差拉扯视频
    source_video_frames = []

    for t in tqdm(range(T), desc=f"🎥 渲染闭环追踪动态 [{src_cam}]"):
        frame_views = []
        base_img_t = src_video[t].copy()

        for tgt_cam in tgt_cams:
            img_t = base_img_t.copy()
            src_pts_t = src_tracks_2d[t]
            proj_pts_t = proj_2d_in_src_dict[tgt_cam][t]
            mask_t = valid_3d_mask_dict[tgt_cam][t]

            error_dists = []

            for i in range(N):
                if mask_t[i]:
                    p_src = tuple(np.round(src_pts_t[i]).astype(int))
                    p_proj = tuple(np.round(proj_pts_t[i]).astype(int))

                    dist = np.linalg.norm(src_pts_t[i] - proj_pts_t[i])
                    error_dists.append(dist)

                    # 原点：黄色 | 闭环点：洋红 | 连线：红色 (误差越大线越长)
                    cv2.circle(img_t, p_src, 3, (0, 255, 255), -1)
                    cv2.circle(img_t, p_proj, 3, (255, 0, 255), -1)
                    cv2.line(img_t, p_src, p_proj, (0, 0, 255), 2)

            avg_err = np.mean(error_dists) if error_dists else 0.0

            # 🌟 提前计算能够满足“出勤率 >= 5”法庭底线要求的人数
            potential_survivors = np.sum(total_attendance_dict[tgt_cam] >= 5)

            # 绘制信息 HUD
            header1 = f"Tgt: {tgt_cam} | Surv(Now): {len(error_dists)}"
            header2 = f"Avg Err: {avg_err:.1f} px"
            header3 = f">=5 Frames: {potential_survivors}"

            cv2.putText(img_t, header1, (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 0), 4)
            cv2.putText(img_t, header1, (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)

            cv2.putText(img_t, header2, (20, 90), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 0), 4)
            cv2.putText(img_t, header2, (20, 90), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 255), 2)

            cv2.putText(img_t, header3, (20, 130), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 0), 4)
            cv2.putText(img_t, header3, (20, 130), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)

            frame_views.append(img_t)

        # 横向拼接当前 Source 视角比对的多个 Target
        row_concat = np.concatenate(frame_views, axis=1)
        source_video_frames.append(row_concat)

    all_source_vids.append(np.array(source_video_frames))

if len(all_source_vids) > 0:
    print("\n🎬 正在展示全景闭环误差动态 (黄点:2D追踪原点 | 洋红:闭环重投影点 | 红线:物理飘移误差)")
    # 将多个 Source 的对比视频纵向堆叠，生成一个超级矩阵视频，高度调整为 384
    media.show_video(np.concatenate(all_source_vids, axis=1), fps=15, codec='gif', height=540)

In [ ]:
# @title 🦾 URDF 物理刚体轨迹生成器 (边缘排爆安全版)
import numpy as np
import cv2
import pybullet as p
from scipy.spatial.transform import Rotation as R
from tqdm import tqdm

class URDFKinematicsTracker:
    def __init__(self, pb_renderer):
        """专职利用 URDF 模型正向推演机械臂/夹爪轨迹的引擎"""
        self.pb_renderer = pb_renderer

    def get_link_transform(self, obj_id, link_id):
        """提取 PyBullet 中指定连杆的 4x4 变换矩阵"""
        if link_id == -1:
            pos, orn = p.getBasePositionAndOrientation(obj_id)
        else:
            state = p.getLinkState(obj_id, link_id)
            pos, orn = state[0], state[1]
        T = np.eye(4)
        T[:3, :3] = R.from_quat(orn).as_matrix()
        T[:3, 3] = pos
        return T

    def extract_robot_tracks(self, src_cam, scene_constants, scene_state, safe_margin=7):
        print(f"\n🦾 启动 URDF 纯物理运动学追踪 | 视角: [{src_cam}]")

        src_data = scene_constants['camera'][src_cam]
        src_state = scene_state[src_cam]
        K_mat = src_data['K_mat']
        extrinsics = src_state['extrinsics']
        h_img, w_img = src_data['video_rgb'][0].shape[:2]
        n_frames = len(src_data['video_rgb'])

        # 我们依然以 CoTracker 第 0 帧撒的点作为初始种子
        tracks_2d_t0 = src_data['tracks_2d'][0]

        # =========================================================
        # 1. 时光倒流：恢复第 0 帧状态，寻找落在机器人上的种子点
        # =========================================================
        joint_angles_t0 = scene_constants['robot']['joint_positions'][0]
        gripper_state_t0 = scene_constants['robot']['gripper_positions'][0]
        self.pb_renderer.update_robot_pose(joint_angles_t0, gripper_state=gripper_state_t0)

        # 渲染第 0 帧的 Mask 和 Depth
        cam_pos = extrinsics[0][:3, 3]
        target_pos = cam_pos + extrinsics[0][:3, 2]
        view_matrix = p.computeViewMatrix(cam_pos, target_pos, -extrinsics[0][:3, 1])
        proj_matrix = self.pb_renderer._get_projection_matrix(K_mat, w_img, h_img)

        _, _, _, depth_buffer, seg_buffer = p.getCameraImage(
            w_img, h_img, viewMatrix=view_matrix, projectionMatrix=proj_matrix,
            renderer=p.ER_BULLET_HARDWARE_OPENGL,
            flags=p.ER_SEGMENTATION_MASK_OBJECT_AND_LINKINDEX
        )

        # 解析 URDF 原生深度
        urdf_depth_t0 = 0.1 / (10.0 - 9.99 * np.reshape(depth_buffer, (h_img, w_img)))
        urdf_depth_t0 = np.where(urdf_depth_t0 < 9.9, urdf_depth_t0, 0.0)

        # 解析 Segmentation
        seg_array = np.reshape(seg_buffer, (h_img, w_img)).astype(np.int32)
        obj_ids = seg_array & 0xFFFFFF
        link_ids = (seg_array >> 24) - 1

        # 锁定落在本体或夹爪上的像素
        is_robot = (obj_ids == self.pb_renderer.robot_id) | (obj_ids == self.pb_renderer.ghost_id)

        # 🌟 核心升级：对机器人 Mask 进行腐蚀 (Erosion)，安全排爆边缘点！
        # 使用 safe_margin (默认 7x7) 的核向内收缩，确保追踪点深埋在机械臂内部
        kernel = np.ones((safe_margin, safe_margin), np.uint8)
        is_robot_safe = cv2.erode(is_robot.astype(np.uint8), kernel, iterations=1) > 0

        # 判断追踪点哪些属于机器人
        u0 = np.clip(np.round(tracks_2d_t0[:, 0]).astype(int), 0, w_img - 1)
        v0 = np.clip(np.round(tracks_2d_t0[:, 1]).astype(int), 0, h_img - 1)

        # 🌟 使用内缩后的绝对安全 Mask 进行筛选
        on_robot_mask = is_robot_safe[v0, u0]

        robot_indices = np.where(on_robot_mask)[0]

        if len(robot_indices) == 0:
            print("  ⚠️ 第 0 帧没有识别到任何安全的机械臂/夹爪点，无法执行物理追踪！")
            return None, None, None, None

        print(f"  🔍 成功在第 0 帧捕获 {len(robot_indices)} 个【安全内缩】的刚体表面点！")

        # 提取这些点对应的物体、连杆和深度
        robot_objs = obj_ids[v0[robot_indices], u0[robot_indices]]
        robot_links = link_ids[v0[robot_indices], u0[robot_indices]]
        z0 = urdf_depth_t0[v0[robot_indices], u0[robot_indices]]

        # 升维得到初始 3D 世界坐标
        pts_world_t0 = unproject_points_np(
            tracks_2d_t0[robot_indices, 0],
            tracks_2d_t0[robot_indices, 1],
            z0, K_mat, extrinsics[0]
        )

        # 将世界坐标绑定到局部连杆坐标系 (Local Frame)
        unique_parts = set(zip(robot_objs, robot_links))
        local_pts_dict = {}
        for obj_id, link_id in unique_parts:
            part_mask = (robot_objs == obj_id) & (robot_links == link_id)
            part_pts = pts_world_t0[part_mask]

            T_link_world_t0 = self.get_link_transform(obj_id, link_id)
            T_world_link_t0 = np.linalg.inv(T_link_world_t0)

            P_homo = np.hstack([part_pts, np.ones((len(part_pts), 1))]).T
            local_pts_dict[(obj_id, link_id)] = (part_mask, T_world_link_t0 @ P_homo)

        # =========================================================
        # 2. 正向推演：生成全序列 3D 与 2D 轨迹
        # =========================================================
        traj_3d = np.zeros((n_frames, len(robot_indices), 3), dtype=np.float32)
        traj_2d = np.zeros((n_frames, len(robot_indices), 2), dtype=np.float32)
        vis_2d = np.zeros((n_frames, len(robot_indices)), dtype=bool)

        for t in tqdm(range(n_frames), desc="  ⚙️ 正向运动学推演与遮挡计算"):
            # 更新物理引擎关节
            self.pb_renderer.update_robot_pose(
                scene_constants['robot']['joint_positions'][t],
                gripper_state=scene_constants['robot']['gripper_positions'][t]
            )

            # 根据连杆的位移，计算最新的 3D 世界坐标
            for obj_id, link_id in unique_parts:
                part_mask, P_local_homo = local_pts_dict[(obj_id, link_id)]
                T_link_world_t = self.get_link_transform(obj_id, link_id)
                P_world_t = T_link_world_t @ P_local_homo
                traj_3d[t, part_mask, :] = P_world_t[:3, :].T

            # 拍平到当前相机的 2D 平面
            u_t, v_t, z_pred_t = project_points_np(traj_3d[t], K_mat, extrinsics[t])
            traj_2d[t, :, 0] = u_t
            traj_2d[t, :, 1] = v_t

            # =========================================================
            # 3. 动态可见性判定 (越界 + 自遮挡 + 环境遮挡)
            # =========================================================
            urdf_depth_t = self.pb_renderer.render_depth(extrinsics[t], K_mat, w_img, h_img)
            raw_depth_t = src_data['raw_depth'][t]

            ui = np.clip(np.round(u_t).astype(int), 0, w_img - 1)
            vi = np.clip(np.round(v_t).astype(int), 0, h_img - 1)

            # 法则一：不能飞出屏幕
            in_bounds = (u_t >= 0) & (u_t < w_img) & (v_t >= 0) & (v_t < h_img) & (z_pred_t > 0)

            # 法则二：不能跑到机械臂自己背后 (2cm 容错)
            z_urdf = urdf_depth_t[vi, ui]
            not_self_occ = (z_urdf > 0) & (z_pred_t <= z_urdf + 0.015)

            # 法则三：不能被环境中其他物体遮挡 (3cm 容错)
            z_sensor = raw_depth_t[vi, ui]
            not_env_occ = ~((z_sensor > 0) & (z_pred_t > z_sensor + 0.02))

            vis_2d[t] = in_bounds & not_self_occ & not_env_occ

        return traj_3d, traj_2d, vis_2d, robot_indices

In [ ]:
# @title 👁️ URDF 物理 3D 轨迹的全域 2D 验证阵列
import numpy as np
import cv2
import matplotlib.pyplot as plt
from tqdm import tqdm
import mediapy as media

def project_and_check_visibility(traj_3d, tgt_cam, scene_constants, scene_state, pb_renderer):
    """将 3D 轨迹投影到目标相机，并执行严格的三重物理遮挡检测"""
    tgt_data = scene_constants['camera'][tgt_cam]
    tgt_state = scene_state[tgt_cam]
    K_mat = tgt_data['K_mat']
    extrinsics = tgt_state['extrinsics']
    n_frames = len(traj_3d)
    n_pts = traj_3d.shape[1]
    h_img, w_img = tgt_data['video_rgb'][0].shape[:2]

    tgt_traj_2d = np.zeros((n_frames, n_pts, 2), dtype=np.float32)
    tgt_vis_2d = np.zeros((n_frames, n_pts), dtype=bool)

    for t in range(n_frames):
        # 1. 强制同步物理引擎姿态，以便渲染深度图做遮挡判断
        current_joints = scene_constants['robot']['joint_positions'][t]
        current_gripper = scene_constants['robot']['gripper_positions'][t]
        pb_renderer.update_robot_pose(current_joints, gripper_state=current_gripper)

        # 2. 将 3D 轨迹拍平到目标相机的 2D 平面
        u_t, v_t, z_pred_t = project_points_np(traj_3d[t], K_mat, extrinsics[t])
        tgt_traj_2d[t, :, 0] = u_t
        tgt_traj_2d[t, :, 1] = v_t

        urdf_depth_t = pb_renderer.render_depth(extrinsics[t], K_mat, w_img, h_img)
        raw_depth_t = tgt_data['raw_depth'][t]

        ui = np.clip(np.round(u_t).astype(int), 0, w_img - 1)
        vi = np.clip(np.round(v_t).astype(int), 0, h_img - 1)

        # 3. 三重判定：视场越界 + 自遮挡 + 环境遮挡
        in_bounds = (u_t >= 0) & (u_t < w_img) & (v_t >= 0) & (v_t < h_img) & (z_pred_t > 0)
        z_urdf = urdf_depth_t[vi, ui]
        # 只要点比 URDF 表面深超过 1cm，说明它已经跑到机械臂内部或背面了
        not_self_occ = (z_urdf > 0) & (z_pred_t <= z_urdf + 0.01)
        z_sensor = raw_depth_t[vi, ui]
        # 只要点比真实世界的遮挡物深超过 1cm，说明它被桌子/其他物品挡住了
        not_env_occ = ~((z_sensor > 0) & (z_pred_t > z_sensor + 0.01))

        tgt_vis_2d[t] = in_bounds & not_self_occ & not_env_occ

    return tgt_traj_2d, tgt_vis_2d

def render_urdf_cross_view(src_cam, tgt_cam, scene_constants, scene_state, pb_renderer, traj_2d, vis_2d, robot_indices, y_vals_src):
    """渲染单一交叉视角的动画帧"""
    cam_data = scene_constants['camera'][tgt_cam]
    cam_state = scene_state[tgt_cam]
    video_frames = cam_data['video_rgb']
    K_mat = cam_data['K_mat']
    extrinsics = cam_state['extrinsics']
    n_frames = len(video_frames)
    h_img, w_img = video_frames[0].shape[:2]

    # 🌟 核心：使用 Source 视角的 Y 坐标初始化调色板，保证同一批点在不同视角下颜色绝对一致！
    norm = plt.Normalize(y_vals_src.min() - 1, y_vals_src.max() + 1)
    point_colors = plt.cm.gist_rainbow(norm(y_vals_src))[:, :3] * 255

    out_frames = []
    # 进度条描述精简一下，避免太长
    for t in tqdm(range(n_frames), desc=f"渲染 [Src: {src_cam[-4:]} -> Tgt: {tgt_cam[-4:]}]"):
        img = video_frames[t].copy()

        current_joints = scene_constants['robot']['joint_positions'][t]
        current_gripper = scene_constants['robot']['gripper_positions'][t]
        pb_renderer.update_robot_pose(current_joints, gripper_state=current_gripper)

        robot_mask = pb_renderer.render_mask(extrinsics[t], K_mat, w_img, h_img) > 0

        overlay = img.copy()
        overlay[robot_mask] = [50, 150, 255]
        img = cv2.addWeighted(img, 0.6, overlay, 0.4, 0)

        for i in range(len(robot_indices)):
            if vis_2d[t, i]:
                pt = (int(np.round(traj_2d[t, i, 0])), int(np.round(traj_2d[t, i, 1])))
                color = tuple(map(int, point_colors[i]))
                cv2.circle(img, pt, 4, (0, 0, 0), -1, cv2.LINE_AA)
                cv2.circle(img, pt, 3, color, -1, cv2.LINE_AA)

        label = f"Src:{src_cam[-6:]} -> Tgt:{tgt_cam[-6:]}"
        cv2.putText(img, label, (15, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 4)
        cv2.putText(img, label, (15, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

        # 缩小尺寸以适应九宫格展示，防止内存爆炸
        img = cv2.resize(img, (320, int(320 * h_img / w_img)))
        out_frames.append(img)

    return out_frames

# ================== 启动全域交叉验证 ==================
if 'pb_renderer_ultimate' not in locals():
    pb_renderer_ultimate = PyBulletRenderer_Robotiq()

urdf_tracker = URDFKinematicsTracker(pb_renderer_ultimate)
camera_ids = list(sliced_scene_constants['camera'].keys())
all_grid_rows = []

print(f"\n🚀 启动全域交叉投影阵列构建...")

for src_cam in camera_ids:
    print(f"\n" + "="*60)
    print(f"🌍 正在以 [{src_cam}] 为源视角，提取 3D 物理轨迹...")

    # 1. 从 Source 视角提取原生 3D 物理轨迹
    traj_3d, src_traj_2d, src_vis_2d, robot_indices = urdf_tracker.extract_robot_tracks(
        src_cam, sliced_scene_constants, sliced_scene_state
    )

    if src_traj_2d is None:
        continue

    y_vals_src = src_traj_2d[0, :, 1]
    row_videos = []

    # 2. 将这批 3D 点投影到所有的 Target 视角（包括自己）
    for tgt_cam in camera_ids:
        if src_cam == tgt_cam:
            tgt_traj_2d, tgt_vis_2d = src_traj_2d, src_vis_2d
        else:
            print(f"  🎯 向目标视角 [{tgt_cam}] 投影并计算遮挡...")
            tgt_traj_2d, tgt_vis_2d = project_and_check_visibility(
                traj_3d, tgt_cam, sliced_scene_constants, sliced_scene_state, pb_renderer_ultimate
            )

        # 3. 渲染该交叉视角的视频
        vid_frames = render_urdf_cross_view(
            src_cam, tgt_cam, sliced_scene_constants, sliced_scene_state,
            pb_renderer_ultimate, tgt_traj_2d, tgt_vis_2d, robot_indices, y_vals_src
        )
        row_videos.append(np.array(vid_frames))

    # 将同一个 Source 下的多个 Target 横向拼接 (Width维度)
    all_grid_rows.append(np.concatenate(row_videos, axis=2))

# 4. 纵向拼接所有 Source 的行 (Height维度)
if len(all_grid_rows) > 0:
    print("\n🎬 渲染完毕！正在展示 3x3 URDF 交叉投影大盘：")
    combined_video = np.concatenate(all_grid_rows, axis=1)
    media.show_video(combined_video, fps=15, codec='gif', height=540)
else:
    print("\n❌ 提取失败。")

In [ ]:
# @title 👁️ 15 票全域共识视觉追踪器 (Track A)
import numpy as np
import torch
import cv2
import matplotlib.pyplot as plt
from tqdm import tqdm
import mediapy as media
import warnings
from scipy.ndimage import gaussian_filter1d

class ConsensusVisualTracker:
    def __init__(self, model, device='cuda:0'):
        """专职利用多视角时空共识，提纯环境点云 (Track A) 的引擎"""
        self.model = model
        self.device = device

    def extract_env_tracks(self, src_cam, env_indices, scene_constants, scene_state, max_env_depth=5.0, occ_margin=0.10):
        print(f"\n👁️ 启动 15 票全域时空共识追踪 | 主视角: [{src_cam}] | 追踪点数: {len(env_indices)}")

        src_data = scene_constants['camera'][src_cam]
        src_state = scene_state[src_cam]
        T_total = len(src_data['video_rgb'])
        N_env = len(env_indices)

        # 15 票的时空刻度 (3个相机 × 5个关键帧)
        self.keyframes = [0, T_total//4, T_total//2, 3*T_total//4, T_total-1]
        self.camera_ids = list(scene_constants['camera'].keys())

        # 提取当前主视角下，纯净环境点的基础 2D 追踪大盘
        src_tracks = src_data['tracks_2d'][:, env_indices, :]
        src_vis = src_data['vis_2d'][:, env_indices]

        vote_traj_dict = {}
        vote_vis_dict = {}

        # =========================================================
        # 1. 跨时空双向追踪 (生成 15 票 3D 轨迹)
        # =========================================================
        for tgt_cam in self.camera_ids:
            tgt_data = scene_constants['camera'][tgt_cam]
            tgt_state = scene_state[tgt_cam]
            h_img, w_img = tgt_data['video_rgb'][0].shape[:2]

            # 视频推入显存，准备接受批量追踪请求
            video_tensor = torch.from_numpy(tgt_data['video_rgb']).permute(0, 3, 1, 2)[None].float().to(self.device)

            for t_k in tqdm(self.keyframes, desc=f"  🚀 跨时空追踪 -> {tgt_cam[-4:]}"):
                vote_name = f"{tgt_cam}_keyframe_{t_k}"

                # --- a. 局部时空锚定 ---
                u_src, v_src = src_tracks[t_k, :, 0], src_tracks[t_k, :, 1]
                vis_src = src_vis[t_k]

                ui_src = np.clip(np.round(u_src).astype(int), 0, w_img - 1)
                vi_src = np.clip(np.round(v_src).astype(int), 0, h_img - 1)
                z_src = src_data['raw_depth'][t_k, vi_src, ui_src]

                # 🌟 环境点的物理界限截断设为 5.0 米
                valid_src = vis_src & (z_src > 0.05) & (z_src < max_env_depth)

                # 升维到 3D
                pts_3d_tk = unproject_points_np(u_src, v_src, z_src, src_data['K_mat'], src_state['extrinsics'][t_k])

                # 投影到 Target
                u_tgt, v_tgt, z_pred = project_points_np(pts_3d_tk, tgt_data['K_mat'], tgt_state['extrinsics'][t_k])
                ui_tgt = np.clip(np.round(u_tgt).astype(int), 0, w_img - 1)
                vi_tgt = np.clip(np.round(v_tgt).astype(int), 0, h_img - 1)
                z_sensor = tgt_data['raw_depth'][t_k, vi_tgt, ui_tgt]

                # 遮挡检测 (10cm 遮挡容差应对远景噪声)
                in_bounds = (u_tgt >= 0) & (u_tgt < w_img) & (v_tgt >= 0) & (v_tgt < h_img)
                is_visible = (z_sensor > 0) & (z_sensor >= z_pred - occ_margin)
                valid_query_mask = valid_src & in_bounds & is_visible
                valid_indices = np.where(valid_query_mask)[0]

                if len(valid_indices) == 0:
                    vote_traj_dict[vote_name] = np.zeros((T_total, N_env, 3), dtype=np.float32)
                    vote_vis_dict[vote_name] = np.zeros((T_total, N_env), dtype=bool)
                    continue

                # --- b. 启动 CoTracker 逆流与顺流双向追踪 ---
                queries_np = np.stack([np.full_like(u_tgt[valid_indices], t_k),
                                       u_tgt[valid_indices],
                                       v_tgt[valid_indices]], axis=-1)
                queries_tensor = torch.tensor(queries_np, dtype=torch.float32, device=self.device)[None, ...]

                with torch.no_grad():
                    pred_tracks, pred_vis = self.model(
                        video_tensor, queries=queries_tensor, backward_tracking=True
                    )

                tgt_tracks_N = np.zeros((T_total, N_env, 2), dtype=np.float32)
                tgt_vis_N = np.zeros((T_total, N_env), dtype=bool)
                tgt_tracks_N[:, valid_indices, :] = pred_tracks[0].cpu().numpy()
                tgt_vis_N[:, valid_indices] = pred_vis[0].cpu().numpy()

                # --- c. 追踪结果重升维到 3D ---
                traj_3d_K = np.zeros((T_total, N_env, 3), dtype=np.float32)
                vis_3d_K = np.zeros((T_total, N_env), dtype=bool)

                for t in range(T_total):
                    u_t, v_t = tgt_tracks_N[t, :, 0], tgt_tracks_N[t, :, 1]
                    vis_t = tgt_vis_N[t]
                    ui_t = np.clip(np.round(u_t).astype(int), 0, w_img - 1)
                    vi_t = np.clip(np.round(v_t).astype(int), 0, h_img - 1)
                    z_t = tgt_data['raw_depth'][t, vi_t, ui_t]

                    valid_t = vis_t & (z_t > 0.05) & (z_t < max_env_depth) & (u_t >= 0) & (u_t < w_img) & (v_t >= 0) & (v_t < h_img)
                    vis_3d_K[t] = valid_t
                    if valid_t.any():
                        traj_3d_K[t, valid_t] = unproject_points_np(
                            u_t[valid_t], v_t[valid_t], z_t[valid_t],
                            tgt_data['K_mat'], tgt_state['extrinsics'][t]
                        )

                vote_traj_dict[vote_name] = traj_3d_K
                vote_vis_dict[vote_name] = vis_3d_K

# =========================================================
        # 2. 召唤最高法庭，执行 15 票 NaN 中位数融合与平滑
        # =========================================================
        print(f"  ⚖️ 追踪完毕！召唤最高法庭，执行 15 票全景时空融合...")
        stacked_traj = np.stack(list(vote_traj_dict.values()), axis=0)
        stacked_vis = np.stack(list(vote_vis_dict.values()), axis=0)

        # 把不可见的点设为 NaN，利用中位数自动免疫追踪失败的幻觉点！
        stacked_traj_nan = np.where(stacked_vis[..., None], stacked_traj, np.nan)

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            unified_traj_3d = np.nanmedian(stacked_traj_nan, axis=0)

        view_counts = np.sum(stacked_vis, axis=0)
        unified_vis = view_counts >= 3 # 至少需要 3 票支持才算合法 3D 点

        # 🌟🌟🌟 核心修复：引入 Pandas 双向插值取代归零塌陷！🌟🌟🌟
        print(f"  🌊 执行时序高斯平滑 (利用 Pandas 修复 NaN 引力塌陷)...")
        import pandas as pd
        smoothed_traj_3d = unified_traj_3d.copy()

        # 展平以便用 Pandas 进行插值
        df = pd.DataFrame(smoothed_traj_3d.reshape(T_total, -1))
        # 双向线性插值，填补遮挡造成的 NaN 空洞
        df = df.interpolate(method='linear', limit_direction='both')
        interpolated_traj = df.to_numpy().reshape(T_total, N_env, 3)

        # 对连贯的物理轨迹进行轻度高斯平滑
        smoothed_traj_3d = gaussian_filter1d(interpolated_traj, sigma=2.0, axis=0)
        # 把原本就不可见的点重新设回 NaN，保持物理遮挡的逻辑闭环
        smoothed_traj_3d[~unified_vis] = np.nan

        # =========================================================
        # 3. 将白金 3D 真值点重投影回主视角的 2D 平面，完成闭环！
        # =========================================================
        # 🌟 修改：把最后一维改成 3，用来存放 Z 深度！
        refined_2d_traj = np.zeros((T_total, N_env, 3), dtype=np.float32)
        for t in range(T_total):
            u_r, v_r, z_r = project_points_np(smoothed_traj_3d[t], src_data['K_mat'], src_state['extrinsics'][t])
            refined_2d_traj[t, :, 0] = u_r
            refined_2d_traj[t, :, 1] = v_r
            refined_2d_traj[t, :, 2] = z_r # 存入深度供画师遮挡判定

        return smoothed_traj_3d, refined_2d_traj, unified_vis

# 🌟 增加 raw_depth 传入
def visualize_track_A_consensus(src_cam, original_2d, refined_2d, unified_vis, video_frames, raw_depth):
    print(f"\n🎨 正在渲染 Track A (全量环境点) 共识提纯前后对比视频...")
    n_frames = len(video_frames)
    h_img, w_img = video_frames[0].shape[:2]

    y_vals = original_2d[0, :, 1]
    norm = plt.Normalize(y_vals.min() - 1, y_vals.max() + 1)
    point_colors = plt.cm.gist_rainbow(norm(y_vals))[:, :3] * 255

    out_frames = []
    for t in tqdm(range(n_frames), desc="渲染验证帧"):
        img = video_frames[t].copy()

        for i in range(len(y_vals)):
            if unified_vis[t, i]:
                pt_raw = (int(np.round(original_2d[t, i, 0])), int(np.round(original_2d[t, i, 1])))
                u_ref, v_ref = refined_2d[t, i, 0], refined_2d[t, i, 1]
                z_pred = refined_2d[t, i, 2] # 取出 3D 预测深度

                # 安全越界截断
                ui = np.clip(int(np.round(u_ref)), 0, w_img - 1)
                vi = np.clip(int(np.round(v_ref)), 0, h_img - 1)

                # 🌟 核心：X光透视修复 (Z-Buffer Occlusion Test)
                z_sensor = raw_depth[t, vi, ui]
                # 如果相机在这里拍到了物理表面(z_sensor>0)，并且该表面明显挡在我们的环境点前面(容错5cm)
                if z_sensor > 0 and z_sensor < z_pred - 0.05:
                    continue # 被前景(机械臂/桌子)挡住了，隐身！不画！

                pt_refined = (ui, vi)
                color = tuple(map(int, point_colors[i]))

                dist = np.linalg.norm(np.array(pt_raw) - np.array(pt_refined))
                if dist > 3.0:
                    cv2.line(img, pt_raw, pt_refined, (0, 0, 255), 2, cv2.LINE_AA)
                    cv2.circle(img, pt_raw, 3, (180, 180, 180), 1, cv2.LINE_AA)

                cv2.circle(img, pt_refined, 4, (0, 0, 0), -1, cv2.LINE_AA)
                cv2.circle(img, pt_refined, 3, color, -1, cv2.LINE_AA)

        cv2.putText(img, f"15-Vote Consensus | {src_cam}", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 4)
        cv2.putText(img, f"15-Vote Consensus | {src_cam}", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        out_frames.append(img)

    return out_frames

# ================== 启动验证 ==================
# 1. 确保引擎实例化
consensus_tracker = ConsensusVisualTracker(cotracker_model, device)

if 'pb_renderer_ultimate' not in locals():
    pb_renderer_ultimate = PyBulletRenderer_Robotiq()

camera_ids = list(sliced_scene_constants['camera'].keys())
all_consensus_videos = []

# 2. 大循环：遍历所有相机视角
for cam_id in camera_ids:
    print(f"\n" + "="*60)
    print(f"🎬 开始处理并提取视角: [{cam_id}] 的环境点...")

    # 3. 🌟 核心升级：利用物理引擎精准剔除机械臂，获取所有全域环境点
    cam_data = sliced_scene_constants['camera'][cam_id]
    cam_state = sliced_scene_state[cam_id]
    h_img, w_img = cam_data['video_rgb'][0].shape[:2]

    # 同步第 0 帧的物理姿态
    joint_angles_t0 = sliced_scene_constants['robot']['joint_positions'][0]
    gripper_state_t0 = sliced_scene_constants['robot']['gripper_positions'][0]
    pb_renderer_ultimate.update_robot_pose(joint_angles_t0, gripper_state=gripper_state_t0)

    # 渲染第 0 帧机械臂的物理遮罩
    robot_mask = pb_renderer_ultimate.render_mask(cam_state['extrinsics'][0], cam_data['K_mat'], w_img, h_img) > 0

    # 🌟 新增：对物理 Mask 进行膨胀 (Dilation)，安全排爆边缘点
    # 使用 15x15 的核，相当于在机械臂外围建立约 7 个像素的隔离警戒区
    kernel = np.ones((15, 15), np.uint8)
    robot_mask_dilated = cv2.dilate(robot_mask.astype(np.uint8), kernel, iterations=1) > 0

    # 提取 2D 追踪点的初始坐标
    u0 = np.clip(np.round(cam_data['tracks_2d'][0, :, 0]).astype(int), 0, w_img-1)
    v0 = np.clip(np.round(cam_data['tracks_2d'][0, :, 1]).astype(int), 0, h_img-1)

    # 取反获取所有不在机械臂上的点 (🌟 这里使用膨胀后的安全 Mask)
    is_env_point = ~robot_mask_dilated[v0, u0]

    # 安全起见，同时要求这些点在初始帧有正常的物理深度读数
    z_t0 = cam_data['raw_depth'][0]
    has_valid_depth = (z_t0[v0, u0] > 0.05) & (z_t0[v0, u0] < 5.0)

    # 🔥 锁定所有完美的环境点，解除点数限制！
    test_env_indices = np.where(is_env_point & has_valid_depth)[0]
    print(f"  🌌 成功剥离机械臂 (已应用形态学膨胀警戒区)！提取到 {len(test_env_indices)} 个全量环境点送审。")

    # 4. 执行 Track A 独立推演
    smoothed_3d, refined_2d, unified_vis = consensus_tracker.extract_env_tracks(
        src_cam=cam_id,
        env_indices=test_env_indices,
        scene_constants=sliced_scene_constants,
        scene_state=sliced_scene_state
    )

    # 5. 渲染误差拉扯对比动画
    original_2d = sliced_scene_constants['camera'][cam_id]['tracks_2d'][:, test_env_indices, :]
    video_frames = sliced_scene_constants['camera'][cam_id]['video_rgb']
    raw_depth = sliced_scene_constants['camera'][cam_id]['raw_depth'] # 🌟 取出深度图

    # 🌟 传入深度图
    consensus_video = visualize_track_A_consensus(cam_id, original_2d, refined_2d, unified_vis, video_frames, raw_depth)
    all_consensus_videos.append(np.array(consensus_video))

# 6. 横向拼接并展示大盘
if len(all_consensus_videos) > 0:
    print("\n🎬 渲染完毕！正在展示所有视角的 Track A 全量环境点共识追踪验证阵列：")
    combined_video = np.concatenate(all_consensus_videos, axis=2)
    media.show_video(combined_video, fps=15, codec='gif', height=256)
else:
    print("\n❌ 提取失败。")

In [ ]:
# @title 👁️ 环境点提纯与拉丝纠错可视化 (Standalone Track A 终极过滤版 + X光透视修复 + 抗闪烁)
import numpy as np
import cv2
import matplotlib.pyplot as plt
from tqdm import tqdm
import mediapy as media

# 🌟 增加 raw_depth 传入
def render_standalone_env_survivors(src_cam, original_2d, refined_2d, unified_vis, video_frames, raw_depth):
    print(f"\n🎨 正在渲染视角 [{src_cam}] 的环境点拉丝纠错动画...")
    n_frames = len(video_frames)
    h_img, w_img = video_frames[0].shape[:2]

    y_vals = original_2d[0, :, 1]
    # 防除零保护
    if len(y_vals) > 1 and y_vals.max() > y_vals.min():
        norm = plt.Normalize(y_vals.min() - 1, y_vals.max() + 1)
    else:
        norm = plt.Normalize(y_vals.min() - 1, y_vals.min() + 1)
    point_colors = plt.cm.gist_rainbow(norm(y_vals))[:, :3] * 255

    out_frames = []
    for t in tqdm(range(n_frames), desc="渲染验证帧"):
        img = video_frames[t].copy()

        for i in range(len(y_vals)):
            # 🌟 核心：此时传入的已经是经历了双重严苛过滤的白金点！
            if unified_vis[t, i]:
                pt_raw = (int(np.round(original_2d[t, i, 0])), int(np.round(original_2d[t, i, 1])))

                u_ref, v_ref = refined_2d[t, i, 0], refined_2d[t, i, 1]
                z_pred = refined_2d[t, i, 2] # 取出 3D 预测深度

                # 安全越界截断
                ui = np.clip(int(np.round(u_ref)), 0, w_img - 1)
                vi = np.clip(int(np.round(v_ref)), 0, h_img - 1)

                # 🌟 核心：X光透视修复 (Z-Buffer Occlusion Test)
                z_sensor = raw_depth[t, vi, ui]
                if z_sensor > 0 and z_sensor < z_pred - 0.05:
                    continue # 被前景(机械臂/桌子)挡住了，隐身！不画！

                pt_refined = (ui, vi)
                color = tuple(map(int, point_colors[i]))

                dist = np.linalg.norm(np.array(pt_raw) - np.array(pt_refined))

                # 允许微小修正 (呈现拉丝效果)，拒绝满屏乱扯
                if dist > 2.0:
                    cv2.line(img, pt_raw, pt_refined, (0, 0, 255), 2, cv2.LINE_AA)
                    cv2.circle(img, pt_raw, 3, (180, 180, 180), 1, cv2.LINE_AA) # 灰点代表原始幻觉点

                # 绘制坚如磐石的物理共识点
                cv2.circle(img, pt_refined, 4, (0, 0, 0), -1, cv2.LINE_AA)
                cv2.circle(img, pt_refined, 3, color, -1, cv2.LINE_AA)

        # 盖戳
        cv2.putText(img, f"Env Survivors | {src_cam}", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 4)
        cv2.putText(img, f"Env Survivors | {src_cam}", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        out_frames.append(img)

    return out_frames

# ================== 启动验证 ==================
if 'pb_renderer_ultimate' not in locals():
    pb_renderer_ultimate = PyBulletRenderer_Robotiq()

# 确保追踪器就绪
consensus_tracker = ConsensusVisualTracker(cotracker_model, device)
camera_ids = list(sliced_scene_constants['camera'].keys())
all_env_videos = []

for cam_id in camera_ids:
    print(f"\n" + "="*60)
    print(f"🎬 开始处理并提取视角: [{cam_id}] 的纯净环境点...")

    cam_data = sliced_scene_constants['camera'][cam_id]
    cam_state = sliced_scene_state[cam_id]
    h_img, w_img = cam_data['video_rgb'][0].shape[:2]

    # =========================================================
    # 1. 物理引擎精准剔除机械臂 (使用膨胀安全警戒区)
    # =========================================================
    joint_angles_t0 = sliced_scene_constants['robot']['joint_positions'][0]
    gripper_state_t0 = sliced_scene_constants['robot']['gripper_positions'][0]
    pb_renderer_ultimate.update_robot_pose(joint_angles_t0, gripper_state=gripper_state_t0)

    robot_mask = pb_renderer_ultimate.render_mask(cam_state['extrinsics'][0], cam_data['K_mat'], w_img, h_img) > 0
    kernel = np.ones((15, 15), np.uint8)
    robot_mask_dilated = cv2.dilate(robot_mask.astype(np.uint8), kernel, iterations=1) > 0

    # 提取 2D 追踪点的初始坐标
    u0 = np.clip(np.round(cam_data['tracks_2d'][0, :, 0]).astype(int), 0, w_img-1)
    v0 = np.clip(np.round(cam_data['tracks_2d'][0, :, 1]).astype(int), 0, h_img-1)

    # 锁定物理意义上的“纯环境点”
    is_env_point = ~robot_mask_dilated[v0, u0]
    z_t0 = cam_data['raw_depth'][0]
    has_valid_depth = (z_t0[v0, u0] > 0.05) & (z_t0[v0, u0] < 5.0)

    test_env_indices = np.where(is_env_point & has_valid_depth)[0]
    print(f"  🌌 成功剥离机械臂！提取到 {len(test_env_indices)} 个环境点送审。")

    # =========================================================
    # 2. 执行 Track A 独立推演，获取法庭审判结果
    # =========================================================
    smoothed_3d, refined_2d, unified_vis = consensus_tracker.extract_env_tracks(
        src_cam=cam_id,
        env_indices=test_env_indices,
        scene_constants=sliced_scene_constants,
        scene_state=sliced_scene_state
    )

    # =========================================================
    # 3. 🌟 终极漏斗过滤：彻底剔除不合理的垃圾点与闪烁点！
    # =========================================================
    original_2d = sliced_scene_constants['camera'][cam_id]['tracks_2d'][:, test_env_indices, :]
    T_frames = unified_vis.shape[0]

    # a) 计算所有时间步内的出勤率
    valid_counts = np.sum(unified_vis, axis=0)

    # b) 计算 2D 追踪点与 3D 共识点降维后的绝对物理漂移误差
    D_2D = np.linalg.norm(original_2d - refined_2d[:, :, :2], axis=-1)
    max_drift = np.max(np.where(unified_vis, D_2D, 0), axis=0)

    # c) 🌟 新增：抗闪烁检测 (Flicker Detection)
    # 计算相邻两帧可见性状态发生反转 (True->False 或 False->True) 的次数
    flicker_counts = np.sum(unified_vis[:-1] != unified_vis[1:], axis=0)

    # 🔪 判决底线：
    # 1. 至少在 5 帧里可见
    # 2. 哪怕有一次漂移超过 10 像素，一律死刑
    # 3. 闪烁次数超过总帧数的 10%，判定为边缘不稳定点，一律死刑
    survivor_mask = (valid_counts >= 5) & (max_drift < 10.0) & (flicker_counts <= T_frames * 0.10)
    golden_local_indices = np.where(survivor_mask)[0]

    print(f"  🔪 [终极过滤] 审判完毕！剔除严重漂移点与高频闪烁点，幸存: {len(golden_local_indices)} 个白金点。")

    if len(golden_local_indices) == 0:
        print(f"  ⚠️ 视角 [{cam_id}] 无点幸存，跳过渲染。")
        continue

    # 真正对数组进行切片降维，仅保留存活点
    original_2d_golden = original_2d[:, golden_local_indices, :]
    refined_2d_golden = refined_2d[:, golden_local_indices, :]
    unified_vis_golden = unified_vis[:, golden_local_indices]

    # =========================================================
    # 4. 渲染极品点阵的误差拉扯对比动画
    # =========================================================
    video_frames = sliced_scene_constants['camera'][cam_id]['video_rgb']
    raw_depth = sliced_scene_constants['camera'][cam_id]['raw_depth'] # 🌟 取出深度图

    # 🌟 传入 raw_depth 供遮挡判定
    survivor_video = render_standalone_env_survivors(
        cam_id, original_2d_golden, refined_2d_golden, unified_vis_golden, video_frames, raw_depth
    )
    all_env_videos.append(np.array(survivor_video))

# =========================================================
# 5. 横向拼接并展示大盘
# =========================================================
if len(all_env_videos) > 0:
    print("\n🎬 渲染完毕！正在展示所有视角的 Track A 纯环境点共识追踪与拉丝纠错阵列：")
    combined_video = np.concatenate(all_env_videos, axis=2)
    media.show_video(combined_video, fps=15, codec='gif', height=256)
else:
    print("\n❌ 提取失败。")

In [ ]:
# @title 环境点终极过滤尸检大盘 (新版漏斗规则适配)

import matplotlib.pyplot as plt
import numpy as np
import cv2

def env_funnel_autopsy(src_cam, original_2d, refined_2d, unified_vis, scene_data):
    print(f"\n" + "="*60)
    print(f"🔍 [Track A 环境点漏斗剖析] 主视角: {src_cam}")

    T_frames, N = unified_vis.shape

    # ==========================================
    # 🕵️‍♂️ 1. 计算最新版 Track A 的各项终极审判指标
    # ==========================================
    valid_counts = np.sum(unified_vis, axis=0)
    D_2D = np.linalg.norm(original_2d - refined_2d[:, :, :2], axis=-1)
    max_drift = np.max(np.where(unified_vis, D_2D, 0), axis=0)
    flicker_counts = np.sum(unified_vis[:-1] != unified_vis[1:], axis=0)

    # ==========================================
    # 🔪 2. 漏斗过滤机制 (互斥归类死因)
    # ==========================================
    mask_step0_all = np.ones(N, dtype=bool)

    # [死因 A]: 15票共识出勤率不足 5 帧 (通常是被长时间遮挡的背景)
    killed_by_attendance = valid_counts < 5

    # [死因 B]: 发生了 >= 10px 的严重物理拉扯 (通常是被机械臂带跑的幻觉点)
    passed_A = ~killed_by_attendance
    killed_by_drift = passed_A & (max_drift >= 10.0)

    # [死因 C]: 在可见/不可见之间高频闪烁 (通常是边缘反光、深度极度不稳定的噪点)
    passed_AB = passed_A & (~killed_by_drift)
    killed_by_flicker = passed_AB & (flicker_counts > T_frames * 0.10)

    # [幸存者]: 历经三道关卡的白金真值点
    survivors = passed_AB & (~killed_by_flicker)

    print(f"  🔻 [Step 0] 初始提取纯环境点数: {N} 个")
    print(f"  🩸 [Step 1] 死于「出勤率 < 5 帧」: 剔除 {np.sum(killed_by_attendance)} 个")
    print(f"  🩸 [Step 2] 死于「严重漂移 >= 10px」: 剔除 {np.sum(killed_by_drift)} 个")
    print(f"  🩸 [Step 3] 死于「高频闪烁 > 10%」: 剔除 {np.sum(killed_by_flicker)} 个")
    print(f"  🏆 [Step 4] 历经重重考验的白金真值点: 幸存 {np.sum(survivors)} 个")

    # ==========================================
    # 🎨 3. 5 联屏死因空间分布可视化
    # ==========================================
    fig, axes = plt.subplots(1, 5, figsize=(28, 4.5))
    fig.suptitle(f"Track A (Environment) Funnel Autopsy: Source [{src_cam}]", fontsize=18, fontweight='bold', y=1.05)

    img_t0 = scene_data['video_rgb'][0].copy()
    pts_t0 = original_2d[0]

    def draw_points(ax, img, mask, color, title):
        canvas = img.copy()
        for i in range(N):
            if mask[i]:
                x, y = int(np.round(pts_t0[i, 0])), int(np.round(pts_t0[i, 1]))
                if 0 <= x < canvas.shape[1] and 0 <= y < canvas.shape[0]:
                    cv2.circle(canvas, (x, y), 5, color, -1)
                    cv2.circle(canvas, (x, y), 5, (0,0,0), 1)
        ax.imshow(canvas)
        ax.set_title(title, fontsize=13)
        ax.axis('off')

    # 图 1: 全量点 (灰色)
    draw_points(axes[0], img_t0, mask_step0_all, (200, 200, 200), f"All Env Points ({N})")
    # 图 2: 死于出勤率 (黄色)
    draw_points(axes[1], img_t0, killed_by_attendance, (255, 255, 0), f"Killed: Low Attendance ({np.sum(killed_by_attendance)})")
    # 图 3: 死于严重拉扯 (红色)
    draw_points(axes[2], img_t0, killed_by_drift, (255, 0, 0), f"Killed: Severe Drift ({np.sum(killed_by_drift)})")
    # 图 4: 死于高频闪烁 (橙色)
    draw_points(axes[3], img_t0, killed_by_flicker, (255, 165, 0), f"Killed: High Flicker ({np.sum(killed_by_flicker)})")
    # 图 5: 最终幸存 (绿色)
    draw_points(axes[4], img_t0, survivors, (0, 255, 0), f"Platinum Survivors ({np.sum(survivors)})")

    plt.tight_layout()
    plt.show()

# =========================================================
# 🚀 启动执行：一键自动化跑通 Track A 追踪与尸检
# =========================================================
print("🚀 启动 [Track A] 环境点漏斗过滤尸检大盘...")

if 'pb_renderer_ultimate' not in locals():
    pb_renderer_ultimate = PyBulletRenderer_Robotiq()

if 'consensus_tracker' not in locals():
    consensus_tracker = ConsensusVisualTracker(cotracker_model, device)

camera_ids = list(sliced_scene_constants['camera'].keys())

for cam_id in camera_ids:
    cam_data = sliced_scene_constants['camera'][cam_id]
    cam_state = sliced_scene_state[cam_id]
    h_img, w_img = cam_data['video_rgb'][0].shape[:2]

    # 1. 精准提取 Track A 纯环境点 (URDF 边缘排爆)
    joint_angles_t0 = sliced_scene_constants['robot']['joint_positions'][0]
    gripper_state_t0 = sliced_scene_constants['robot']['gripper_positions'][0]
    pb_renderer_ultimate.update_robot_pose(joint_angles_t0, gripper_state=gripper_state_t0)

    robot_mask = pb_renderer_ultimate.render_mask(cam_state['extrinsics'][0], cam_data['K_mat'], w_img, h_img) > 0
    kernel = np.ones((15, 15), np.uint8)
    robot_mask_dilated = cv2.dilate(robot_mask.astype(np.uint8), kernel, iterations=1) > 0

    u0 = np.clip(np.round(cam_data['tracks_2d'][0, :, 0]).astype(int), 0, w_img-1)
    v0 = np.clip(np.round(cam_data['tracks_2d'][0, :, 1]).astype(int), 0, h_img-1)

    is_env_point = ~robot_mask_dilated[v0, u0]
    z_t0 = cam_data['raw_depth'][0]
    has_valid_depth = (z_t0[v0, u0] > 0.05) & (z_t0[v0, u0] < 5.0)

    test_env_indices = np.where(is_env_point & has_valid_depth)[0]

    # 2. 执行 Consensus 15 票独立推演获取追踪与修复掩码
    smoothed_3d, refined_2d, unified_vis = consensus_tracker.extract_env_tracks(
        src_cam=cam_id,
        env_indices=test_env_indices,
        scene_constants=sliced_scene_constants,
        scene_state=sliced_scene_state
    )

    original_2d = sliced_scene_constants['camera'][cam_id]['tracks_2d'][:, test_env_indices, :]

    # 3. 交给法医进行尸检分析并渲染
    env_funnel_autopsy(
        src_cam=cam_id,
        original_2d=original_2d,
        refined_2d=refined_2d,
        unified_vis=unified_vis,
        scene_data=cam_data
    )

In [ ]:
# @title 终极验证：双轨大一统白金真值闭环拉丝动态 (Track A + Track B)

import numpy as np
import cv2
import matplotlib.pyplot as plt
from tqdm import tqdm
import mediapy as media

# 🌟 新增 orig_vis 参数，接入 CoTracker 原生可见度
def render_combined_platinum_tracks(cam_id, orig_2d, proj_2d, vis, orig_vis, video_frames):
    print(f"🎨 正在渲染全域白金点阵拉丝纠错动画 [{cam_id}]...")
    T_frames, N = vis.shape
    out_frames = []

    for t in tqdm(range(T_frames), desc=f"🎥 渲染白金点闭环动态 [{cam_id}]"):
        img_t = video_frames[t].copy()
        error_dists = []

        # 获取当前帧在 3D 物理世界中所有理论上可见的点
        visible_indices = np.where(vis[t])[0]

        for i in visible_indices:
            p_proj = tuple(np.round(proj_2d[t, i]).astype(int))

            # 🌟 无论如何，画出绝对的物理真值点 (洋红)
            cv2.circle(img_t, p_proj, 3, (255, 0, 255), -1, cv2.LINE_AA)

            # 🌟 绝杀：只有当 CoTracker 也认为这个点可见时，才画出它的预测和误差拉丝！
            # 彻底杜绝 CoTracker 丢失目标后产生的 [0,0] 满屏飞线 Bug！
            if orig_vis[t, i]:
                p_src = tuple(np.round(orig_2d[t, i]).astype(int))
                dist = np.linalg.norm(orig_2d[t, i] - proj_2d[t, i])
                error_dists.append(dist)

                # 允许微小修正拉丝，拒绝满屏乱扯 (大于 2px 才连线，画面更清爽)
                if dist > 2.0:
                    cv2.line(img_t, p_src, p_proj, (0, 0, 255), 2, cv2.LINE_AA)

                # 画出 CoTracker 的 2D 预测点 (黄点)
                cv2.circle(img_t, p_src, 3, (0, 255, 255), -1, cv2.LINE_AA)

        avg_err = np.mean(error_dists) if error_dists else 0.0

        # 绘制信息 HUD
        header1 = f"Cam: [{cam_id}] | Survivors: {len(visible_indices)}/{N}"
        header2 = f"Avg Err: {avg_err:.1f} px"

        cv2.putText(img_t, header1, (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 0), 4)
        cv2.putText(img_t, header1, (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)

        cv2.putText(img_t, header2, (20, 90), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 0), 4)
        cv2.putText(img_t, header2, (20, 90), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 255), 2)

        out_frames.append(img_t)

    return np.array(out_frames)


# ================== 启动全域大一统验证 ==================
print("🔄 启动双轨大一统闭环重投影终极验收 (URDF 物理点 + 15票共识白金点)...")

if 'pb_renderer_ultimate' not in locals():
    pb_renderer_ultimate = PyBulletRenderer_Robotiq()

# 确保双引擎就绪
urdf_tracker = URDFKinematicsTracker(pb_renderer_ultimate)
consensus_tracker = ConsensusVisualTracker(cotracker_model, device)

camera_ids = list(sliced_scene_constants['camera'].keys())
all_source_vids = []

for cam_id in camera_ids:
    print(f"\n" + "="*60)
    print(f"🎬 开始处理并合并视角: [{cam_id}] 的双轨极品真值点...")

    cam_data = sliced_scene_constants['camera'][cam_id]
    cam_state = sliced_scene_state[cam_id]
    video_frames = cam_data['video_rgb']
    raw_depth = cam_data['raw_depth']
    tracks_2d = cam_data['tracks_2d']
    vis_2d = cam_data['vis_2d'] # 🌟 提取原生可见度
    h_img, w_img = video_frames[0].shape[:2]
    T_frames = len(video_frames)

    orig_2d_all = []
    proj_2d_all = []
    vis_all = []
    orig_vis_all = [] # 🌟 新增：收集原生可见度

    # =========================================================
    # 🦾 Track B: 提取 URDF 机械臂纯物理点 (内部自带安全内缩防爆)
    # =========================================================
    traj_3d_rob, proj_2d_rob, vis_rob, robot_indices = urdf_tracker.extract_robot_tracks(
        cam_id, sliced_scene_constants, sliced_scene_state
    )
    if proj_2d_rob is not None:
        orig_2d_rob = tracks_2d[:, robot_indices, :]
        orig_vis_rob = vis_2d[:, robot_indices] # 🌟
        orig_2d_all.append(orig_2d_rob)
        proj_2d_all.append(proj_2d_rob)
        vis_all.append(vis_rob)
        orig_vis_all.append(orig_vis_rob) # 🌟

    # =========================================================
    # 👁️ Track A: 提取纯环境点并执行终极法庭过滤
    # =========================================================
    # 1. 膨胀机械臂 Mask，锁定安全的纯环境区域
    joint_angles_t0 = sliced_scene_constants['robot']['joint_positions'][0]
    gripper_state_t0 = sliced_scene_constants['robot']['gripper_positions'][0]
    pb_renderer_ultimate.update_robot_pose(joint_angles_t0, gripper_state=gripper_state_t0)

    robot_mask = pb_renderer_ultimate.render_mask(cam_state['extrinsics'][0], cam_data['K_mat'], w_img, h_img) > 0
    kernel = np.ones((15, 15), np.uint8)
    robot_mask_dilated = cv2.dilate(robot_mask.astype(np.uint8), kernel, iterations=1) > 0

    u0 = np.clip(np.round(tracks_2d[0, :, 0]).astype(int), 0, w_img-1)
    v0 = np.clip(np.round(tracks_2d[0, :, 1]).astype(int), 0, h_img-1)
    is_env_point = ~robot_mask_dilated[v0, u0]
    has_valid_depth = (raw_depth[0, v0, u0] > 0.05) & (raw_depth[0, v0, u0] < 5.0)
    test_env_indices = np.where(is_env_point & has_valid_depth)[0]

    # 2. 15 票共识推演
    smoothed_3d, refined_2d, unified_vis = consensus_tracker.extract_env_tracks(
        src_cam=cam_id, env_indices=test_env_indices,
        scene_constants=sliced_scene_constants, scene_state=sliced_scene_state
    )

    # 3. 终极漏斗过滤 (出勤率、严重漂移、高频闪烁)
    original_2d_env_raw = tracks_2d[:, test_env_indices, :]
    valid_counts = np.sum(unified_vis, axis=0)
    D_2D = np.linalg.norm(original_2d_env_raw - refined_2d[:, :, :2], axis=-1)
    max_drift = np.max(np.where(unified_vis, D_2D, 0), axis=0)
    flicker_counts = np.sum(unified_vis[:-1] != unified_vis[1:], axis=0)

    survivor_mask = (valid_counts >= 5) & (max_drift < 10.0) & (flicker_counts <= T_frames * 0.10)
    golden_indices = np.where(survivor_mask)[0]

    # 4. 提取极品点并追加 X 光透视修复 (Z-Buffer Occlusion Test)
    if len(golden_indices) > 0:
        orig_2d_env = original_2d_env_raw[:, golden_indices, :]
        orig_vis_env = vis_2d[:, test_env_indices][:, golden_indices] # 🌟
        proj_2d_env = refined_2d[:, golden_indices, :2]
        z_pred_env = refined_2d[:, golden_indices, 2]
        vis_env = unified_vis[:, golden_indices].copy()

        # 为环境点打上物理遮挡补丁
        for t in range(T_frames):
            ui = np.clip(np.round(proj_2d_env[t, :, 0]).astype(int), 0, w_img - 1)
            vi = np.clip(np.round(proj_2d_env[t, :, 1]).astype(int), 0, h_img - 1)
            z_sensor = raw_depth[t, vi, ui]
            xray_occluded = (z_sensor > 0) & (z_sensor < z_pred_env[t] - 0.05)
            vis_env[t] = vis_env[t] & (~xray_occluded)

        orig_2d_all.append(orig_2d_env)
        proj_2d_all.append(proj_2d_env)
        vis_all.append(vis_env)
        orig_vis_all.append(orig_vis_env) # 🌟

    # =========================================================
    # 🤝 双轨大一统拼接与渲染
    # =========================================================
    if not orig_2d_all:
        print(f"  ⚠️ 视角 [{cam_id}] 无任何存活点，跳过。")
        continue

    combined_orig_2d = np.concatenate(orig_2d_all, axis=1)
    combined_proj_2d = np.concatenate(proj_2d_all, axis=1)
    combined_vis = np.concatenate(vis_all, axis=1)
    combined_orig_vis = np.concatenate(orig_vis_all, axis=1) # 🌟

    print(f"  🏆 双轨合璧成功！机器臂点 + 环境点共计留存 {combined_vis.shape[1]} 个白金真值点。")

    vid_frames = render_combined_platinum_tracks(
        cam_id, combined_orig_2d, combined_proj_2d, combined_vis, combined_orig_vis, video_frames
    )
    all_source_vids.append(np.array(vid_frames))

# =========================================================
# 5. 横向拼接并展示大盘
# =========================================================
if len(all_source_vids) > 0:
    print("\n🎬 正在展示全景闭环误差动态 (黄点:2D追踪原点 | 洋红:闭环重投影点 | 红线:物理飘移误差)")
    combined_video = np.concatenate(all_source_vids, axis=2)
    media.show_video(combined_video, fps=15, codec='gif', height=256)
else:
    print("\n❌ 提取失败。")

In [ ]:
# @title 👁️ 终极 2D 追踪多视角全景投影阵列 (双轨大一统版)
import numpy as np
import cv2
import matplotlib.pyplot as plt
from tqdm import tqdm
import mediapy as media

print("🔄 正在提取双轨大一统真值，并渲染多视角 2D 追踪流星阵列...")

# 确保追踪引擎就绪
if 'pb_renderer_ultimate' not in locals():
    pb_renderer_ultimate = PyBulletRenderer_Robotiq()
if 'urdf_tracker' not in locals():
    urdf_tracker = URDFKinematicsTracker(pb_renderer_ultimate)
if 'consensus_tracker' not in locals():
    consensus_tracker = ConsensusVisualTracker(cotracker_model, device)

camera_ids = list(sliced_scene_constants['camera'].keys())
all_grid_rows = []

for src_cam in camera_ids:
    print(f"\n" + "="*60)
    print(f"🎬 正在以 [{src_cam}] 为源视角提取双轨白金真值...")

    cam_data = sliced_scene_constants['camera'][src_cam]
    cam_state = sliced_scene_state[src_cam]
    h_img, w_img = cam_data['video_rgb'][0].shape[:2]
    T_frames = len(cam_data['video_rgb'])
    tracks_2d = cam_data['tracks_2d']
    raw_depth = cam_data['raw_depth']

    orig_2d_all = []
    traj_3d_all = []
    vis_all = []

    # =========================================================
    # 🦾 Track B: 提取 URDF 机械臂纯物理点
    # =========================================================
    traj_3d_rob, proj_2d_rob, vis_rob, robot_indices = urdf_tracker.extract_robot_tracks(
        src_cam, sliced_scene_constants, sliced_scene_state
    )
    if proj_2d_rob is not None and len(robot_indices) > 0:
        orig_2d_all.append(tracks_2d[:, robot_indices, :])
        traj_3d_all.append(traj_3d_rob)
        vis_all.append(vis_rob)

    # =========================================================
    # 👁️ Track A: 提取纯环境点并执行终极法庭过滤
    # =========================================================
    joint_angles_t0 = sliced_scene_constants['robot']['joint_positions'][0]
    gripper_state_t0 = sliced_scene_constants['robot']['gripper_positions'][0]
    pb_renderer_ultimate.update_robot_pose(joint_angles_t0, gripper_state=gripper_state_t0)

    robot_mask = pb_renderer_ultimate.render_mask(cam_state['extrinsics'][0], cam_data['K_mat'], w_img, h_img) > 0
    kernel = np.ones((15, 15), np.uint8)
    robot_mask_dilated = cv2.dilate(robot_mask.astype(np.uint8), kernel, iterations=1) > 0

    u0 = np.clip(np.round(tracks_2d[0, :, 0]).astype(int), 0, w_img-1)
    v0 = np.clip(np.round(tracks_2d[0, :, 1]).astype(int), 0, h_img-1)
    is_env_point = ~robot_mask_dilated[v0, u0]
    has_valid_depth = (raw_depth[0, v0, u0] > 0.05) & (raw_depth[0, v0, u0] < 5.0)
    test_env_indices = np.where(is_env_point & has_valid_depth)[0]

    if len(test_env_indices) > 0:
        smoothed_3d, refined_2d, unified_vis = consensus_tracker.extract_env_tracks(
            src_cam, test_env_indices, sliced_scene_constants, sliced_scene_state
        )
        original_2d_env_raw = tracks_2d[:, test_env_indices, :]
        valid_counts = np.sum(unified_vis, axis=0)
        D_2D = np.linalg.norm(original_2d_env_raw - refined_2d[:, :, :2], axis=-1)
        max_drift = np.max(np.where(unified_vis, D_2D, 0), axis=0)
        flicker_counts = np.sum(unified_vis[:-1] != unified_vis[1:], axis=0)

        survivor_mask = (valid_counts >= 5) & (max_drift < 10.0) & (flicker_counts <= T_frames * 0.10)
        golden_indices = np.where(survivor_mask)[0]

        if len(golden_indices) > 0:
            traj_3d_env = smoothed_3d[:, golden_indices, :]
            vis_env = unified_vis[:, golden_indices].copy()
            orig_2d_env = original_2d_env_raw[:, golden_indices, :]
            proj_2d_env = refined_2d[:, golden_indices, :2]
            z_pred_env = refined_2d[:, golden_indices, 2]

            for t in range(T_frames):
                ui = np.clip(np.round(proj_2d_env[t, :, 0]).astype(int), 0, w_img - 1)
                vi = np.clip(np.round(proj_2d_env[t, :, 1]).astype(int), 0, h_img - 1)
                z_sensor = raw_depth[t, vi, ui]
                xray_occluded = (z_sensor > 0) & (z_sensor < z_pred_env[t] - 0.05)
                vis_env[t] = vis_env[t] & (~xray_occluded)

            orig_2d_all.append(orig_2d_env)
            traj_3d_all.append(traj_3d_env)
            vis_all.append(vis_env)

    # =========================================================
    # 🤝 双轨合并与渲染大盘构建
    # =========================================================
    if not orig_2d_all:
        print(f"  ⚠️ 视角 [{src_cam}] 无存活点，跳过。")
        continue

    combined_orig_2d = np.concatenate(orig_2d_all, axis=1)
    combined_traj_3d = np.concatenate(traj_3d_all, axis=1)
    combined_vis = np.concatenate(vis_all, axis=1)
    N_pts = combined_traj_3d.shape[1]

    print(f"  🏆 提取成功！共 {N_pts} 个双轨极品点。开始投影至全域视角...")

    y_coords = combined_orig_2d[0, :, 1]
    if len(y_coords) > 1 and y_coords.max() > y_coords.min():
        norm = plt.Normalize(y_coords.min(), y_coords.max())
    else:
        norm = plt.Normalize(y_coords.min() - 1, y_coords.min() + 1)
    point_colors = plt.cm.gist_rainbow(norm(y_coords))[:, :3] * 255

    row_videos = []

    for tgt_cam in camera_ids:
        tgt_data = sliced_scene_constants['camera'][tgt_cam]
        tgt_state = sliced_scene_state[tgt_cam]
        tgt_video = tgt_data['video_rgb']

        # 🌟 绝杀 Bug：用 -1000.0 替代 0.0 初始化，防止渲染器在 [0,0] 处狂画幽灵圆圈！
        proj_2d = np.full((T_frames, N_pts, 2), -1000.0, dtype=np.float32)
        proj_vis = np.zeros((T_frames, N_pts), dtype=bool)

        for t in range(T_frames):
            valid_3d = ~np.isnan(combined_traj_3d[t, :, 2])
            if valid_3d.any():
                u, v, z_p = project_points_np(combined_traj_3d[t, valid_3d], tgt_data['K_mat'], tgt_state['extrinsics'][t])

                ui = np.clip(np.round(u).astype(int), 0, w_img - 1)
                vi = np.clip(np.round(v).astype(int), 0, h_img - 1)

                in_bounds = (u >= 0) & (u < w_img) & (v >= 0) & (v < h_img) & (z_p > 0)
                z_sensor = tgt_data['raw_depth'][t, vi, ui]
                xray_occluded = (z_sensor > 0) & (z_sensor < z_p - 0.05)

                vis_t = np.zeros(N_pts, dtype=bool)
                vis_t[valid_3d] = in_bounds & (~xray_occluded)

                proj_vis[t] = vis_t & combined_vis[t]

                # 🌟 绝杀 Bug：用 -1000.0 初始化单帧坐标池！
                coords = np.full((N_pts, 2), -1000.0, dtype=np.float32)
                coords[valid_3d, 0] = u
                coords[valid_3d, 1] = v
                proj_2d[t] = coords

        print(f"    🎨 渲染流星特效 [Src: {src_cam[-4:]} -> Tgt: {tgt_cam[-4:]}]...")
        rendered_frames = render_2d_tracking_video(
            video_frames=tgt_video,
            tracks=proj_2d,
            visibility=proj_vis,
            global_colors=point_colors,
            linewidth=3
        )

        tgt_h, tgt_w = 256, int(256 * w_img / h_img)
        cam_vid = []
        for img in rendered_frames:
            label = f"Src:{src_cam[-6:]} -> Tgt:{tgt_cam[-6:]}"
            cv2.putText(img, label, (10, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 0), 4)
            cv2.putText(img, label, (10, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)
            cam_vid.append(cv2.resize(img, (tgt_w, tgt_h)))

        row_videos.append(np.array(cam_vid))

    all_grid_rows.append(np.concatenate(row_videos, axis=2))

if len(all_grid_rows) > 0:
    print("\n🎬 渲染完毕！正在展示 3x3 双轨大一统 2D 追踪流星阵列：")
    combined_video = np.concatenate(all_grid_rows, axis=1)
    media.show_video(combined_video, fps=15, codec='gif')
else:
    print("\n❌ 没有任何可展示的追踪阵列。")

In [ ]:
# @title 4D 彩虹流星可视化 (双轨大一统真值接入安全防爆版 - Polyscope 霓虹管版)
import polyscope as ps
import mediapy as media
import numpy as np
import cv2
import os
import matplotlib.pyplot as plt
from tqdm import tqdm

# =========================================================
# 渲染核心模块：完全解耦，直接接收清洗好的 3D 轨迹
# =========================================================
def render_4d_rainbow_meteor_flow(scene_constants, scene_state, src_cam,
                                 traj_3d, traj_valid, point_colors,
                                 min_depth=0.15, max_depth=1.5, max_env_points=300000,
                                 tail_length=100, width=640, height=360, pullback_dist=0.85):
    # 1. 自动环境解析
    camera_ids = sorted(scene_constants['camera'].keys())
    wrist_serial = scene_constants['meta']['wrist_serial']
    ext_cams = [cam for cam in camera_ids if cam != wrist_serial]
    cam_start, cam_end = ext_cams[0], ext_cams[-1]

    n_frames = len(scene_constants['camera'][cam_start]['video_rgb'])

    if traj_3d.shape[1] == 0:
        print(f"⚠️ 视角 [{src_cam}] 无存活的极品点，无法渲染流星！")
        return []

    # 🌟 2. Polyscope 影视级初始化
    ps.set_allow_headless_backends(True)
    ps.init()
    ps.set_up_dir("z_up")

    # 【极夜模式】深邃的暗调背景，让霓虹灯彻底爆发
    ps.set_background_color((0., 0., 0.))
    ps.set_ground_plane_mode("shadow_only")
    ps.set_window_size(width, height)

    # 锁定绝对物理尺度
    ps.set_automatically_compute_scene_extents(False)
    ps.set_length_scale(1.0)
    ps.set_bounding_box((-1.5, -1.5, -0.5), (1.5, 1.5, 1.5))

    video_frames = []
    tmp_img_path = "tmp_neon_meteor.jpg"

    # 3. 4D 影视级渲染循环
    for t in tqdm(range(n_frames), desc=f"🎥 4D 霓虹流星 (Polyscope)"):
        # --- A. 计算极度平滑的相机轨迹 (Orbit: cam_start -> cam_end) ---
        alpha = t / max(1, n_frames - 1)
        # 提取起止相机的 Eye (位置) 和 Forward (Z轴朝向)
        eye_start = scene_state[cam_start]['extrinsics'][t][:3, 3]
        eye_end = scene_state[cam_end]['extrinsics'][t][:3, 3]
        z_start = scene_state[cam_start]['extrinsics'][t][:3, 2]
        z_end = scene_state[cam_end]['extrinsics'][t][:3, 2]

        eye_interp = (1 - alpha) * eye_start + alpha * eye_end
        target_interp = eye_interp + ((1 - alpha) * z_start + alpha * z_end)

        # Pullback 运镜：让镜头顺着视线往后拉远一点，获得更宏大的视野
        look_dir = target_interp - eye_interp
        look_dir /= (np.linalg.norm(look_dir) + 1e-6)
        eye_interp -= look_dir * pullback_dist

        ps.look_at(tuple(eye_interp), tuple(target_interp))

        # --- B. 缝合静态环境 (并人为压暗亮度，营造“暗场”效果) ---
        env_results = [
            unproject_to_3d(
                depth=scene_constants['camera'][cam]['raw_depth'][t].astype(np.float32),
                color_img=scene_constants['camera'][cam]['video_rgb'][t].astype(np.float32) / 255.0,
                K_mat=scene_constants['camera'][cam]['K_mat'],
                T_cam2world=scene_state[cam]['extrinsics'][t],
                min_depth=min_depth, max_depth=max_depth
            ) for cam in camera_ids
        ]
        env_pts, env_cols = map(np.vstack, zip(*env_results))

        if len(env_pts) > 0:
            idx = np.random.choice(len(env_pts), min(max_env_points, len(env_pts)), replace=False)
            # 🌟 核心技巧：将环境光降低到 35%，充当赛博朋克的暗夜背景！
            dimmed_cols = np.clip(env_cols[idx] * 0.35, 0, 1)
            ps_env = ps.register_point_cloud("env", env_pts[idx], radius=0.002, point_render_mode='sphere')
            ps_env.add_color_quantity("rgb", dimmed_cols, enabled=True)

        # --- C. 构建 3D 霓虹圆管拖尾 (Curve Network) ---
        start_t = max(0, t - tail_length)
        all_nodes, all_edges, all_node_colors = [], [], []
        node_offset = 0

        for i in range(traj_3d.shape[1]):
            window_valid = traj_valid[start_t:t+1, i]
            if not window_valid.any(): continue

            valid_times = np.where(window_valid)[0]
            actual_times = start_t + valid_times
            pts = traj_3d[actual_times, i]
            n_pts = len(pts)

            if n_pts > 0:
                all_nodes.append(pts)
                # 计算透明度 Fade：越早的轨迹越暗
                fade = (actual_times - start_t) / max(1, t - start_t)
                fade_colors = point_colors[i] * fade[:, None]
                all_node_colors.append(fade_colors)

            # 把这一个点的历史轨迹连成线 (edges)
            if n_pts > 1:
                edges = np.column_stack((np.arange(n_pts - 1), np.arange(1, n_pts))) + node_offset
                all_edges.append(edges)
            node_offset += n_pts

        if all_nodes:
            cat_nodes = np.vstack(all_nodes)
            cat_cols = np.vstack(all_node_colors)
            cat_edges = np.vstack(all_edges) if all_edges else np.empty((0, 2), dtype=int)

            if len(cat_edges) > 0:
                # 🌟 注册为真 3D 圆管，并使用 flat 无光照材质发出霓虹光！
                ps_net = ps.register_curve_network("neon_tails", cat_nodes, cat_edges, radius=0.0025, material='flat')
                ps_net.add_color_quantity("fade_colors", cat_cols, defined_on='nodes', enabled=True)

        # --- D. 绘制流星头部 (明亮的球体) ---
        current_valid = traj_valid[t]
        if current_valid.any():
            head_pts = traj_3d[t, current_valid]
            head_cols = point_colors[current_valid]
            # 头部半径略大于拖尾，同样使用纯粹的 flat 材质
            ps_heads = ps.register_point_cloud("neon_heads", head_pts, radius=0.004, point_render_mode='sphere', material='flat')
            ps_heads.add_color_quantity("head_colors", head_cols, enabled=True)

        # --- E. 极速物理落盘与盖水印 ---
        ps.screenshot(tmp_img_path, transparent_bg=False)

        if os.path.exists(tmp_img_path):
            img_rgb = media.read_image(tmp_img_path).copy()
            watermark = f"4D Neon Flow | Src: {src_cam} | Orbit: {cam_start} -> {cam_end}"
            cv2.putText(img_rgb, watermark, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 4)
            cv2.putText(img_rgb, watermark, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
            video_frames.append(img_rgb)

        ps.remove_all_structures()

    return video_frames

# =========================================================
# 主数据重构流水线：重新提取双轨白金真值并输送给渲染器
# =========================================================
camera_keys = list(sliced_scene_constants['camera'].keys())
target_src_cam = camera_keys[1] if len(camera_keys) > 1 else camera_keys[0]

print(f"🔄 正在为 4D 渲染重构 3D 白金真值 (主视角: {target_src_cam})...")

T_frames = len(sliced_scene_constants['camera'][target_src_cam]['video_rgb'])
cam_data = sliced_scene_constants['camera'][target_src_cam]
cam_state = sliced_scene_state[target_src_cam]
h_img, w_img = cam_data['video_rgb'][0].shape[:2]
tracks_2d = cam_data['tracks_2d']
raw_depth = cam_data['raw_depth']

# 确保双引擎就绪
if 'pb_renderer_ultimate' not in locals():
    pb_renderer_ultimate = PyBulletRenderer_Robotiq()
if 'urdf_tracker' not in locals():
    urdf_tracker = URDFKinematicsTracker(pb_renderer_ultimate)
if 'consensus_tracker' not in locals():
    consensus_tracker = ConsensusVisualTracker(cotracker_model, device)

# ---------------------------------------------------------
# 🦾 1. 提取 Track B (URDF 物理刚体点)
# ---------------------------------------------------------
traj_3d_rob, proj_2d_rob, vis_rob, robot_indices = urdf_tracker.extract_robot_tracks(
    target_src_cam, sliced_scene_constants, sliced_scene_state
)
if proj_2d_rob is not None:
    orig_2d_rob = tracks_2d[:, robot_indices, :]
else:
    traj_3d_rob = np.empty((T_frames, 0, 3), dtype=np.float32)
    vis_rob = np.empty((T_frames, 0), dtype=bool)
    orig_2d_rob = np.empty((T_frames, 0, 2), dtype=np.float32)

# ---------------------------------------------------------
# 👁️ 2. 提取 Track A (15票共识环境点 + 终极漏斗过滤)
# ---------------------------------------------------------
joint_angles_t0 = sliced_scene_constants['robot']['joint_positions'][0]
gripper_state_t0 = sliced_scene_constants['robot']['gripper_positions'][0]
pb_renderer_ultimate.update_robot_pose(joint_angles_t0, gripper_state=gripper_state_t0)

robot_mask = pb_renderer_ultimate.render_mask(cam_state['extrinsics'][0], cam_data['K_mat'], w_img, h_img) > 0
kernel = np.ones((15, 15), np.uint8)
robot_mask_dilated = cv2.dilate(robot_mask.astype(np.uint8), kernel, iterations=1) > 0

u0 = np.clip(np.round(tracks_2d[0, :, 0]).astype(int), 0, w_img-1)
v0 = np.clip(np.round(tracks_2d[0, :, 1]).astype(int), 0, h_img-1)
is_env_point = ~robot_mask_dilated[v0, u0]
has_valid_depth = (raw_depth[0, v0, u0] > 0.05) & (raw_depth[0, v0, u0] < 5.0)
test_env_indices = np.where(is_env_point & has_valid_depth)[0]

smoothed_3d, refined_2d, unified_vis = consensus_tracker.extract_env_tracks(
    target_src_cam, test_env_indices, sliced_scene_constants, sliced_scene_state
)

original_2d_env_raw = tracks_2d[:, test_env_indices, :]
valid_counts = np.sum(unified_vis, axis=0)
D_2D = np.linalg.norm(original_2d_env_raw - refined_2d[:, :, :2], axis=-1)
max_drift = np.max(np.where(unified_vis, D_2D, 0), axis=0)
flicker_counts = np.sum(unified_vis[:-1] != unified_vis[1:], axis=0)

survivor_mask = (valid_counts >= 5) & (max_drift < 10.0) & (flicker_counts <= T_frames * 0.10)
golden_indices = np.where(survivor_mask)[0]

if len(golden_indices) > 0:
    traj_3d_env = smoothed_3d[:, golden_indices, :]
    vis_env = unified_vis[:, golden_indices].copy()
    orig_2d_env = original_2d_env_raw[:, golden_indices, :]
    proj_2d_env = refined_2d[:, golden_indices, :2]
    z_pred_env = refined_2d[:, golden_indices, 2]

    # 附加 X 光透视物理遮挡修复
    for t in range(T_frames):
        ui = np.clip(np.round(proj_2d_env[t, :, 0]).astype(int), 0, w_img - 1)
        vi = np.clip(np.round(proj_2d_env[t, :, 1]).astype(int), 0, h_img - 1)
        z_sensor = raw_depth[t, vi, ui]
        xray_occluded = (z_sensor > 0) & (z_sensor < z_pred_env[t] - 0.05)
        vis_env[t] = vis_env[t] & (~xray_occluded)
else:
    traj_3d_env = np.empty((T_frames, 0, 3), dtype=np.float32)
    vis_env = np.empty((T_frames, 0), dtype=bool)
    orig_2d_env = np.empty((T_frames, 0, 2), dtype=np.float32)

# ---------------------------------------------------------
# 🤝 3. 双轨合并并生成调色板，送入 4D 渲染器
# ---------------------------------------------------------
combined_traj_3d = np.concatenate([traj_3d_rob, traj_3d_env], axis=1)
combined_vis = np.concatenate([vis_rob, vis_env], axis=1)
combined_orig_2d = np.concatenate([orig_2d_rob, orig_2d_env], axis=1)

# 生成色彩矩阵
y_coords = combined_orig_2d[0, :, 1]
if len(y_coords) > 1 and y_coords.max() > y_coords.min():
    norm = plt.Normalize(y_coords.min(), y_coords.max())
else:
    norm = plt.Normalize(y_coords.min() - 1, y_coords.min() + 1)
point_colors = plt.cm.gist_rainbow(norm(y_coords))[:, :3]

# 启动引擎！
output_video = render_4d_rainbow_meteor_flow(
    scene_constants=sliced_scene_constants,
    scene_state=sliced_scene_state,
    src_cam=target_src_cam,
    traj_3d=combined_traj_3d,        # 🌟 直接注入合并后的纯净 3D 轨迹
    traj_valid=combined_vis,         # 🌟 直接注入合并后的生命周期掩码
    point_colors=point_colors,       # 🌟 注入对应的彩虹配色
    max_env_points=400000
)

if len(output_video) > 0:
    media.show_video(output_video, fps=15, codec='gif')

### 生成Droid数据集

In [ ]:
# @title 🚀 全自动批处理引擎 (双轨大一统终极版)

import os
import gc
import cv2
import numpy as np
import traceback

def extract_and_export_tapvid(export_root, scene_constants, scene_state, urdf_tracker, consensus_tracker, pb_renderer):
    """基于双轨大一统架构 (Track A+B) 提取极品 3D 真值并序列化为 TAPVid 格式"""
    print("  🧹 [4/4] 启动双轨提取、跨视角重投影与 TAPVid 最终落盘...")
    episode_id = scene_constants['meta']['episode_id']
    camera_ids = list(scene_constants['camera'].keys())
    T_frames = len(scene_constants['camera'][camera_ids[0]]['video_rgb'])

    all_final_tracks = []
    all_queries = []
    all_visibility = {cam: [] for cam in camera_ids}
    total_survivors = 0

    # 🌟 1. 遍历每一个机位作为“主视角 (Source View)”
    for v_source_idx, src_cam in enumerate(camera_ids):
        print(f"    🔍 正在提取视角 [{src_cam}] 的双轨极品真值...")
        cam_data = scene_constants['camera'][src_cam]
        cam_state = scene_state[src_cam]
        h_img, w_img = cam_data['video_rgb'][0].shape[:2]
        tracks_2d = cam_data['tracks_2d']
        raw_depth = cam_data['raw_depth']

        orig_2d_all = []
        traj_3d_all = []
        vis_all = []

        # ==========================================
        # 🦾 Track B: 提取 URDF 机械臂纯物理点
        # ==========================================
        traj_3d_rob, proj_2d_rob, vis_rob, robot_indices = urdf_tracker.extract_robot_tracks(
            src_cam, scene_constants, scene_state
        )
        if proj_2d_rob is not None and len(robot_indices) > 0:
            orig_2d_all.append(tracks_2d[:, robot_indices, :])
            traj_3d_all.append(traj_3d_rob)
            vis_all.append(vis_rob)

        # ==========================================
        # 👁️ Track A: 提取纯环境点并执行终极法庭过滤
        # ==========================================
        joint_angles_t0 = scene_constants['robot']['joint_positions'][0]
        gripper_state_t0 = scene_constants['robot']['gripper_positions'][0]
        pb_renderer.update_robot_pose(joint_angles_t0, gripper_state=gripper_state_t0)

        robot_mask = pb_renderer.render_mask(cam_state['extrinsics'][0], cam_data['K_mat'], w_img, h_img) > 0
        kernel = np.ones((15, 15), np.uint8)
        robot_mask_dilated = cv2.dilate(robot_mask.astype(np.uint8), kernel, iterations=1) > 0

        u0 = np.clip(np.round(tracks_2d[0, :, 0]).astype(int), 0, w_img-1)
        v0 = np.clip(np.round(tracks_2d[0, :, 1]).astype(int), 0, h_img-1)
        is_env_point = ~robot_mask_dilated[v0, u0]
        has_valid_depth = (raw_depth[0, v0, u0] > 0.05) & (raw_depth[0, v0, u0] < 5.0)
        test_env_indices = np.where(is_env_point & has_valid_depth)[0]

        if len(test_env_indices) > 0:
            smoothed_3d, refined_2d, unified_vis = consensus_tracker.extract_env_tracks(
                src_cam, test_env_indices, scene_constants, scene_state
            )
            original_2d_env_raw = tracks_2d[:, test_env_indices, :]
            valid_counts = np.sum(unified_vis, axis=0)
            D_2D = np.linalg.norm(original_2d_env_raw - refined_2d[:, :, :2], axis=-1)
            max_drift = np.max(np.where(unified_vis, D_2D, 0), axis=0)
            flicker_counts = np.sum(unified_vis[:-1] != unified_vis[1:], axis=0)

            # 🔪 [漏斗过滤]: 出勤 >= 5，漂移 < 10px，闪烁 <= 10%
            survivor_mask = (valid_counts >= 5) & (max_drift < 10.0) & (flicker_counts <= T_frames * 0.10)
            golden_indices = np.where(survivor_mask)[0]

            if len(golden_indices) > 0:
                traj_3d_env = smoothed_3d[:, golden_indices, :]
                vis_env = unified_vis[:, golden_indices].copy()
                orig_2d_env = original_2d_env_raw[:, golden_indices, :]
                proj_2d_env = refined_2d[:, golden_indices, :2]
                z_pred_env = refined_2d[:, golden_indices, 2]

                # 附加 X 光透视物理遮挡修复
                for t in range(T_frames):
                    ui = np.clip(np.round(proj_2d_env[t, :, 0]).astype(int), 0, w_img - 1)
                    vi = np.clip(np.round(proj_2d_env[t, :, 1]).astype(int), 0, h_img - 1)
                    z_sensor = raw_depth[t, vi, ui]
                    xray_occluded = (z_sensor > 0) & (z_sensor < z_pred_env[t] - 0.05)
                    vis_env[t] = vis_env[t] & (~xray_occluded)

                orig_2d_all.append(orig_2d_env)
                traj_3d_all.append(traj_3d_env)
                vis_all.append(vis_env)

        # ==========================================
        # 🤝 双轨合并检查
        # ==========================================
        if not orig_2d_all:
            print(f"    ⚠️ 视角 [{src_cam}] 未保留下任何极品点位。")
            continue

        combined_orig_2d = np.concatenate(orig_2d_all, axis=1) # [T, N, 2]
        combined_traj_3d = np.concatenate(traj_3d_all, axis=1) # [T, N, 3]
        combined_vis = np.concatenate(vis_all, axis=1)         # [T, N]

        N_survivors = combined_traj_3d.shape[1]
        total_survivors += N_survivors
        all_final_tracks.append(combined_traj_3d)

        # 收集 Queries (对应初始帧)
        queries = np.zeros((N_survivors, 4), dtype=np.float32)
        queries[:, 0] = combined_orig_2d[0, :, 0]
        queries[:, 1] = combined_orig_2d[0, :, 1]
        queries[:, 2] = 0
        queries[:, 3] = v_source_idx
        all_queries.append(queries)

        # ==========================================
        # 🎯 Target 视角交叉投影与 X 光遮挡解算
        # ==========================================
        for tgt_cam in camera_ids:
            if tgt_cam == src_cam:
                all_visibility[tgt_cam].append(combined_vis)
                continue

            tgt_data = scene_constants['camera'][tgt_cam]
            tgt_state = scene_state[tgt_cam]
            tgt_raw_depth = tgt_data['raw_depth']
            tgt_K = tgt_data['K_mat']
            tgt_ext = tgt_state['extrinsics']

            tgt_vis = np.zeros((T_frames, N_survivors), dtype=bool)

            for t in range(T_frames):
                # 过滤掉 Track A 里面本身就被遮挡插值为 NaN 的无根之水
                valid_3d = ~np.isnan(combined_traj_3d[t, :, 2])

                if valid_3d.any():
                    u_v, v_v, z_p = project_points_np(combined_traj_3d[t, valid_3d], tgt_K, tgt_ext[t])

                    ui = np.clip(np.round(u_v).astype(int), 0, w_img - 1)
                    vi = np.clip(np.round(v_v).astype(int), 0, h_img - 1)

                    in_bounds = (u_v >= 0) & (u_v < w_img) & (v_v >= 0) & (v_v < h_img) & (z_p > 0)
                    z_sensor = tgt_raw_depth[t, vi, ui]

                    # Target 视角的 X光透视屏蔽
                    xray_occluded = (z_sensor > 0) & (z_sensor < z_p - 0.05)

                    tgt_vis_t = np.zeros(N_survivors, dtype=bool)
                    tgt_vis_t[valid_3d] = in_bounds & (~xray_occluded)
                    tgt_vis[t] = tgt_vis_t

            all_visibility[tgt_cam].append(tgt_vis)

    if total_survivors == 0:
        print(f"  ⚠️ 警告：所有视角均无有效点位！跳过落盘。")
        return

    # 🌟 2. 大一统拼接 (将三个视角的点云聚合)
    final_tracks_xyz = np.concatenate(all_final_tracks, axis=1) # 最终形状: [T, N_total, 3]
    queries_xytv = np.concatenate(all_queries, axis=0)          # 最终形状: [N_total, 4]

    # 🌟 3. 全局文件落盘
    seq_dir = os.path.join(export_root, episode_id)
    os.makedirs(seq_dir, exist_ok=True)
    np.save(os.path.join(seq_dir, "tracks_xyz.npy"), final_tracks_xyz.astype(np.float32))
    np.save(os.path.join(seq_dir, "queries_xytv.npy"), queries_xytv)

    # 🌟 4. 各个 Target View 的底层数据分别落盘
    for v_idx, cam in enumerate(camera_ids):
        cam_dir = os.path.join(seq_dir, str(v_idx))
        os.makedirs(cam_dir, exist_ok=True)
        cam_data = scene_constants['camera'][cam]

        # 视频帧 JPEG 字节流化
        jpeg_bytes_list = [cv2.imencode('.jpg', cv2.cvtColor(img, cv2.COLOR_RGB2BGR))[1].tobytes() for img in cam_data['video_rgb']]
        np.save(os.path.join(cam_dir, "images_jpeg_bytes.npy"), np.array(jpeg_bytes_list, dtype=object))

        # 将这个目标相机对【全部源视角点云】的可见性横向拼接成 [T, N_total]
        final_vis = np.concatenate(all_visibility[cam], axis=1)
        np.save(os.path.join(cam_dir, "visibility.npy"), final_vis)

        # 相机内外参落盘
        K = cam_data['K_mat']
        np.save(os.path.join(cam_dir, "intrinsics.npy"), np.array([K[0,0], K[1,1], K[0,2], K[1,2]], dtype=np.float32))
        np.save(os.path.join(cam_dir, "extrinsics_w2c.npy"), np.linalg.inv(scene_state[cam]['extrinsics']).astype(np.float32))

    print(f"  🎉 海量提纯成功！三视角共保留 {total_survivors} 个极品点位，已存储至: {seq_dir}")


# ================= 全自动淘金总调度 =================
export_root = "/content/DROID-mv-eval"
os.makedirs(export_root, exist_ok=True)

# 黄金目标：截取动作末尾的 48 帧
SLICE_FRAMES = 48
TARGET_EPISODES = valid_ids[:30] # 测试前 30 个，跑通后可改为全量 (valid_ids)

print("\n" + "="*60)
print("🚀 启动终极全自动批处理流水线 (双轨大一统白金提纯版)")
print("="*60)

# 🌟 提前初始化物理引擎与双引擎，避免在循环中反复建棚销毁带来的时间损耗
pb_renderer_ultimate = PyBulletRenderer_Robotiq()
urdf_tracker = URDFKinematicsTracker(pb_renderer_ultimate)
# 注意：确保前面的 Cell 中已初始化了 cotracker_model
consensus_tracker = ConsensusVisualTracker(cotracker_model, device)

for ep_idx, episode_id in enumerate(TARGET_EPISODES):
    print(f"\n🎬 [Episode {ep_idx+1}/{len(TARGET_EPISODES)}] 开始处理: {episode_id}")

    # 预先声明局部指针，确保 finally 块中回收显存时不报错
    scene_constants, init_scene_state = None, None
    pybullet_scene_state, ultimate_scene_state = None, None

    try:
        # 🌟 1. 基础数据下载与解包 (移除了过时的 intrinsics_db)
        scene_constants = download_episode(episode_id, root_path, id_to_path, serials_db, keep_ranges)

        if scene_constants['meta']['valid_indices'] is None or len(scene_constants['meta']['valid_indices']) < SLICE_FRAMES:
            print("  ⏭️ 视频有效动作太短，跳过。")
            continue

        scene_constants = extract_svo_video(scene_constants)
        scene_constants = parse_robot_kinematics(scene_constants)
        scene_constants = align_temporal_streams(scene_constants)
        scene_constants = filter_idle_frames(scene_constants)

        # 🌟 2. 组装初始外参字典
        init_scene_state = init_camera_states(scene_constants, extrinsics_db)

        # 🌟 3. 核心提速魔法：在此刻直接切出最后 48 帧！
        scene_constants, init_scene_state = extract_last_n_frames(scene_constants, init_scene_state, n=SLICE_FRAMES)

        # 🌟 4. 视觉特征重度提取
        scene_constants = compute_stereo_depth(scene_constants, device)
        scene_constants = extract_2d_tracks(cotracker_model, scene_constants)

        # 🌟 5. 检查外参是否齐全，缺的话用 VGGT 补齐
        all_extrinsics_exist = all(state['extrinsics'] is not None for state in init_scene_state.values())
        if not all_extrinsics_exist:
            init_scene_state = vggt_warmup_extrinsics(scene_constants)

        # 🌟 6. 4D 全局物理大一统联合优化
        # [阶段一]：精准锁定外部相机外参 (Robot Alignment)
        pybullet_scene_state = run_stage2_robot_alignment(
            scene_constants=scene_constants,
            init_scene_state=init_scene_state,
            pb_renderer=pb_renderer_ultimate
        )

        # [阶段二]：环境缝合与夹爪微调 (Joint Alignment & Gravity Field)
        ultimate_scene_state = run_stage3_joint_alignment(
            scene_constants=scene_constants,
            stage2_scene_state=pybullet_scene_state,
            pb_renderer=pb_renderer_ultimate
        )

        # 🌟 7. 双轨大一统提取与 TAPVid 格式落盘
        extract_and_export_tapvid(
            export_root=export_root,
            scene_constants=scene_constants,
            scene_state=ultimate_scene_state,
            urdf_tracker=urdf_tracker,
            consensus_tracker=consensus_tracker,
            pb_renderer=pb_renderer_ultimate
        )

    except Exception as e:
        print(f"  ❌ 处理该集时发生意外错误: {e}")
        traceback.print_exc()

    finally:
        # 🌟 8. 滴水不漏的显存回收机制
        del scene_constants, init_scene_state
        del pybullet_scene_state, ultimate_scene_state
        gc.collect()
        torch.cuda.empty_cache()

print("\n🏆 所有指定视频批量处理完成！请移步下一单元格进行验证！")

In [ ]:
# @title ☁️ 自动打包并上传到 Google Drive

import os
import shutil
from google.colab import drive

# 🌟 1. 挂载云端硬盘 (建立跨界通道)
print("🔗 正在请求挂载 Google Drive (此时会弹出一个授权窗口，请点击允许)...")
drive.mount('/content/drive')

# 🌟 2. 语义化路径指针：杜绝硬编码，让数据的流向一目了然
source_dataset_dir = "/content/DROID-mv-eval"
local_zip_path = "/content/DROID_TAPVid3D_eval.zip"
drive_target_dir = "/content/drive/MyDrive/TAPVid3D-mv_v1"

os.makedirs(drive_target_dir, exist_ok=True)

# 🌟 3. 纯粹的 Python 极速打包：利用 shutil 替代生硬杂乱的 Shell 命令
print("\n📦 正在将高纯度数据集打包为 ZIP (开启极速传输模式)...")
# make_archive 极其优雅，会自动补充 .zip 后缀，且在底层执行最高效的流式压缩
shutil.make_archive(local_zip_path.replace('.zip', ''), 'zip', source_dataset_dir)

# 🌟 4. 平滑跨界转移：安全落盘至 Google Drive
print(f"\n🚀 正在将压缩包转移至你的云端硬盘: {drive_target_dir} ...")
shutil.copy2(local_zip_path, drive_target_dir)
print("✅ 备份完美收官！压缩包已安全躺在你的 Google Drive 中。")

# 🌟 5. 终极打扫战场：滴水不漏地释放 Colab 宝贵的本地空间
print("\n🧹 正在清理 Colab 本地缓存，释放硬盘空间...")
shutil.rmtree(source_dataset_dir, ignore_errors=True)
if os.path.exists(local_zip_path):
    os.remove(local_zip_path)
print("✨ 本地空间已清空，随时可以开始下一批次的提纯！")

In [ ]:
# @title 📊 读取落盘数据并执行终极视觉校验

export_root = "/content/DROID-mv-eval"
saved_episodes = [d for d in os.listdir(export_root) if os.path.isdir(os.path.join(export_root, d))]

if not saved_episodes:
    print("⚠️ 未找到任何保存的数据，请先运行前面的全自动批处理引擎！")
else:
    # 🌟 1. 随机抽查一个已落盘的 Episode
    test_ep = random.choice(saved_episodes)
    print(f"\n🔍 正在读取并校验落盘数据: {test_ep}")
    ep_dir = os.path.join(export_root, test_ep)

    # 🌟 2. 提取 3D 轨迹真值
    traj_3d = np.load(os.path.join(ep_dir, "tracks_xyz.npy")) # [T, N, 3]
    n_frames, n_points, _ = traj_3d.shape
    print(f"  ✅ 成功读取 3D 轨迹！帧数: {n_frames}, 极品跟踪点: {n_points} 个")

    # 🌟🌟🌟 核心修复：直接读取 Query 坐标作为全局唯一的调色板基准 🌟🌟🌟
    queries = np.load(os.path.join(ep_dir, "queries_xytv.npy"))
    y_vals_src = queries[:, 1] # 提取源视角初始帧的 Y 坐标
    global_colors = plt.cm.gist_rainbow(plt.Normalize(y_vals_src.min(), y_vals_src.max())(y_vals_src))[:, :3] * 255

    # 获取所有相机视角 (自动识别 0, 1, 2 文件夹)
    cam_dirs = sorted([d for d in os.listdir(ep_dir) if d.isdigit()], key=int)
    all_grid_videos = []

    for cam_idx in cam_dirs:
        cam_path = os.path.join(ep_dir, cam_idx)
        print(f"  🎥 正在重构视图 [{cam_idx}] 的矩阵与画面...")

        # 读取核心矩阵与可见性
        visibility = np.load(os.path.join(cam_path, "visibility.npy")) # [T, N]
        intrinsics = np.load(os.path.join(cam_path, "intrinsics.npy")) # [fx, fy, cx, cy]
        extrinsics_w2c = np.load(os.path.join(cam_path, "extrinsics_w2c.npy")) # [T, 4, 4]

        # 优雅还原 K 矩阵
        K_mat = np.array([
            [intrinsics[0], 0, intrinsics[2]],
            [0, intrinsics[1], intrinsics[3]],
            [0, 0, 1]
        ])

        # 还原世界系下的相机位姿 (T_cam2world)
        extrinsics_c2w = np.linalg.inv(extrinsics_w2c)

        # 瞬间解码 JPEG 字节流
        jpeg_bytes = np.load(os.path.join(cam_path, "images_jpeg_bytes.npy"), allow_pickle=True)
        video_frames = []
        for b in jpeg_bytes:
            img_bgr = cv2.imdecode(np.frombuffer(b, np.uint8), cv2.IMREAD_COLOR)
            video_frames.append(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))

        # 🌟 3. 将 3D 轨迹拍回 2D 平面
        proj_trk = np.zeros((n_frames, n_points, 2))
        for t in range(n_frames):
            u, v, _ = project_points_np(traj_3d[t], K_mat, extrinsics_c2w[t])
            proj_trk[t, :, 0] = u
            proj_trk[t, :, 1] = v

        # 🌟 4. 调用原装画笔模块，强行注入刚刚算好的 global_colors！
        rendered_frames = render_2d_tracking_video(
            video_frames=video_frames,
            tracks=proj_trk,
            visibility=visibility,
            global_colors=global_colors, # <--- 颜色在这里同步！
            linewidth=4,
            pad_value=0
        )

        # 盖上专属验收戳与缩放对齐
        cam_vid = []
        for img in rendered_frames:
            text = f"View {cam_idx} Re-Proj"
            cv2.putText(img, text, (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 0), 4)
            cv2.putText(img, text, (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 2)
            cam_vid.append(cv2.resize(img, (320, int(320 * img.shape[0] / img.shape[1]))))

        all_grid_videos.append(np.array(cam_vid))

    # 🌟 5. 横向拼接放映
    print("\n🎨 2D 重投影质检完成！三个视角的彩虹流星现在绝对同步！")
    media.show_video(np.concatenate(all_grid_videos, axis=2), fps=15, codec='gif')

    # 🌟 6. 附赠 3D 稀疏骨架展台 (同样使用同步颜色)
    print("\n🌌 正在为您生成落盘 3D 点云分布 (TAPVid 格式仅保留 Sparse 追踪点)...")
    pts_3d_f0 = traj_3d[0]

    show_plotly_point_cloud(
        pts_3d_f0,
        global_colors, # <--- 3D 空间也用同样的颜色！
        title=f"Sparse 3D Tracks (Frame 0) - {test_ep}",
        max_points=n_points,
        eye_pos=(0, -0.8, -1.5)
    )

In [ ]:
# import sys
# import os
# import cv2
# import json
# import pyzed.sl as sl
# from tqdm import tqdm

# def extract_svo_to_png(svo_path, output_dir):
#     print(f"🔄 正在拆解: {svo_path}")

#     # 1. 初始化 ZED 相机参数 (离线模式)
#     init_params = sl.InitParameters()
#     init_params.set_from_svo_file(svo_path)
#     init_params.svo_real_time_mode = False  # 必须关闭实时模式，确保不漏帧

#     zed = sl.Camera()
#     status = zed.open(init_params)
#     if status != sl.ERROR_CODE.SUCCESS:
#         print(f"❌ 打开 SVO 失败: {status}")
#         return

#     # 2. 提取物理元数据 (内参、分辨率、基线)
#     cam_info = zed.get_camera_information()
#     calib = cam_info.camera_configuration.calibration_parameters

#     metadata = {
#         "resolution": {
#             "width": cam_info.camera_configuration.resolution.width,
#             "height": cam_info.camera_configuration.resolution.height
#         },
#         "baseline_meters": calib.get_camera_baseline(),
#         "left_intrinsics": {
#             "fx": calib.left_cam.fx,
#             "fy": calib.left_cam.fy,
#             "cx": calib.left_cam.cx,
#             "cy": calib.left_cam.cy,
#             "k1": calib.left_cam.disto[0],
#             "k2": calib.left_cam.disto[1]
#         },
#         "right_intrinsics": {
#             "fx": calib.right_cam.fx,
#             "fy": calib.right_cam.fy,
#             "cx": calib.right_cam.cx,
#             "cy": calib.right_cam.cy
#         }
#     }

#     # 3. 创建纯净的输出目录
#     os.makedirs(output_dir, exist_ok=True)
#     left_dir = os.path.join(output_dir, "left")
#     right_dir = os.path.join(output_dir, "right")
#     os.makedirs(left_dir, exist_ok=True)
#     os.makedirs(right_dir, exist_ok=True)

#     # 保存元数据
#     with open(os.path.join(output_dir, "camera_info.json"), 'w') as f:
#         json.dump(metadata, f, indent=4)

#     # 4. 逐帧无损导出
#     left_mat = sl.Mat()
#     right_mat = sl.Mat()
#     n_frames = zed.get_svo_number_of_frames()

#     for i in tqdm(range(n_frames), desc="导出 PNG 序列"):
#         if zed.grab() == sl.ERROR_CODE.SUCCESS:
#             # 提取左右眼画面
#             zed.retrieve_image(left_mat, sl.VIEW.LEFT)
#             zed.retrieve_image(right_mat, sl.VIEW.RIGHT)

#             # 转换为 NumPy 数组并去除 Alpha 通道 (BGRA -> BGR)
#             img_l = cv2.cvtColor(left_mat.get_data(), cv2.COLOR_BGRA2BGR)
#             img_r = cv2.cvtColor(right_mat.get_data(), cv2.COLOR_BGRA2BGR)

#             # 使用 cv2.imwrite 保存为默认的无损 PNG
#             cv2.imwrite(os.path.join(left_dir, f"frame_{i:05d}.png"), img_l)
#             cv2.imwrite(os.path.join(right_dir, f"frame_{i:05d}.png"), img_r)
#         else:
#             break

#     zed.close()
#     print(f"✅ 提取完成！数据已保存至: {output_dir}\n")

# if __name__ == "__main__":
#     # 使用示例
#     # extract_svo_to_png("path/to/your/video.svo", "output/cam_13263313")
#     pass